In [39]:
path="/data_lake/gold/intlpris/"

In [40]:
from pyspark.sql import SparkSession
from impala.dbapi import connect
import pandas as pd
import os
import json
import requests
from pyspark.sql import functions as sf, types as T
import sys
from datetime import datetime, timedelta

spark = SparkSession.builder \
            .appName("Inteligência Prisional") \
            .config("spark.dynamicAllocation.enabled", "true") \
            .config("spark.dynamicAllocation.initialExecutors", "1") \
            .config("spark.dynamicAllocation.minExecutors", "1") \
            .config("spark.dynamicAllocation.maxExecutors", "4") \
            .config("spark.executor.memory", "4g") \
            .config("spark.executor.cores", "1") \
            .config("spark.driver.memory", "4g") \
            .config("spark.driver.cores", "1") \
            .config("spark.yarn.executor.memoryOverhead", "1g") \
            .config("spark.executor.memoryOverhead", "1g") \
            .config("spark.sql.parquet.int96RebaseModeInWrite", "LEGACY") \
            .config("spark.sql.parquet.datetimeRebaseModeInWrite", "LEGACY") \
            .master('local') \
            .enableHiveSupport() \
            .getOrCreate()

In [3]:
def get_dtype_df(dataframe):
    string_cols = [col for col, dtype in dataframe.dtypes if dtype == "string"]
    
    # max em BYTES (UTF-8) para cada coluna string
    if string_cols:
        df_max_bytes = (
            dataframe.select([
                sf.max(sf.length(sf.encode(sf.col(col), "UTF-8"))).alias(col)
                for col in string_cols
            ])
            .first()
            .asDict()
        )
    else:
        df_max_bytes = {}
    
    df_schema = pd.json_normalize(json.loads(dataframe.schema.json()), record_path=['fields'])[['name', 'type']]

    def adjust_dtype(name, dtype):
        if dtype == "long":
            return "bigint"
        
        if dtype == 'string':
            max_length = df_max_bytes.get(name) or 1
            
            # ajusta para CHAR(n) se for até 16 caracteres, senão usa VARCHAR(n)
            if max_length <= 16:
                return f"char({max_length})"  # garantir CHAR(1) como mínimo
            else:
                return f"varchar({max_length})" 
            
        else:
            return dtype

    df_schema['type'] = df_schema.apply(lambda x: adjust_dtype(x['name'], x['type']), axis=1)
    return df_schema

def write_impala_table_partioned(df, impala_schema, impala_table, br_path):
    dtype_df = get_dtype_df(df)
    dtypes_sting = ",\n    ".join([f"{row['name']} {row['type']}" for index, row in dtype_df.iterrows()])
    creation_query = f"""CREATE EXTERNAL TABLE {impala_schema}.{impala_table} 
                        ({dtypes_sting})
                        STORED AS PARQUET LOCATION '{br_path}'
                        """
    drop_query = f"""DROP TABLE {impala_schema}.{impala_table} """
    conn = connect(host='worker03-prod.sejus.es.gov.br', 
                   port=21050, 
                   database='bronze', 
                   auth_mechanism='GSSAPI',
                   kerberos_service_name='impala',
                   use_ssl=True,
                   ca_cert="/var/lib/cloudera-scm-agent/agent-cert/cm-auto-global_cacerts.pem")

    cursor = conn.cursor()
    print(creation_query)
    print(drop_query)
    try:
        cursor.execute(drop_query)
        cursor.execute(creation_query)
        cursor.execute(f"COMPUTE STATS {impala_schema}.{impala_table}")
        print(f"Estatísticas atualizadas para {impala_schema}.{impala_table}")
    except: 
        cursor.execute(creation_query)
        cursor.execute(f"COMPUTE STATS {impala_schema}.{impala_table}")
        print(f"Estatísticas atualizadas para {impala_schema}.{impala_table}")
        
def enviar_gold_para_postgres(nome_tabela_origem, pk_postgres):
    """
    Cria/substitui uma tabela no PostgreSQL a partir de uma tabela Spark/Hive
    e define uma chave primária no campo informado, caso pk_postgres seja informado.

    Parâmetros:
        nome_tabela_origem : str
            Nome completo da tabela no Spark, ex: "gold.sinp_pres_loc_atual"

        pk_postgres : str
            Nome do campo que será chave primária no PostgreSQL.
            Se vier "" ou None, a tabela será criada sem PK.

    Premissas:
        - a tabela de origem já existe no Spark
        - o driver JDBC do PostgreSQL está disponível no cluster
        - o schema/tabela de destino no PostgreSQL terão o mesmo nome da origem
          Ex: gold.sinp_pres_loc_atual -> schema "sinp", tabela "sinp_pres_loc_atual"
    """

    url = "jdbc:postgresql://10.242.38.126:5432/sinp_db"
    usuario = "usr_sinp"
    senha = "u9oLzKOato#nksFZ"
    driver = "org.postgresql.Driver"

    partes = nome_tabela_origem.split(".")
    if len(partes) != 2:
        raise ValueError("Informe a tabela no formato schema.tabela. Ex: gold.sinp_pres_loc_atual")

    schema_destino = "sinp"
    tabela_destino = partes[1]

    spark.catalog.clearCache()
    spark.sql(f"REFRESH TABLE {nome_tabela_origem}")
    df = spark.table(nome_tabela_origem).cache()
    df.count()

    colunas_df = df.columns

    tem_pk = pk_postgres is not None and str(pk_postgres).strip() != ""
    pk_postgres = str(pk_postgres).strip() if pk_postgres is not None else ""

    if tem_pk and pk_postgres not in colunas_df:
        raise ValueError(f"A PK '{pk_postgres}' não existe na tabela de origem. Colunas disponíveis: {colunas_df}")

    def mapear_tipo_postgres(campo):
        tipo = campo.dataType.simpleString().lower()

        if tipo.startswith("string"):
            return "varchar"
        elif tipo.startswith("int"):
            return "integer"
        elif tipo.startswith("bigint") or tipo.startswith("long"):
            return "bigint"
        elif tipo.startswith("double"):
            return "double precision"
        elif tipo.startswith("float"):
            return "real"
        elif tipo.startswith("boolean"):
            return "boolean"
        elif tipo.startswith("timestamp"):
            return "timestamp"
        elif tipo.startswith("date"):
            return "date"
        elif tipo.startswith("smallint"):
            return "smallint"
        elif tipo.startswith("decimal"):
            return tipo.replace("decimal", "numeric")
        else:
            return "text"

    ddl_colunas = []
    for campo in df.schema.fields:
        nome_coluna = campo.name
        tipo_pg = mapear_tipo_postgres(campo)
        ddl_colunas.append(f'"{nome_coluna}" {tipo_pg}')

    ddl_create_schema = f'create schema if not exists "{schema_destino}"'
    ddl_drop_table = f'drop table if exists "{schema_destino}"."{tabela_destino}"'

    if tem_pk:
        ddl_create_table = f'''
            create table "{schema_destino}"."{tabela_destino}" (
                {", ".join(ddl_colunas)},
                constraint pk_{tabela_destino} primary key ("{pk_postgres}")
            )
        '''
    else:
        ddl_create_table = f'''
            create table "{schema_destino}"."{tabela_destino}" (
                {", ".join(ddl_colunas)}
            )
        '''

    jvm = spark._sc._gateway.jvm
    jvm.java.lang.Class.forName(driver)

    conn = jvm.java.sql.DriverManager.getConnection(url, usuario, senha)
    stmt = conn.createStatement()

    try:
        stmt.execute(ddl_create_schema)
        stmt.execute(ddl_drop_table)
        stmt.execute(ddl_create_table)
    finally:
        stmt.close()
        conn.close()

    propriedades = {
        "user": usuario,
        "password": senha,
        "driver": driver
    }

    df.write \
        .mode("append") \
        .jdbc(
            url=url,
            table=f'{schema_destino}.{tabela_destino}',
            properties=propriedades
        )

    print(f"Tabela enviada com sucesso para o PostgreSQL: {schema_destino}.{tabela_destino}")
    if tem_pk:
        print(f"PK definida: {pk_postgres}")
    else:
        print("Tabela criada sem PK.")

In [4]:
spark.sql("REFRESH TABLE bronze.infopen_preso_documentos")
spark.sql("REFRESH TABLE bronze.infopen_documentos_tipos")
spark.sql("REFRESH TABLE bronze.infopen_presos")
spark.sql("REFRESH TABLE bronze.infopen_preso_filiacao")
spark.sql("REFRESH TABLE bronze.infopen_preso_cor_pele_etnia")
spark.sql("REFRESH TABLE bronze.infopen_cor_pele_etnia")

spark.sql("REFRESH TABLE bronze.infopen_social_estado_civil")
spark.sql("REFRESH TABLE bronze.infopen_estado_civil")
spark.sql("REFRESH TABLE bronze.infopen_social_escolaridade")
spark.sql("REFRESH TABLE bronze.infopen_escolaridade")
spark.sql("REFRESH TABLE bronze.infopen_grau_instrucao")
spark.sql("REFRESH TABLE bronze.infopen_presos_profissao")
spark.sql("REFRESH TABLE bronze.infopen_profissao")
spark.sql("REFRESH TABLE bronze.infopen_social_religiao")
spark.sql("REFRESH TABLE bronze.infopen_religiao")
spark.sql("REFRESH TABLE bronze.infopen_social_quantidadefilho")
spark.sql("REFRESH TABLE bronze.infopen_preso_quantidadefilho")
spark.sql("REFRESH TABLE bronze.infopen_preso_naturalidade")
spark.sql("REFRESH TABLE bronze.infopen_geral_municipios")
spark.sql("REFRESH TABLE bronze.infopen_preso_nacionalidade_estrangeiro")
spark.sql("REFRESH TABLE bronze.infopen_necessidade_especial")
spark.sql("REFRESH TABLE bronze.infopen_ficha_social")
spark.sql("REFRESH TABLE bronze.infopen_preso_prontuario_social_corr")

spark.catalog.clearCache()

base_sql = """
with docs_base as (
    select
        cast(d.id_preso as string) as id_preso,
        d.id_documentotipo,
        regexp_replace(coalesce(d.presodocumento_numero, ''), '[^0-9]', '') as id_documento_limpo
    from bronze.infopen_preso_documentos d
    where d.presodocumento_numero is not null
      and regexp_replace(coalesce(d.presodocumento_numero, ''), '[^0-9]', '') <> ''
      and cast(regexp_replace(coalesce(d.presodocumento_numero, ''), '[^0-9]', '') as bigint) >= 1000
),

classificacao as (
    select
        id_preso,
        max(case when id_documentotipo in (18,19) then 1 else 0 end) as tem_doc_nacional,
        max(case when id_documentotipo = 26 then 1 else 0 end) as tem_passaporte
    from docs_base
    group by id_preso
),

filiacao as (
    select
        cast(id_preso as string) as id_preso,
        max(
            case
                when upper(regexp_replace(coalesce(presofiliacao_mae, ''), '\\\\s+', ' ')) rlike ' OU '
                    then trim(
                        regexp_extract(
                            regexp_replace(coalesce(presofiliacao_mae, ''), '\\\\s+', ' '),
                            '^(.*?)\\\\s+(?i:OU)\\\\s+.*$',
                            1
                        )
                    )
                else trim(regexp_replace(coalesce(presofiliacao_mae, ''), '\\\\s+', ' '))
            end
        ) as nome_mae,
        max(
            case
                when upper(regexp_replace(coalesce(presofiliacao_pai, ''), '\\\\s+', ' ')) rlike ' OU '
                    then trim(
                        regexp_extract(
                            regexp_replace(coalesce(presofiliacao_pai, ''), '\\\\s+', ' '),
                            '^(.*?)\\\\s+(?i:OU)\\\\s+.*$',
                            1
                        )
                    )
                else trim(regexp_replace(coalesce(presofiliacao_pai, ''), '\\\\s+', ' '))
            end
        ) as nome_pai
    from bronze.infopen_preso_filiacao
    group by cast(id_preso as string)
),

etnia_base as (
    select
        cast(cpe.id_preso as string) as id_preso,
        concat_ws(', ', collect_set(cor.corpeleetnia_descricao)) as etnia
    from bronze.infopen_preso_cor_pele_etnia cpe
    inner join bronze.infopen_cor_pele_etnia cor
        on cor.id_corpeleetnia = cpe.id_corpeleetnia
    group by cast(cpe.id_preso as string)
),

estado_civil_social_rank as (
    select
        cast(s.id_preso as string) as id_preso,
        trim(regexp_replace(coalesce(ec.estadocivil_descricao, ''), '\\\\s+', ' ')) as estado_civil_social,
        row_number() over (
            partition by cast(s.id_preso as string)
            order by
                case when s.social_estadocivil_data is not null then 1 else 2 end,
                s.social_estadocivil_data desc,
                s.id_socialestadocivil desc
        ) as rn
    from bronze.infopen_social_estado_civil s
    left join bronze.infopen_estado_civil ec
        on ec.id_estadocivil = s.id_estadocivil
),

estado_civil_social_base as (
    select
        id_preso,
        estado_civil_social
    from estado_civil_social_rank
    where rn = 1
),

ficha_social_rank as (
    select
        cast(fs.id_preso as string) as id_preso,
        trim(regexp_replace(coalesce(ec.estadocivil_descricao, ''), '\\\\s+', ' ')) as estado_civil_ficha,
        trim(regexp_replace(coalesce(gi.grauinstrucao_descricao, ''), '\\\\s+', ' ')) as escolaridade_ficha,
        trim(regexp_replace(coalesce(pr.profissao_descricao, ''), '\\\\s+', ' ')) as profissao_ficha,
        trim(regexp_replace(coalesce(rg.religiao_descricao, ''), '\\\\s+', ' ')) as religiao_ficha,
        trim(regexp_replace(coalesce(ne.necessidadeespecial_descricao, ''), '\\\\s+', ' ')) as necessidade_especial_ficha,
        row_number() over (
            partition by cast(fs.id_preso as string)
            order by fs.id_fichasocial desc
        ) as rn
    from bronze.infopen_ficha_social fs
    left join bronze.infopen_estado_civil ec
        on ec.id_estadocivil = fs.id_estadocivil
    left join bronze.infopen_grau_instrucao gi
        on gi.id_grauinstrucao = fs.id_grauinstrucao
    left join bronze.infopen_profissao pr
        on pr.id_profissao = fs.id_profissao
    left join bronze.infopen_religiao rg
        on rg.id_religiao = fs.id_religiao
    left join bronze.infopen_necessidade_especial ne
        on ne.id_necessidadeespecial = fs.id_necessidadeespecial
),

ficha_social_base as (
    select
        id_preso,
        estado_civil_ficha,
        escolaridade_ficha,
        profissao_ficha,
        religiao_ficha,
        necessidade_especial_ficha
    from ficha_social_rank
    where rn = 1
),

escolaridade_social_rank as (
    select
        cast(s.id_preso as string) as id_preso,
        trim(regexp_replace(coalesce(e.escolaridade_descricao, ''), '\\\\s+', ' ')) as escolaridade_social,
        row_number() over (
            partition by cast(s.id_preso as string)
            order by
                case when s.social_escolaridade_data is not null then 1 else 2 end,
                s.social_escolaridade_data desc,
                s.id_socialescolaridade desc
        ) as rn
    from bronze.infopen_social_escolaridade s
    left join bronze.infopen_escolaridade e
        on e.id_escolaridade = s.id_escolaridade
),

escolaridade_social_base as (
    select
        id_preso,
        escolaridade_social
    from escolaridade_social_rank
    where rn = 1
),

profissao_rank as (
    select
        cast(pf.id_preso as string) as id_preso,
        trim(regexp_replace(coalesce(pf.descricao_profissao, p.profissao_descricao, ''), '\\\\s+', ' ')) as profissao_social,
        row_number() over (
            partition by cast(pf.id_preso as string)
            order by
                case when pf.dt_profissao is not null then 1 else 2 end,
                pf.dt_profissao desc,
                pf.id_presoprofissao desc
        ) as rn
    from bronze.infopen_presos_profissao pf
    left join bronze.infopen_profissao p
        on p.id_profissao = pf.id_profissao
),

profissao_base as (
    select
        id_preso,
        profissao_social
    from profissao_rank
    where rn = 1
),

religiao_social_rank as (
    select
        cast(s.id_preso as string) as id_preso,
        trim(regexp_replace(coalesce(r.religiao_descricao, ''), '\\\\s+', ' ')) as religiao_social,
        row_number() over (
            partition by cast(s.id_preso as string)
            order by
                case when s.social_religiao_data is not null then 1 else 2 end,
                s.social_religiao_data desc,
                s.id_socialreligiao desc
        ) as rn
    from bronze.infopen_social_religiao s
    left join bronze.infopen_religiao r
        on r.id_religiao = s.id_religiao
),

religiao_social_base as (
    select
        id_preso,
        religiao_social
    from religiao_social_rank
    where rn = 1
),

quantidade_filho_rank as (
    select
        cast(s.id_preso as string) as id_preso,
        trim(regexp_replace(coalesce(q.presoquantidadefilho_descricao, ''), '\\\\s+', ' ')) as quantidade_filhos,
        row_number() over (
            partition by cast(s.id_preso as string)
            order by
                case when s.social_quantidadefilho_data is not null then 1 else 2 end,
                s.social_quantidadefilho_data desc,
                s.id_socialquantidadefilho desc
        ) as rn
    from bronze.infopen_social_quantidadefilho s
    left join bronze.infopen_preso_quantidadefilho q
        on q.id_presoquantidadefilho = s.id_presoquantidadefilho
),

quantidade_filho_base as (
    select
        id_preso,
        quantidade_filhos
    from quantidade_filho_rank
    where rn = 1
),

naturalidade_base as (
    select
        cast(n.id_preso as string) as id_preso,
        trim(regexp_replace(coalesce(m.municipio_nome, ''), '\\\\s+', ' ')) as naturalidade_municipio,
        trim(regexp_replace(coalesce(m.municipio_siglauf, ''), '\\\\s+', ' ')) as naturalidade_uf
    from bronze.infopen_preso_naturalidade n
    left join bronze.infopen_geral_municipios m
        on m.id_municipio = n.id_municipio
),

nacionalidade_estrangeiro_base as (
    select
        cast(id_preso as string) as id_preso,
        trim(regexp_replace(coalesce(presonacionalidadeestrangeiro_paisorigem, ''), '\\\\s+', ' ')) as pais_origem_estrangeiro,
        trim(regexp_replace(coalesce(presonacionalidadeestrangeiro_cidade, ''), '\\\\s+', ' ')) as cidade_origem_estrangeiro
    from bronze.infopen_preso_nacionalidade_estrangeiro
),

prontuario_social_rank as (
    select
        cast(id_preso as string) as id_preso,

        case
            when lower(trim(cast(st_tem_filhos as string))) in ('1', 'true', 't', 'sim', 's') then 1
            when lower(trim(cast(st_tem_filhos as string))) in ('0', 'false', 'f', 'nao', 'não', 'n') then 0
            else null
        end as flag_tem_filhos_prontuario,

        case
            when lower(trim(cast(st_sabe_ler as string))) in ('1', 'true', 't', 'sim', 's') then 1
            when lower(trim(cast(st_sabe_ler as string))) in ('0', 'false', 'f', 'nao', 'não', 'n') then 0
            else null
        end as flag_sabe_ler,

        case
            when lower(trim(cast(st_sabe_escrever as string))) in ('1', 'true', 't', 'sim', 's') then 1
            when lower(trim(cast(st_sabe_escrever as string))) in ('0', 'false', 'f', 'nao', 'não', 'n') then 0
            else null
        end as flag_sabe_escrever,

        case
            when lower(trim(cast(st_recebe_visita as string))) in ('1', 'true', 't', 'sim', 's') then 1
            when lower(trim(cast(st_recebe_visita as string))) in ('0', 'false', 'f', 'nao', 'não', 'n') then 0
            else null
        end as flag_recebe_visita,

        case
            when lower(trim(cast(st_relac_conjugal as string))) in ('1', 'true', 't', 'sim', 's') then 1
            when lower(trim(cast(st_relac_conjugal as string))) in ('0', 'false', 'f', 'nao', 'não', 'n') then 0
            else null
        end as flag_relacao_conjugal,

        row_number() over (
            partition by cast(id_preso as string)
            order by
                case when dt_cadastro_ficha is not null then 1 else 2 end,
                to_date(dt_cadastro_ficha) desc,
                id_prontuario_social desc
        ) as rn
    from bronze.infopen_preso_prontuario_social_corr
),

prontuario_social_base as (
    select
        id_preso,
        flag_tem_filhos_prontuario,
        flag_sabe_ler,
        flag_sabe_escrever,
        flag_recebe_visita,
        flag_relacao_conjugal
    from prontuario_social_rank
    where rn = 1
),

base_documento as (
    select
        d.id_preso,
        d.id_documentotipo,
        dt.documentotipo_descricao,
        d.id_documento_limpo,
        c.tem_doc_nacional,
        c.tem_passaporte,
        case
            when upper(regexp_replace(coalesce(p.preso_nome, ''), '\\\\s+', ' ')) rlike ' OU '
                then trim(
                    regexp_extract(
                        regexp_replace(coalesce(p.preso_nome, ''), '\\\\s+', ' '),
                        '^(.*?)\\\\s+(?i:OU)\\\\s+.*$',
                        1
                    )
                )
            else trim(regexp_replace(coalesce(p.preso_nome, ''), '\\\\s+', ' '))
        end as nome_pessoa,
        p.preso_sexo as sexo_pessoa,
        p.preso_datanascimento as data_nascimento_pessoa,
        p.preso_dataultimaprisao as data_ultima_prisao,
        f.nome_mae,
        f.nome_pai,
        e.etnia,

        coalesce(ec.estado_civil_social, fs.estado_civil_ficha) as estado_civil,
        coalesce(es.escolaridade_social, fs.escolaridade_ficha) as escolaridade,
        coalesce(pr.profissao_social, fs.profissao_ficha) as profissao,
        coalesce(rg.religiao_social, fs.religiao_ficha) as religiao,

        case
            when ps.flag_tem_filhos_prontuario is not null then ps.flag_tem_filhos_prontuario
            when upper(coalesce(qf.quantidade_filhos, '')) in ('', 'NAO INFORMADO', 'NÃO INFORMADO') then 0
            when upper(coalesce(qf.quantidade_filhos, '')) rlike '^(0|ZERO|NENHUM|NENHUMA|SEM FILHOS)$' then 0
            else 1
        end as flag_tem_filhos,

        qf.quantidade_filhos,
        nat.naturalidade_municipio,
        nat.naturalidade_uf,
        estr.pais_origem_estrangeiro,
        estr.cidade_origem_estrangeiro,
        fs.necessidade_especial_ficha as necessidade_especial,
        ps.flag_sabe_ler,
        ps.flag_sabe_escrever,
        ps.flag_recebe_visita,
        ps.flag_relacao_conjugal,

        row_number() over (
            partition by d.id_preso
            order by case
                when d.id_documentotipo = 19 then 1
                when d.id_documentotipo = 18 then 2
                when d.id_documentotipo = 26 then 3
                else 4
            end,
            length(d.id_documento_limpo) desc,
            d.id_documento_limpo desc
        ) as rn_doc
    from docs_base d
    inner join bronze.infopen_presos p
        on cast(p.id_preso as string) = d.id_preso
    inner join classificacao c
        on c.id_preso = d.id_preso
    left join bronze.infopen_documentos_tipos dt
        on dt.id_documentotipo = d.id_documentotipo
    left join filiacao f
        on f.id_preso = d.id_preso
    left join etnia_base e
        on e.id_preso = d.id_preso
    left join estado_civil_social_base ec
        on ec.id_preso = d.id_preso
    left join ficha_social_base fs
        on fs.id_preso = d.id_preso
    left join escolaridade_social_base es
        on es.id_preso = d.id_preso
    left join profissao_base pr
        on pr.id_preso = d.id_preso
    left join religiao_social_base rg
        on rg.id_preso = d.id_preso
    left join quantidade_filho_base qf
        on qf.id_preso = d.id_preso
    left join naturalidade_base nat
        on nat.id_preso = d.id_preso
    left join nacionalidade_estrangeiro_base estr
        on estr.id_preso = d.id_preso
    left join prontuario_social_base ps
        on ps.id_preso = d.id_preso
),

melhor_por_preso as (
    select
        id_preso,
        id_documentotipo,
        documentotipo_descricao,
        id_documento_limpo,
        tem_doc_nacional,
        tem_passaporte,
        nome_pessoa,
        sexo_pessoa,
        data_nascimento_pessoa,
        data_ultima_prisao,
        nome_mae,
        nome_pai,
        etnia,
        estado_civil,
        escolaridade,
        profissao,
        religiao,
        flag_tem_filhos,
        quantidade_filhos,
        naturalidade_municipio,
        naturalidade_uf,
        pais_origem_estrangeiro,
        cidade_origem_estrangeiro,
        necessidade_especial,
        flag_sabe_ler,
        flag_sabe_escrever,
        flag_recebe_visita,
        flag_relacao_conjugal,
        case
            when tem_doc_nacional = 1 then concat('NAC_', id_documento_limpo)
            when tem_doc_nacional = 0 and tem_passaporte = 1 and id_documentotipo = 26 then concat('EST_', id_documento_limpo)
            else concat('NAC_', id_documento_limpo)
        end as id_pessoa
    from base_documento
    where rn_doc = 1
),

dedup_id_pessoa as (
    select
        *,
        row_number() over (
            partition by id_pessoa
            order by
                case when data_ultima_prisao is not null then 1 else 2 end,
                to_date(data_ultima_prisao) desc,
                case when data_nascimento_pessoa is not null then 1 else 2 end,
                to_date(data_nascimento_pessoa) desc,
                id_preso desc
        ) as rn_pessoa,
        count(*) over (
            partition by id_pessoa
        ) as qtd_mesmo_id_pessoa,
        first_value(id_preso) over (
            partition by id_pessoa
            order by
                case when data_ultima_prisao is not null then 1 else 2 end,
                to_date(data_ultima_prisao) desc,
                case when data_nascimento_pessoa is not null then 1 else 2 end,
                to_date(data_nascimento_pessoa) desc,
                id_preso desc
        ) as id_preso_original
    from melhor_por_preso
),

base_final as (
    select
        cast(id_preso as string) as id_preso,
        cast(id_preso_original as string) as id_preso_original,
        id_pessoa,
        case
            when tem_doc_nacional = 1 then 'NACIONAL'
            when tem_doc_nacional = 0 and tem_passaporte = 1 then 'ESTRANGEIRO'
            else 'NACIONAL'
        end as origem,
        id_documentotipo as cod_documento_referencia,
        documentotipo_descricao as desc_documento_referencia,
        case
            when id_documentotipo = 19 then concat(
                substr(lpad(id_documento_limpo, 11, '0'), 1, 3), '.',
                substr(lpad(id_documento_limpo, 11, '0'), 4, 3), '.',
                substr(lpad(id_documento_limpo, 11, '0'), 7, 3), '-',
                substr(lpad(id_documento_limpo, 11, '0'), 10, 2)
            )
            else id_documento_limpo
        end as documento,
        nome_pessoa,
        sexo_pessoa,
        1 as flag_presidiario,
        0 as flag_advogado,
        0 as flag_servidor,
        0 as flag_visitante,
        0 as flag_ocorrencia_10d,
        0 as flag_ocorrencia_30d,
        0 as flag_ocorrencia_60d,
        to_date(data_nascimento_pessoa) as data_nascimento_pessoa,
        to_date(data_ultima_prisao) as data_ultima_prisao,
        nome_mae,
        nome_pai,
        etnia,

        estado_civil,
        escolaridade,
        profissao,
        religiao,
        flag_tem_filhos,
        quantidade_filhos,
        naturalidade_municipio,
        naturalidade_uf,
        pais_origem_estrangeiro,
        cidade_origem_estrangeiro,
        necessidade_especial,
        flag_sabe_ler,
        flag_sabe_escrever,
        flag_recebe_visita,
        flag_relacao_conjugal,

        rn_pessoa,
        qtd_mesmo_id_pessoa
    from dedup_id_pessoa
)

select * from base_final
"""

df_base_pessoa = spark.sql(base_sql)
df_base_pessoa.createOrReplaceTempView("vw_base_pessoa_dedup")

df_pres = spark.sql("""
select
    id_preso,
    id_pessoa,
    origem,
    cod_documento_referencia,
    desc_documento_referencia,
    documento,
    nome_pessoa,
    sexo_pessoa,
    flag_presidiario,
    flag_advogado,
    flag_servidor,
    flag_visitante,
    flag_ocorrencia_10d,
    flag_ocorrencia_30d,
    flag_ocorrencia_60d,
    data_nascimento_pessoa,
    data_ultima_prisao,
    nome_mae,
    nome_pai,
    etnia,

    estado_civil,
    escolaridade,
    profissao,
    religiao,
    flag_tem_filhos,
    quantidade_filhos,
    naturalidade_municipio,
    naturalidade_uf,
    pais_origem_estrangeiro,
    cidade_origem_estrangeiro,
    necessidade_especial,
    flag_sabe_ler,
    flag_sabe_escrever,
    flag_recebe_visita,
    flag_relacao_conjugal

from vw_base_pessoa_dedup
where rn_pessoa = 1
""")

df_pres_outras = spark.sql("""
select
    id_preso,
    id_preso_original,
    id_pessoa,
    origem,
    cod_documento_referencia,
    desc_documento_referencia,
    documento,
    nome_pessoa,
    sexo_pessoa,
    flag_presidiario,
    flag_advogado,
    flag_servidor,
    flag_visitante,
    flag_ocorrencia_10d,
    flag_ocorrencia_30d,
    flag_ocorrencia_60d,
    data_nascimento_pessoa,
    data_ultima_prisao,
    nome_mae,
    nome_pai,
    etnia,

    estado_civil,
    escolaridade,
    profissao,
    religiao,
    flag_tem_filhos,
    quantidade_filhos,
    naturalidade_municipio,
    naturalidade_uf,
    pais_origem_estrangeiro,
    cidade_origem_estrangeiro,
    necessidade_especial,
    flag_sabe_ler,
    flag_sabe_escrever,
    flag_recebe_visita,
    flag_relacao_conjugal,

    rn_pessoa,
    qtd_mesmo_id_pessoa,
    'EXCLUIDO_NA_DEDUPLICACAO_POR_ID_PESSOA' as motivo_exclusao
from vw_base_pessoa_dedup
where rn_pessoa > 1
""")

df_ponte = spark.sql("""
select distinct id_preso, id_pessoa, nome_pessoa
from vw_base_pessoa_dedup
""")

tabela = "sinp_ent_pes_p1"
df_pres.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_pres, "gold", tabela, f"{path}{tabela}")


CREATE EXTERNAL TABLE gold.sinp_ent_pes_p1 
                        (id_preso char(6),
    id_pessoa varchar(24),
    origem char(11),
    cod_documento_referencia integer,
    desc_documento_referencia varchar(27),
    documento varchar(20),
    nome_pessoa varchar(99),
    sexo_pessoa char(1),
    flag_presidiario integer,
    flag_advogado integer,
    flag_servidor integer,
    flag_visitante integer,
    flag_ocorrencia_10d integer,
    flag_ocorrencia_30d integer,
    flag_ocorrencia_60d integer,
    data_nascimento_pessoa date,
    data_ultima_prisao date,
    nome_mae varchar(90),
    nome_pai varchar(98),
    etnia char(9),
    estado_civil char(15),
    escolaridade varchar(39),
    profissao varchar(103),
    religiao char(15),
    flag_tem_filhos integer,
    quantidade_filhos char(14),
    naturalidade_municipio varchar(31),
    naturalidade_uf char(2),
    pais_origem_estrangeiro char(2),
    cidade_origem_estrangeiro varchar(21),
    necessidade_especial char(1),
    fla

In [5]:
# ============================================================
# BASE SIARHES
# ============================================================

df_base_siarhes = spark.sql("""
    select
        lpad(
            regexp_replace(
                regexp_replace(cast(cpf as string), '\\\\.0+$', ''),
                '[^0-9]',
                ''
            ),
            11,
            '0'
        ) as cpf_normalizado,
        trim(regexp_replace(coalesce(nome_servidor, ''), '\\\\s+', ' ')) as nome_pessoa,
        trim(regexp_replace(coalesce(cast(rg as string), ''), '\\\\s+', ' ')) as rg,
        dt_extracao,
        numero_funcional,
        cargo,
        categoria,
        subcategoria,
        situacao,
        tipo_vinculo,
        subempresa,
        row_number() over (
            partition by lpad(
                regexp_replace(
                    regexp_replace(cast(cpf as string), '\\\\.0+$', ''),
                    '[^0-9]',
                    ''
                ),
                11,
                '0'
            )
            order by
                case when dt_extracao is not null then 1 else 2 end,
                dt_extracao desc,
                case when coalesce(nome_servidor, '') <> '' then 1 else 2 end,
                case when coalesce(cast(rg as string), '') <> '' then 1 else 2 end,
                cast(numero_funcional as string) desc
        ) as rn
    from bronze.siarhes_servidores
""")

tabela = "tmp_base_siarhes"

df_base_siarhes.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_base_siarhes, "gold", tabela, f"{path}{tabela}")


# ============================================================
# BASE SIARHES FILTRADA
# ============================================================

df_filtrada_siarhes = spark.sql("""
    select
        cpf_normalizado,
        concat(
            substr(cpf_normalizado, 1, 3), '.',
            substr(cpf_normalizado, 4, 3), '.',
            substr(cpf_normalizado, 7, 3), '-',
            substr(cpf_normalizado, 10, 2)
        ) as cpf_formatado,
        nome_pessoa,
        rg,
        dt_extracao,
        numero_funcional,
        cargo,
        categoria,
        subcategoria,
        situacao,
        tipo_vinculo,
        subempresa
    from gold.tmp_base_siarhes
    where rn = 1
      and cpf_normalizado is not null
      and cpf_normalizado <> ''
      and cpf_normalizado <> '00000000000'
      and length(cpf_normalizado) = 11
""")

tabela = "tmp_filtrada_siarhes"

df_filtrada_siarhes.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_filtrada_siarhes, "gold", tabela, f"{path}{tabela}")


# ============================================================
# BASE PESSOA ATUAL PARA CRUZAMENTO
# ============================================================

df_base_pessoa_atual = spark.sql("""
    select
        p.*,
        case
            when p.cod_documento_referencia = 19 then
                lpad(regexp_replace(coalesce(p.documento, ''), '[^0-9]', ''), 11, '0')
            else null
        end as cpf_normalizado
    from gold.sinp_ent_pes_p1 p
""")

tabela = "tmp_base_pessoa_atual"

df_base_pessoa_atual.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_base_pessoa_atual, "gold", tabela, f"{path}{tabela}")


# ============================================================
# PESSOAS ATUALIZADAS COM FLAG_SERVIDOR = 1
# ============================================================

df_pessoa_atualizada_servidor = spark.sql("""
    select
        p.id_preso,
        p.id_pessoa,
        p.origem,
        p.cod_documento_referencia,
        p.desc_documento_referencia,
        p.documento,
        p.nome_pessoa,
        p.sexo_pessoa,
        coalesce(p.flag_presidiario, 0) as flag_presidiario,
        coalesce(p.flag_advogado, 0) as flag_advogado,
        1 as flag_servidor,
        coalesce(p.flag_visitante, 0) as flag_visitante,
        coalesce(p.flag_ocorrencia_10d, 0) as flag_ocorrencia_10d,
        coalesce(p.flag_ocorrencia_30d, 0) as flag_ocorrencia_30d,
        coalesce(p.flag_ocorrencia_60d, 0) as flag_ocorrencia_60d,
        p.data_nascimento_pessoa,
        p.data_ultima_prisao,
        p.nome_mae,
        p.nome_pai,
        p.etnia
    from gold.tmp_base_pessoa_atual p
    inner join gold.tmp_filtrada_siarhes s
        on p.cpf_normalizado = s.cpf_normalizado
""")

tabela = "tmp_pessoa_atualizada_servidor"

df_pessoa_atualizada_servidor.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_pessoa_atualizada_servidor, "gold", tabela, f"{path}{tabela}")


# ============================================================
# PESSOAS MANTIDAS
# ============================================================

df_pessoa_mantida = spark.sql("""
    select
        p.id_preso,
        p.id_pessoa,
        p.origem,
        p.cod_documento_referencia,
        p.desc_documento_referencia,
        p.documento,
        p.nome_pessoa,
        p.sexo_pessoa,
        p.flag_presidiario,
        p.flag_advogado,
        p.flag_servidor,
        p.flag_visitante,
        p.flag_ocorrencia_10d,
        p.flag_ocorrencia_30d,
        p.flag_ocorrencia_60d,
        p.data_nascimento_pessoa,
        p.data_ultima_prisao,
        p.nome_mae,
        p.nome_pai,
        p.etnia
    from gold.tmp_base_pessoa_atual p
    left join gold.tmp_filtrada_siarhes s
        on p.cpf_normalizado = s.cpf_normalizado
    where s.cpf_normalizado is null
""")

tabela = "tmp_pessoa_mantida"

df_pessoa_mantida.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_pessoa_mantida, "gold", tabela, f"{path}{tabela}")


# ============================================================
# NOVAS PESSOAS VINDAS DO SIARHES
# ============================================================

df_pessoa_nova_servidor = spark.sql("""
    select
        cast(null as string) as id_preso,
        concat('CPF_', s.cpf_normalizado) as id_pessoa,
        'SERVIDOR' as origem,
        19 as cod_documento_referencia,
        'CPF' as desc_documento_referencia,
        s.cpf_formatado as documento,
        s.nome_pessoa,
        cast(null as string) as sexo_pessoa,
        0 as flag_presidiario,
        0 as flag_advogado,
        1 as flag_servidor,
        0 as flag_visitante,
        0 as flag_ocorrencia_10d,
        0 as flag_ocorrencia_30d,
        0 as flag_ocorrencia_60d,
        cast(null as date) as data_nascimento_pessoa,
        cast(null as date) as data_ultima_prisao,
        cast(null as string) as nome_mae,
        cast(null as string) as nome_pai,
        cast(null as string) as etnia
    from gold.tmp_filtrada_siarhes s
    left join gold.tmp_base_pessoa_atual p
        on p.cpf_normalizado = s.cpf_normalizado
    where p.cpf_normalizado is null
""")

tabela = "tmp_pessoa_nova_servidor"

df_pessoa_nova_servidor.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_pessoa_nova_servidor, "gold", tabela, f"{path}{tabela}")


# ============================================================
# SAIDA FINAL ATE O UNION DOS MANTIDOS
# ============================================================

df_pessoa_shiares = spark.sql("""
    select * from gold.tmp_pessoa_mantida
    union all
    select * from gold.tmp_pessoa_atualizada_servidor
    union all
    select * from gold.tmp_pessoa_nova_servidor
""")

tabela = "sinp_ent_pessoa_shiares"

df_pessoa_shiares.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_pessoa_shiares, "gold", tabela, f"{path}{tabela}")

CREATE EXTERNAL TABLE gold.tmp_base_siarhes 
                        (cpf_normalizado char(11),
    nome_pessoa varchar(49),
    rg char(15),
    dt_extracao varchar(23),
    numero_funcional char(12),
    cargo varchar(61),
    categoria varchar(20),
    subcategoria varchar(20),
    situacao char(8),
    tipo_vinculo varchar(19),
    subempresa varchar(47),
    rn integer)
                        STORED AS PARQUET LOCATION '/data_lake/gold/intlpris/tmp_base_siarhes'
                        
DROP TABLE gold.tmp_base_siarhes 
Estatísticas atualizadas para gold.tmp_base_siarhes
CREATE EXTERNAL TABLE gold.tmp_filtrada_siarhes 
                        (cpf_normalizado char(11),
    cpf_formatado char(14),
    nome_pessoa varchar(49),
    rg char(15),
    dt_extracao varchar(23),
    numero_funcional char(12),
    cargo varchar(61),
    categoria varchar(20),
    subcategoria varchar(20),
    situacao char(8),
    tipo_vinculo varchar(19),
    subempresa varchar(47))
                      

In [6]:
# ============================================================
# BASE CAVI NORMALIZADA
# ============================================================

df_base_cavi = spark.sql("""
    select
        cast(itemnum as string) as itemnum,
        trim(regexp_replace(coalesce(cast(numeroonbase as string), ''), '\\\\s+', ' ')) as numeroonbase,
        trim(regexp_replace(coalesce(statussolicitacao, ''), '\\\\s+', ' ')) as statussolicitacao,
        trim(regexp_replace(coalesce(resultanalise, ''), '\\\\s+', ' ')) as resultanalise,

        trim(regexp_replace(coalesce(cast(telcelular as string), ''), '\\\\s+', ' ')) as telcelular,
        trim(regexp_replace(coalesce(nomeinteressado, ''), '\\\\s+', ' ')) as nome_pessoa,
        lower(trim(regexp_replace(coalesce(emailinteressado, ''), '\\\\s+', ' '))) as emailinteressado,

        lpad(
            regexp_replace(
                regexp_replace(cast(cpfinteressado as string), '\\\\.0+$', ''),
                '[^0-9]',
                ''
            ),
            11,
            '0'
        ) as cpf_normalizado,

        trim(regexp_replace(coalesce(cast(telresidencial as string), ''), '\\\\s+', ' ')) as telresidencial,

        trim(
            regexp_replace(
                regexp_replace(coalesce(cast(rginteressado as string), ''), '\\\\.0+$', ''),
                '\\\\s+',
                ' '
            )
        ) as rg,

        trim(regexp_replace(coalesce(orgaoemissor, ''), '\\\\s+', ' ')) as orgaoemissor,
        trim(regexp_replace(coalesce(estadoemissor, ''), '\\\\s+', ' ')) as estadoemissor,
        trim(regexp_replace(coalesce(sexo, ''), '\\\\s+', ' ')) as sexo_pessoa,
        trim(regexp_replace(coalesce(profissao, ''), '\\\\s+', ' ')) as profissao,

        lpad(
            regexp_replace(
                regexp_replace(cast(cpfdentento as string), '\\\\.0+$', ''),
                '[^0-9]',
                ''
            ),
            11,
            '0'
        ) as cpf_detento_normalizado,

        trim(regexp_replace(coalesce(nomedetento, ''), '\\\\s+', ' ')) as nome_detento,

        trim(
            regexp_replace(
                regexp_replace(coalesce(cast(rgdetento as string), ''), '\\\\.0+$', ''),
                '\\\\s+',
                ' '
            )
        ) as rg_detento,

        trim(regexp_replace(coalesce(nomemae, ''), '\\\\s+', ' ')) as nome_mae_detento,
        trim(regexp_replace(coalesce(tipovisita, ''), '\\\\s+', ' ')) as tipovisita,
        trim(regexp_replace(coalesce(vinculodetento, ''), '\\\\s+', ' ')) as vinculodetento,
        trim(regexp_replace(coalesce(grauparentesco, ''), '\\\\s+', ' ')) as grauparentesco,
        trim(regexp_replace(coalesce(ctprocesso, ''), '\\\\s+', ' ')) as ctprocesso,
        trim(regexp_replace(coalesce(ctfilaatual, ''), '\\\\s+', ' ')) as ctfilaatual,
        trim(regexp_replace(coalesce(ctfilaorigem, ''), '\\\\s+', ' ')) as ctfilaorigem,
        trim(regexp_replace(coalesce(setorresponsavel, ''), '\\\\s+', ' ')) as setorresponsavel,
        trim(regexp_replace(coalesce(unidadeprisional, ''), '\\\\s+', ' ')) as unidadeprisional
    from bronze.obsejus_cc_requerimentoscavi
""")

tabela = "tmp_base_cavi"

df_base_cavi.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_base_cavi, "gold", tabela, f"{path}{tabela}")


# ============================================================
# BASE CAVI RANKEADA
# ============================================================

df_base_cavi_rank = spark.sql("""
    select
        *,
        row_number() over (
            partition by cpf_normalizado
            order by
                case when itemnum is not null and itemnum <> '' then 1 else 2 end,
                cast(itemnum as bigint) desc,
                case when coalesce(nome_pessoa, '') <> '' then 1 else 2 end,
                case when coalesce(rg, '') <> '' then 1 else 2 end,
                case when coalesce(emailinteressado, '') <> '' then 1 else 2 end
        ) as rn
    from gold.tmp_base_cavi
""")

tabela = "tmp_base_cavi_rank"

df_base_cavi_rank.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_base_cavi_rank, "gold", tabela, f"{path}{tabela}")


# ============================================================
# BASE CAVI FILTRADA
# ============================================================

df_filtrada_cavi = spark.sql("""
    select
        cpf_normalizado,
        concat(
            substr(cpf_normalizado, 1, 3), '.',
            substr(cpf_normalizado, 4, 3), '.',
            substr(cpf_normalizado, 7, 3), '-',
            substr(cpf_normalizado, 10, 2)
        ) as cpf_formatado,
        nome_pessoa,
        rg,
        sexo_pessoa,
        orgaoemissor,
        estadoemissor,
        profissao,
        telcelular,
        telresidencial,
        emailinteressado,
        itemnum,
        numeroonbase,
        statussolicitacao,
        resultanalise,
        tipovisita,
        vinculodetento,
        grauparentesco,
        unidadeprisional
    from gold.tmp_base_cavi_rank
    where rn = 1
      and cpf_normalizado is not null
      and cpf_normalizado <> ''
      and cpf_normalizado <> '00000000000'
      and length(cpf_normalizado) = 11
""")

tabela = "tmp_filtrada_cavi"

df_filtrada_cavi.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_filtrada_cavi, "gold", tabela, f"{path}{tabela}")


# ============================================================
# BASE PESSOA ATUAL PARA CRUZAMENTO
# ============================================================

df_base_pessoa_atual = spark.sql("""
    select
        p.*,
        case
            when p.cod_documento_referencia = 19 then
                lpad(regexp_replace(coalesce(p.documento, ''), '[^0-9]', ''), 11, '0')
            else null
        end as cpf_normalizado
    from gold.sinp_ent_pessoa_shiares p
""")

tabela = "tmp_base_pessoa_atual_cavi"

df_base_pessoa_atual.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_base_pessoa_atual, "gold", tabela, f"{path}{tabela}")


# ============================================================
# PESSOAS ATUALIZADAS COM FLAG_VISITANTE = 1
# ============================================================

df_pessoa_atualizada_visitante = spark.sql("""
    select
        p.id_preso,
        p.id_pessoa,
        p.origem,
        p.cod_documento_referencia,
        p.desc_documento_referencia,
        p.documento,
        p.nome_pessoa,
        case
            when coalesce(p.sexo_pessoa, '') <> '' then p.sexo_pessoa
            else c.sexo_pessoa
        end as sexo_pessoa,
        coalesce(p.flag_presidiario, 0) as flag_presidiario,
        coalesce(p.flag_advogado, 0) as flag_advogado,
        coalesce(p.flag_servidor, 0) as flag_servidor,
        1 as flag_visitante,
        coalesce(p.flag_ocorrencia_10d, 0) as flag_ocorrencia_10d,
        coalesce(p.flag_ocorrencia_30d, 0) as flag_ocorrencia_30d,
        coalesce(p.flag_ocorrencia_60d, 0) as flag_ocorrencia_60d,
        p.data_nascimento_pessoa,
        p.data_ultima_prisao,
        p.nome_mae,
        p.nome_pai,
        p.etnia
    from gold.tmp_base_pessoa_atual_cavi p
    inner join gold.tmp_filtrada_cavi c
        on p.cpf_normalizado = c.cpf_normalizado
""")

tabela = "tmp_pessoa_atualizada_visitante"

df_pessoa_atualizada_visitante.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_pessoa_atualizada_visitante, "gold", tabela, f"{path}{tabela}")


# ============================================================
# PESSOAS MANTIDAS
# ============================================================

df_pessoa_mantida_visitante = spark.sql("""
    select
        p.id_preso,
        p.id_pessoa,
        p.origem,
        p.cod_documento_referencia,
        p.desc_documento_referencia,
        p.documento,
        p.nome_pessoa,
        p.sexo_pessoa,
        p.flag_presidiario,
        p.flag_advogado,
        p.flag_servidor,
        p.flag_visitante,
        p.flag_ocorrencia_10d,
        p.flag_ocorrencia_30d,
        p.flag_ocorrencia_60d,
        p.data_nascimento_pessoa,
        p.data_ultima_prisao,
        p.nome_mae,
        p.nome_pai,
        p.etnia
    from gold.tmp_base_pessoa_atual_cavi p
    left join gold.tmp_filtrada_cavi c
        on p.cpf_normalizado = c.cpf_normalizado
    where c.cpf_normalizado is null
""")

tabela = "tmp_pessoa_mantida_visitante"

df_pessoa_mantida_visitante.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_pessoa_mantida_visitante, "gold", tabela, f"{path}{tabela}")


# ============================================================
# NOVAS PESSOAS VINDAS DO CAVI
# ============================================================

df_pessoa_nova_visitante = spark.sql("""
    select
        cast(null as string) as id_preso,
        concat('CPF_', c.cpf_normalizado) as id_pessoa,
        'VISITANTE' as origem,
        19 as cod_documento_referencia,
        'CPF' as desc_documento_referencia,
        c.cpf_formatado as documento,
        c.nome_pessoa,
        c.sexo_pessoa,
        0 as flag_presidiario,
        0 as flag_advogado,
        0 as flag_servidor,
        1 as flag_visitante,
        0 as flag_ocorrencia_10d,
        0 as flag_ocorrencia_30d,
        0 as flag_ocorrencia_60d,
        cast(null as date) as data_nascimento_pessoa,
        cast(null as date) as data_ultima_prisao,
        cast(null as string) as nome_mae,
        cast(null as string) as nome_pai,
        cast(null as string) as etnia
    from gold.tmp_filtrada_cavi c
    left join gold.tmp_base_pessoa_atual_cavi p
        on p.cpf_normalizado = c.cpf_normalizado
    where p.cpf_normalizado is null
""")

tabela = "tmp_pessoa_nova_visitante"

df_pessoa_nova_visitante.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_pessoa_nova_visitante, "gold", tabela, f"{path}{tabela}")


# ============================================================
# SAIDA FINAL DAS PESSOAS
# ============================================================

df_pessoa_final_visitante = spark.sql("""
    select * from gold.tmp_pessoa_mantida_visitante
    union all
    select * from gold.tmp_pessoa_atualizada_visitante
    union all
    select * from gold.tmp_pessoa_nova_visitante
""")


tabela = "df_pessoa_nova_visitante"

df_pessoa_final_visitante.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_pessoa_final_visitante, "gold", tabela, f"{path}{tabela}")


CREATE EXTERNAL TABLE gold.tmp_base_cavi 
                        (itemnum char(6),
    numeroonbase char(12),
    statussolicitacao char(13),
    resultanalise varchar(22),
    telcelular char(11),
    nome_pessoa varchar(58),
    emailinteressado varchar(44),
    cpf_normalizado char(11),
    telresidencial char(10),
    rg char(11),
    orgaoemissor char(12),
    estadoemissor char(2),
    sexo_pessoa varchar(21),
    profissao varchar(51),
    cpf_detento_normalizado char(11),
    nome_detento varchar(69),
    rg_detento char(11),
    nome_mae_detento varchar(82),
    tipovisita varchar(23),
    vinculodetento varchar(20),
    grauparentesco char(14),
    ctprocesso varchar(20),
    ctfilaatual varchar(33),
    ctfilaorigem varchar(33),
    setorresponsavel varchar(25),
    unidadeprisional varchar(59))
                        STORED AS PARQUET LOCATION '/data_lake/gold/intlpris/tmp_base_cavi'
                        
DROP TABLE gold.tmp_base_cavi 
Estatísticas atualizadas para gol

In [7]:
# ============================================================
# BASE ADVOGADO
# ============================================================

df_base_advogado = spark.sql("""
    select
        cast(id as string) as id_advogado_origem,
        trim(regexp_replace(coalesce(nome, ''), '\\\\s+', ' ')) as nome_pessoa,
        upper(trim(regexp_replace(coalesce(estado, ''), '\\\\s+', ' '))) as estado_oab,
        upper(trim(regexp_replace(coalesce(oab, ''), '\\\\s+', ' '))) as oab_bruta,
        upper(regexp_replace(coalesce(oab, ''), '[^0-9A-Za-z]', '')) as oab_normalizada,
        concat(
            'OAB_',
            upper(trim(regexp_replace(coalesce(estado, ''), '\\\\s+', ' '))),
            '_',
            upper(regexp_replace(coalesce(oab, ''), '[^0-9A-Za-z]', ''))
        ) as id_pessoa,
        concat(
            upper(regexp_replace(coalesce(oab, ''), '[^0-9A-Za-z]', '')),
            '/',
            upper(trim(regexp_replace(coalesce(estado, ''), '\\\\s+', ' ')))
        ) as documento_oab,
        row_number() over (
            partition by
                upper(trim(regexp_replace(coalesce(estado, ''), '\\\\s+', ' '))),
                upper(regexp_replace(coalesce(oab, ''), '[^0-9A-Za-z]', ''))
            order by
                case when coalesce(nome, '') <> '' then 1 else 2 end,
                cast(id as string) desc
        ) as rn
    from bronze.livros_acesso_unidade_advogado
""")

tabela = "tmp_base_advogado"

df_base_advogado.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_base_advogado, "gold", tabela, f"{path}{tabela}")


# ============================================================
# BASE ADVOGADO FILTRADA
# ============================================================

df_filtrada_advogado = spark.sql("""
    select
        id_advogado_origem,
        id_pessoa,
        nome_pessoa,
        estado_oab,
        oab_bruta,
        oab_normalizada,
        documento_oab
    from gold.tmp_base_advogado
    where rn = 1
      and estado_oab is not null
      and estado_oab <> ''
      and oab_normalizada is not null
      and oab_normalizada <> ''
""")

tabela = "tmp_filtrada_advogado"

df_filtrada_advogado.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_filtrada_advogado, "gold", tabela, f"{path}{tabela}")


# ============================================================
# BASE PESSOA ATUAL PARA CRUZAMENTO
# ============================================================

df_base_pessoa_atual_advogado = spark.sql("""
    select
        p.*,
        case
            when p.id_pessoa like 'OAB_%' then p.id_pessoa
            else null
        end as id_pessoa_advogado
    from gold.df_pessoa_nova_visitante p
""")

tabela = "tmp_base_pessoa_atual_advogado"

df_base_pessoa_atual_advogado.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_base_pessoa_atual_advogado, "gold", tabela, f"{path}{tabela}")


# ============================================================
# PESSOAS ATUALIZADAS COM FLAG_ADVOGADO = 1
# ============================================================

df_pessoa_atualizada_advogado = spark.sql("""
    select
        p.id_preso,
        p.id_pessoa,
        p.origem,
        p.cod_documento_referencia,
        case
            when coalesce(p.desc_documento_referencia, '') <> '' then p.desc_documento_referencia
            else 'OAB'
        end as desc_documento_referencia,
        case
            when coalesce(p.documento, '') <> '' then p.documento
            else a.documento_oab
        end as documento,
        case
            when coalesce(p.nome_pessoa, '') <> '' then p.nome_pessoa
            else a.nome_pessoa
        end as nome_pessoa,
        p.sexo_pessoa,
        coalesce(p.flag_presidiario, 0) as flag_presidiario,
        1 as flag_advogado,
        coalesce(p.flag_servidor, 0) as flag_servidor,
        coalesce(p.flag_visitante, 0) as flag_visitante,
        coalesce(p.flag_ocorrencia_10d, 0) as flag_ocorrencia_10d,
        coalesce(p.flag_ocorrencia_30d, 0) as flag_ocorrencia_30d,
        coalesce(p.flag_ocorrencia_60d, 0) as flag_ocorrencia_60d,
        p.data_nascimento_pessoa,
        p.data_ultima_prisao,
        p.nome_mae,
        p.nome_pai,
        p.etnia
    from gold.tmp_base_pessoa_atual_advogado p
    inner join gold.tmp_filtrada_advogado a
        on p.id_pessoa_advogado = a.id_pessoa
""")

tabela = "tmp_pessoa_atualizada_advogado"

df_pessoa_atualizada_advogado.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_pessoa_atualizada_advogado, "gold", tabela, f"{path}{tabela}")


# ============================================================
# PESSOAS MANTIDAS
# ============================================================

df_pessoa_mantida_advogado = spark.sql("""
    select
        p.id_preso,
        p.id_pessoa,
        p.origem,
        p.cod_documento_referencia,
        p.desc_documento_referencia,
        p.documento,
        p.nome_pessoa,
        p.sexo_pessoa,
        p.flag_presidiario,
        p.flag_advogado,
        p.flag_servidor,
        p.flag_visitante,
        p.flag_ocorrencia_10d,
        p.flag_ocorrencia_30d,
        p.flag_ocorrencia_60d,
        p.data_nascimento_pessoa,
        p.data_ultima_prisao,
        p.nome_mae,
        p.nome_pai,
        p.etnia
    from gold.tmp_base_pessoa_atual_advogado p
    left join gold.tmp_filtrada_advogado a
        on p.id_pessoa_advogado = a.id_pessoa
    where a.id_pessoa is null
""")

tabela = "tmp_pessoa_mantida_advogado"

df_pessoa_mantida_advogado.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_pessoa_mantida_advogado, "gold", tabela, f"{path}{tabela}")


# ============================================================
# NOVAS PESSOAS VINDAS DO CADASTRO DE ADVOGADO
# ============================================================

df_pessoa_nova_advogado = spark.sql("""
    select
        cast(null as string) as id_preso,
        a.id_pessoa,
        'ADVOGADO' as origem,
        cast(null as int) as cod_documento_referencia,
        'OAB' as desc_documento_referencia,
        a.documento_oab as documento,
        a.nome_pessoa,
        cast(null as string) as sexo_pessoa,
        0 as flag_presidiario,
        1 as flag_advogado,
        0 as flag_servidor,
        0 as flag_visitante,
        0 as flag_ocorrencia_10d,
        0 as flag_ocorrencia_30d,
        0 as flag_ocorrencia_60d,
        cast(null as date) as data_nascimento_pessoa,
        cast(null as date) as data_ultima_prisao,
        cast(null as string) as nome_mae,
        cast(null as string) as nome_pai,
        cast(null as string) as etnia
    from gold.tmp_filtrada_advogado a
    left join gold.tmp_base_pessoa_atual_advogado p
        on p.id_pessoa_advogado = a.id_pessoa
    where p.id_pessoa_advogado is null
""")

tabela = "tmp_pessoa_nova_advogado"

df_pessoa_nova_advogado.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_pessoa_nova_advogado, "gold", tabela, f"{path}{tabela}")


# ============================================================
# SAIDA FINAL DAS PESSOAS
# ============================================================

df_pessoa_final_advogado = spark.sql("""
    select * from gold.tmp_pessoa_mantida_advogado
    union all
    select * from gold.tmp_pessoa_atualizada_advogado
    union all
    select * from gold.tmp_pessoa_nova_advogado
""")


tabela = "df_pessoa_final_advogado"

df_pessoa_final_advogado.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_pessoa_final_advogado, "gold", tabela, f"{path}{tabela}")


CREATE EXTERNAL TABLE gold.tmp_base_advogado 
                        (id_advogado_origem char(4),
    nome_pessoa varchar(53),
    estado_oab char(2),
    oab_bruta char(8),
    oab_normalizada char(8),
    id_pessoa char(13),
    documento_oab char(9),
    rn integer)
                        STORED AS PARQUET LOCATION '/data_lake/gold/intlpris/tmp_base_advogado'
                        
DROP TABLE gold.tmp_base_advogado 
Estatísticas atualizadas para gold.tmp_base_advogado
CREATE EXTERNAL TABLE gold.tmp_filtrada_advogado 
                        (id_advogado_origem char(4),
    id_pessoa char(13),
    nome_pessoa varchar(53),
    estado_oab char(2),
    oab_bruta char(8),
    oab_normalizada char(8),
    documento_oab char(9))
                        STORED AS PARQUET LOCATION '/data_lake/gold/intlpris/tmp_filtrada_advogado'
                        
DROP TABLE gold.tmp_filtrada_advogado 
Estatísticas atualizadas para gold.tmp_filtrada_advogado
CREATE EXTERNAL TABLE gold.tmp_base_pess

In [8]:
import os

# ============================================================
# LIMPEZA DEFENSIVA DAS TABELAS TEMPORARIAS - PESSOAS FAMILIARES
# ============================================================

spark.sql("drop table if exists gold.tmp_base_visitafamiliar")
spark.sql("drop table if exists gold.tmp_base_visitafamiliar_rank")
spark.sql("drop table if exists gold.tmp_filtrada_visitafamiliar")
spark.sql("drop table if exists gold.tmp_base_pessoa_atual_familiar")
spark.sql("drop table if exists gold.tmp_pessoa_atualizada_familiar")
spark.sql("drop table if exists gold.tmp_pessoa_mantida_familiar")
spark.sql("drop table if exists gold.tmp_pessoa_nova_familiar")
spark.sql("drop table if exists gold.df_pessoa_final_familiar")

os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_base_visitafamiliar >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_base_visitafamiliar_rank >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_filtrada_visitafamiliar >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_base_pessoa_atual_familiar >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_pessoa_atualizada_familiar >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_pessoa_mantida_familiar >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_pessoa_nova_familiar >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}df_pessoa_final_familiar >/dev/null 2>&1")

spark.catalog.clearCache()
spark.sql("refresh table gold.df_pessoa_final_advogado")


# ============================================================
# BASE VISITA FAMILIAR
# ============================================================

df_base_visitafamiliar = spark.sql("""
    select
        cast(id as string) as id_vinculo_origem,
        trim(regexp_replace(coalesce(nome, ''), '\\\\s+', ' ')) as nome_pessoa,
        trim(regexp_replace(coalesce(documento, ''), '\\\\s+', ' ')) as documento_original,
        trim(regexp_replace(coalesce(telefone, ''), '\\\\s+', ' ')) as telefone,
        lpad(
            regexp_replace(
                regexp_replace(cast(documento as string), '\\\\.0+$', ''),
                '[^0-9]',
                ''
            ),
            11,
            '0'
        ) as cpf_normalizado,
        upper(regexp_replace(coalesce(documento, ''), '[^0-9A-Za-z]', '')) as documento_normalizado,
        case
            when length(regexp_replace(coalesce(documento, ''), '[^0-9]', '')) = 11 then
                concat(
                    'CPF_',
                    lpad(regexp_replace(coalesce(documento, ''), '[^0-9]', ''), 11, '0')
                )
            else
                concat(
                    'DOC_',
                    upper(regexp_replace(coalesce(documento, ''), '[^0-9A-Za-z]', ''))
                )
        end as chave_documento
    from bronze.livros_acesso_unidade_visitafamiliar
""")

tabela = "tmp_base_visitafamiliar"

df_base_visitafamiliar.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_base_visitafamiliar, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_visitafamiliar")


# ============================================================
# BASE VISITA FAMILIAR RANKEADA
# ============================================================

df_base_visitafamiliar_rank = spark.sql("""
    select
        *,
        row_number() over (
            partition by chave_documento
            order by
                case when coalesce(nome_pessoa, '') <> '' then 1 else 2 end,
                case when coalesce(telefone, '') <> '' then 1 else 2 end,
                id_vinculo_origem desc
        ) as rn
    from gold.tmp_base_visitafamiliar
""")

tabela = "tmp_base_visitafamiliar_rank"

df_base_visitafamiliar_rank.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_base_visitafamiliar_rank, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_visitafamiliar_rank")


# ============================================================
# BASE FAMILIAR FILTRADA
# ============================================================

df_filtrada_visitafamiliar = spark.sql("""
    select
        id_vinculo_origem,
        nome_pessoa,
        documento_original,
        telefone,
        cpf_normalizado,
        documento_normalizado,
        chave_documento,
        case
            when chave_documento like 'CPF_%' then concat(
                substr(cpf_normalizado, 1, 3), '.',
                substr(cpf_normalizado, 4, 3), '.',
                substr(cpf_normalizado, 7, 3), '-',
                substr(cpf_normalizado, 10, 2)
            )
            else documento_original
        end as documento_formatado,
        case
            when chave_documento like 'CPF_%' then 19
            else cast(null as int)
        end as cod_documento_referencia,
        case
            when chave_documento like 'CPF_%' then 'CPF'
            else 'DOCUMENTO'
        end as desc_documento_referencia
    from gold.tmp_base_visitafamiliar_rank
    where rn = 1
      and chave_documento is not null
      and chave_documento <> 'DOC_'
      and chave_documento <> 'CPF_00000000000'
""")

tabela = "tmp_filtrada_visitafamiliar"

df_filtrada_visitafamiliar.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_filtrada_visitafamiliar, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_filtrada_visitafamiliar")


# ============================================================
# BASE PESSOA ATUAL PARA CRUZAMENTO
# ============================================================

df_base_pessoa_atual_familiar = spark.sql("""
    select
        p.*,
        case
            when p.cod_documento_referencia = 19 then
                concat(
                    'CPF_',
                    lpad(regexp_replace(coalesce(p.documento, ''), '[^0-9]', ''), 11, '0')
                )
            else
                concat(
                    'DOC_',
                    upper(regexp_replace(coalesce(p.documento, ''), '[^0-9A-Za-z]', ''))
                )
        end as chave_documento
    from gold.df_pessoa_final_advogado p
""")

tabela = "tmp_base_pessoa_atual_familiar"

df_base_pessoa_atual_familiar.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_base_pessoa_atual_familiar, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_pessoa_atual_familiar")


# ============================================================
# PESSOAS ATUALIZADAS COM FLAG_VISITANTE = 1
# ============================================================

df_pessoa_atualizada_familiar = spark.sql("""
    select
        p.id_preso,
        p.id_pessoa,
        p.origem,
        case
            when p.cod_documento_referencia is not null then p.cod_documento_referencia
            else f.cod_documento_referencia
        end as cod_documento_referencia,
        case
            when coalesce(p.desc_documento_referencia, '') <> '' then p.desc_documento_referencia
            else f.desc_documento_referencia
        end as desc_documento_referencia,
        case
            when coalesce(p.documento, '') <> '' then p.documento
            else f.documento_formatado
        end as documento,
        case
            when coalesce(p.nome_pessoa, '') <> '' then p.nome_pessoa
            else f.nome_pessoa
        end as nome_pessoa,
        p.sexo_pessoa,
        coalesce(p.flag_presidiario, 0) as flag_presidiario,
        coalesce(p.flag_advogado, 0) as flag_advogado,
        coalesce(p.flag_servidor, 0) as flag_servidor,
        1 as flag_visitante,
        coalesce(p.flag_ocorrencia_10d, 0) as flag_ocorrencia_10d,
        coalesce(p.flag_ocorrencia_30d, 0) as flag_ocorrencia_30d,
        coalesce(p.flag_ocorrencia_60d, 0) as flag_ocorrencia_60d,
        p.data_nascimento_pessoa,
        p.data_ultima_prisao,
        p.nome_mae,
        p.nome_pai,
        p.etnia
    from gold.tmp_base_pessoa_atual_familiar p
    inner join gold.tmp_filtrada_visitafamiliar f
        on p.chave_documento = f.chave_documento
""")

tabela = "tmp_pessoa_atualizada_familiar"

df_pessoa_atualizada_familiar.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_pessoa_atualizada_familiar, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_pessoa_atualizada_familiar")


# ============================================================
# PESSOAS MANTIDAS
# ============================================================

df_pessoa_mantida_familiar = spark.sql("""
    select
        p.id_preso,
        p.id_pessoa,
        p.origem,
        p.cod_documento_referencia,
        p.desc_documento_referencia,
        p.documento,
        p.nome_pessoa,
        p.sexo_pessoa,
        p.flag_presidiario,
        p.flag_advogado,
        p.flag_servidor,
        p.flag_visitante,
        p.flag_ocorrencia_10d,
        p.flag_ocorrencia_30d,
        p.flag_ocorrencia_60d,
        p.data_nascimento_pessoa,
        p.data_ultima_prisao,
        p.nome_mae,
        p.nome_pai,
        p.etnia
    from gold.tmp_base_pessoa_atual_familiar p
    left join gold.tmp_filtrada_visitafamiliar f
        on p.chave_documento = f.chave_documento
    where f.chave_documento is null
""")

tabela = "tmp_pessoa_mantida_familiar"

df_pessoa_mantida_familiar.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_pessoa_mantida_familiar, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_pessoa_mantida_familiar")


# ============================================================
# NOVAS PESSOAS VINDAS DA VISITA FAMILIAR
# ============================================================

df_pessoa_nova_familiar = spark.sql("""
    select
        cast(null as string) as id_preso,
        f.chave_documento as id_pessoa,
        'VISITA_FAMILIAR' as origem,
        f.cod_documento_referencia,
        f.desc_documento_referencia,
        f.documento_formatado as documento,
        f.nome_pessoa,
        cast(null as string) as sexo_pessoa,
        0 as flag_presidiario,
        0 as flag_advogado,
        0 as flag_servidor,
        1 as flag_visitante,
        0 as flag_ocorrencia_10d,
        0 as flag_ocorrencia_30d,
        0 as flag_ocorrencia_60d,
        cast(null as date) as data_nascimento_pessoa,
        cast(null as date) as data_ultima_prisao,
        cast(null as string) as nome_mae,
        cast(null as string) as nome_pai,
        cast(null as string) as etnia
    from gold.tmp_filtrada_visitafamiliar f
    left join gold.tmp_base_pessoa_atual_familiar p
        on p.chave_documento = f.chave_documento
    where p.chave_documento is null
""")

tabela = "tmp_pessoa_nova_familiar"

df_pessoa_nova_familiar.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_pessoa_nova_familiar, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_pessoa_nova_familiar")


# ============================================================
# SAIDA FINAL DAS PESSOAS
# ============================================================

df_pessoa_final_familiar = spark.sql("""
    select * from gold.tmp_pessoa_mantida_familiar
    union all
    select * from gold.tmp_pessoa_atualizada_familiar
    union all
    select * from gold.tmp_pessoa_nova_familiar
""")

tabela = "df_pessoa_final_familiar"

df_pessoa_final_familiar.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_pessoa_final_familiar, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres(f"gold.{tabela}", "id_pessoa")


# ============================================================
# VALIDACAO RAPIDA
# ============================================================

spark.sql("""
select
    count(*) as total_pessoas,
    sum(case when coalesce(flag_visitante, 0) = 1 then 1 else 0 end) as total_visitantes
from gold.df_pessoa_final_familiar
""").show(truncate=False)

CREATE EXTERNAL TABLE gold.tmp_base_visitafamiliar 
                        (id_vinculo_origem char(6),
    nome_pessoa varchar(86),
    documento_original char(11),
    telefone char(15),
    cpf_normalizado char(11),
    documento_normalizado char(11),
    chave_documento char(15))
                        STORED AS PARQUET LOCATION '/data_lake/gold/intlpris/tmp_base_visitafamiliar'
                        
DROP TABLE gold.tmp_base_visitafamiliar 
Estatísticas atualizadas para gold.tmp_base_visitafamiliar
CREATE EXTERNAL TABLE gold.tmp_base_visitafamiliar_rank 
                        (id_vinculo_origem char(6),
    nome_pessoa varchar(86),
    documento_original char(11),
    telefone char(15),
    cpf_normalizado char(11),
    documento_normalizado char(11),
    chave_documento char(15),
    rn integer)
                        STORED AS PARQUET LOCATION '/data_lake/gold/intlpris/tmp_base_visitafamiliar_rank'
                        
DROP TABLE gold.tmp_base_visitafamiliar_rank 
Esta

In [9]:
import os

# ============================================================
# LIMPEZA DEFENSIVA DAS TABELAS TEMPORARIAS - PESSOAS VISITA RELIGIOSA
# ============================================================

spark.sql("drop table if exists gold.tmp_base_visitareligiosa")
spark.sql("drop table if exists gold.tmp_base_visitareligiosa_rank")
spark.sql("drop table if exists gold.tmp_filtrada_visitareligiosa")
spark.sql("drop table if exists gold.tmp_base_pessoa_atual_religiosa")
spark.sql("drop table if exists gold.tmp_pessoa_atualizada_religiosa")
spark.sql("drop table if exists gold.tmp_pessoa_mantida_religiosa")
spark.sql("drop table if exists gold.tmp_pessoa_nova_religiosa")
spark.sql("drop table if exists gold.df_pessoa_final_religiosa")

os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_base_visitareligiosa >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_base_visitareligiosa_rank >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_filtrada_visitareligiosa >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_base_pessoa_atual_religiosa >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_pessoa_atualizada_religiosa >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_pessoa_mantida_religiosa >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_pessoa_nova_religiosa >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}df_pessoa_final_religiosa >/dev/null 2>&1")

spark.catalog.clearCache()
spark.sql("refresh table gold.df_pessoa_final_familiar")


# ============================================================
# BASE VISITA RELIGIOSA
# ============================================================

df_base_visitareligiosa = spark.sql("""
    select
        cast(id as string) as id_visitante_religioso_origem,
        trim(regexp_replace(coalesce(nome, ''), '\\\\s+', ' ')) as nome_pessoa,
        trim(regexp_replace(coalesce(instituicao, ''), '\\\\s+', ' ')) as instituicao,
        trim(regexp_replace(coalesce(documento, ''), '\\\\s+', ' ')) as documento_original,
        cast(presidio_id as string) as presidio_id_origem,

        regexp_replace(
            regexp_replace(cast(documento as string), '\\\\.0+$', ''),
            '[^0-9]',
            ''
        ) as documento_digitos,

        lpad(
            regexp_replace(
                regexp_replace(cast(documento as string), '\\\\.0+$', ''),
                '[^0-9]',
                ''
            ),
            11,
            '0'
        ) as cpf_normalizado,

        upper(regexp_replace(coalesce(documento, ''), '[^0-9A-Za-z]', '')) as documento_normalizado,

        case
            when length(
                regexp_replace(
                    regexp_replace(cast(documento as string), '\\\\.0+$', ''),
                    '[^0-9]',
                    ''
                )
            ) between 1 and 11 then
                concat(
                    'CPF_',
                    lpad(
                        regexp_replace(
                            regexp_replace(cast(documento as string), '\\\\.0+$', ''),
                            '[^0-9]',
                            ''
                        ),
                        11,
                        '0'
                    )
                )
            else
                concat(
                    'DOC_',
                    upper(regexp_replace(coalesce(documento, ''), '[^0-9A-Za-z]', ''))
                )
        end as chave_documento
    from bronze.livros_acesso_unidade_visitareligiosa
""")

tabela = "tmp_base_visitareligiosa"

df_base_visitareligiosa.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_visitareligiosa, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_visitareligiosa")


# ============================================================
# BASE VISITA RELIGIOSA RANKEADA
# ============================================================

df_base_visitareligiosa_rank = spark.sql("""
    select
        *,
        row_number() over (
            partition by chave_documento
            order by
                case when coalesce(nome_pessoa, '') <> '' then 1 else 2 end,
                case when coalesce(instituicao, '') <> '' then 1 else 2 end,
                case when coalesce(presidio_id_origem, '') <> '' then 1 else 2 end,
                id_visitante_religioso_origem desc
        ) as rn
    from gold.tmp_base_visitareligiosa
""")

tabela = "tmp_base_visitareligiosa_rank"

df_base_visitareligiosa_rank.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_visitareligiosa_rank, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_visitareligiosa_rank")


# ============================================================
# BASE RELIGIOSA FILTRADA
# ============================================================

df_filtrada_visitareligiosa = spark.sql("""
    select
        id_visitante_religioso_origem,
        nome_pessoa,
        instituicao,
        documento_original,
        documento_digitos,
        cpf_normalizado,
        documento_normalizado,
        chave_documento,
        presidio_id_origem,

        case
            when length(documento_digitos) between 1 and 11 then concat(
                substr(cpf_normalizado, 1, 3), '.',
                substr(cpf_normalizado, 4, 3), '.',
                substr(cpf_normalizado, 7, 3), '-',
                substr(cpf_normalizado, 10, 2)
            )
            else documento_original
        end as documento_formatado,

        case
            when length(documento_digitos) between 1 and 11 then 19
            else cast(null as int)
        end as cod_documento_referencia,

        case
            when length(documento_digitos) between 1 and 11 then 'CPF'
            else 'DOCUMENTO'
        end as desc_documento_referencia
    from gold.tmp_base_visitareligiosa_rank
    where rn = 1
      and chave_documento is not null
      and chave_documento <> 'DOC_'
      and chave_documento <> 'CPF_00000000000'
""")

tabela = "tmp_filtrada_visitareligiosa"

df_filtrada_visitareligiosa.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_filtrada_visitareligiosa, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_filtrada_visitareligiosa")


# ============================================================
# BASE PESSOA ATUAL PARA CRUZAMENTO
# ============================================================

df_base_pessoa_atual_religiosa = spark.sql("""
    select
        p.*,
        regexp_replace(coalesce(p.documento, ''), '[^0-9]', '') as documento_digitos,
        case
            when length(regexp_replace(coalesce(p.documento, ''), '[^0-9]', '')) between 1 and 11 then
                concat(
                    'CPF_',
                    lpad(regexp_replace(coalesce(p.documento, ''), '[^0-9]', ''), 11, '0')
                )
            else
                concat(
                    'DOC_',
                    upper(regexp_replace(coalesce(p.documento, ''), '[^0-9A-Za-z]', ''))
                )
        end as chave_documento
    from gold.df_pessoa_final_familiar p
""")

tabela = "tmp_base_pessoa_atual_religiosa"

df_base_pessoa_atual_religiosa.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_pessoa_atual_religiosa, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_pessoa_atual_religiosa")


# ============================================================
# PESSOAS ATUALIZADAS COM FLAG_VISITANTE = 1
# ============================================================

df_pessoa_atualizada_religiosa = spark.sql("""
    select
        p.id_preso,
        p.id_pessoa,
        p.origem,
        case
            when p.cod_documento_referencia is not null then p.cod_documento_referencia
            else r.cod_documento_referencia
        end as cod_documento_referencia,
        case
            when coalesce(p.desc_documento_referencia, '') <> '' then p.desc_documento_referencia
            else r.desc_documento_referencia
        end as desc_documento_referencia,
        case
            when coalesce(p.documento, '') <> '' then p.documento
            else r.documento_formatado
        end as documento,
        case
            when coalesce(p.nome_pessoa, '') <> '' then p.nome_pessoa
            else r.nome_pessoa
        end as nome_pessoa,
        p.sexo_pessoa,
        coalesce(p.flag_presidiario, 0) as flag_presidiario,
        coalesce(p.flag_advogado, 0) as flag_advogado,
        coalesce(p.flag_servidor, 0) as flag_servidor,
        1 as flag_visitante,
        coalesce(p.flag_ocorrencia_10d, 0) as flag_ocorrencia_10d,
        coalesce(p.flag_ocorrencia_30d, 0) as flag_ocorrencia_30d,
        coalesce(p.flag_ocorrencia_60d, 0) as flag_ocorrencia_60d,
        p.data_nascimento_pessoa,
        p.data_ultima_prisao,
        p.nome_mae,
        p.nome_pai,
        p.etnia
    from gold.tmp_base_pessoa_atual_religiosa p
    inner join gold.tmp_filtrada_visitareligiosa r
        on p.chave_documento = r.chave_documento
""")

tabela = "tmp_pessoa_atualizada_religiosa"

df_pessoa_atualizada_religiosa.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_pessoa_atualizada_religiosa, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_pessoa_atualizada_religiosa")


# ============================================================
# PESSOAS MANTIDAS
# ============================================================

df_pessoa_mantida_religiosa = spark.sql("""
    select
        p.id_preso,
        p.id_pessoa,
        p.origem,
        p.cod_documento_referencia,
        p.desc_documento_referencia,
        p.documento,
        p.nome_pessoa,
        p.sexo_pessoa,
        p.flag_presidiario,
        p.flag_advogado,
        p.flag_servidor,
        p.flag_visitante,
        p.flag_ocorrencia_10d,
        p.flag_ocorrencia_30d,
        p.flag_ocorrencia_60d,
        p.data_nascimento_pessoa,
        p.data_ultima_prisao,
        p.nome_mae,
        p.nome_pai,
        p.etnia
    from gold.tmp_base_pessoa_atual_religiosa p
    left join gold.tmp_filtrada_visitareligiosa r
        on p.chave_documento = r.chave_documento
    where r.chave_documento is null
""")

tabela = "tmp_pessoa_mantida_religiosa"

df_pessoa_mantida_religiosa.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_pessoa_mantida_religiosa, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_pessoa_mantida_religiosa")


# ============================================================
# NOVAS PESSOAS VINDAS DA VISITA RELIGIOSA
# ============================================================

df_pessoa_nova_religiosa = spark.sql("""
    select
        cast(null as string) as id_preso,
        r.chave_documento as id_pessoa,
        'VISITA_RELIGIOSA' as origem,
        r.cod_documento_referencia,
        r.desc_documento_referencia,
        r.documento_formatado as documento,
        r.nome_pessoa,
        cast(null as string) as sexo_pessoa,
        0 as flag_presidiario,
        0 as flag_advogado,
        0 as flag_servidor,
        1 as flag_visitante,
        0 as flag_ocorrencia_10d,
        0 as flag_ocorrencia_30d,
        0 as flag_ocorrencia_60d,
        cast(null as date) as data_nascimento_pessoa,
        cast(null as date) as data_ultima_prisao,
        cast(null as string) as nome_mae,
        cast(null as string) as nome_pai,
        cast(null as string) as etnia
    from gold.tmp_filtrada_visitareligiosa r
    left join gold.tmp_base_pessoa_atual_religiosa p
        on p.chave_documento = r.chave_documento
    where p.chave_documento is null
""")

tabela = "tmp_pessoa_nova_religiosa"

df_pessoa_nova_religiosa.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_pessoa_nova_religiosa, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_pessoa_nova_religiosa")


# ============================================================
# SAIDA FINAL DAS PESSOAS
# ============================================================

df_pessoa_final_religiosa = spark.sql("""
    select * from gold.tmp_pessoa_mantida_religiosa
    union all
    select * from gold.tmp_pessoa_atualizada_religiosa
    union all
    select * from gold.tmp_pessoa_nova_religiosa
""")

tabela = "df_pessoa_final_religiosa"

#df_pessoa_final_religiosa.write \
#    .mode("overwrite") \
#    .option("maxRecordsPerFile", 1_000_000) \
#    .option("compression", "snappy") \
#    .parquet(f"{path}{tabela}")
#write_impala_table_partioned(df_pessoa_final_religiosa, "gold", tabela, f"{path}{tabela}")


CREATE EXTERNAL TABLE gold.tmp_base_visitareligiosa 
                        (id_visitante_religioso_origem char(5),
    nome_pessoa varchar(75),
    instituicao varchar(106),
    documento_original varchar(20),
    presidio_id_origem char(2),
    documento_digitos char(13),
    cpf_normalizado char(11),
    documento_normalizado varchar(17),
    chave_documento varchar(19))
                        STORED AS PARQUET LOCATION '/data_lake/gold/intlpris/tmp_base_visitareligiosa'
                        
DROP TABLE gold.tmp_base_visitareligiosa 
Estatísticas atualizadas para gold.tmp_base_visitareligiosa
CREATE EXTERNAL TABLE gold.tmp_base_visitareligiosa_rank 
                        (id_visitante_religioso_origem char(5),
    nome_pessoa varchar(75),
    instituicao varchar(106),
    documento_original varchar(20),
    presidio_id_origem char(2),
    documento_digitos char(13),
    cpf_normalizado char(11),
    documento_normalizado varchar(17),
    chave_documento varchar(19),
    rn in

In [10]:
tabela = "sinp_ent_pessoa"

df_pessoa_final_religiosa.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_pessoa_final_religiosa, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql(f"refresh table gold.{tabela}")


df_pessoa_final_religiosa_pg = df_pessoa_final_religiosa.withColumn(
    "id_pessoa",
    F.when(
        F.col("id_pessoa").isNull(),
        F.lit(None)
    ).otherwise(
        F.upper(F.substring(F.sha2(F.col("id_pessoa").cast("string"), 256), 1, 30))
    )
)

tabela_pg = "tmp_pg_sinp_ent_pessoa"

spark.sql(f"drop table if exists gold.{tabela_pg}")
os.system(f"hdfs dfs -rm -r -skipTrash {path}{tabela_pg} >/dev/null 2>&1")

df_pessoa_final_religiosa_pg.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela_pg}")

write_impala_table_partioned(df_pessoa_final_religiosa_pg, "gold", tabela_pg, f"{path}{tabela_pg}")

spark.catalog.clearCache()
spark.sql(f"refresh table gold.{tabela_pg}")

enviar_gold_para_postgres(f"gold.{tabela_pg}", "id_pessoa")


tabela = "sinp_ent_pessoa_outras"
df_pres_outras.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_pres_outras, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres(f"gold.{tabela}", "id_preso")


tabela = "sinp_pnt_pessoa_preso"
df_ponte.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_pres_outras, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres(f"gold.{tabela}", "id_preso")

spark.sql("drop table if exists gold.tmp_base_siarhes")
spark.sql("drop table if exists gold.tmp_filtrada_siarhes")
spark.sql("drop table if exists gold.tmp_base_pessoa_atual")
spark.sql("drop table if exists gold.tmp_pessoa_atualizada_servidor")
spark.sql("drop table if exists gold.tmp_pessoa_mantida")
spark.sql("drop table if exists gold.tmp_pessoa_nova_servidor")
spark.sql("drop table if exists gold.tmp_pessoa_final")
spark.sql("drop table if exists gold.sinp_ent_pes_p1")
spark.sql("drop table if exists gold.tmp_base_cavi")
spark.sql("drop table if exists gold.tmp_base_cavi_rank")
spark.sql("drop table if exists gold.tmp_filtrada_cavi")
spark.sql("drop table if exists gold.tmp_base_pessoa_atual_cavi")
spark.sql("drop table if exists gold.tmp_pessoa_atualizada_visitante")
spark.sql("drop table if exists gold.tmp_pessoa_mantida_visitante")
spark.sql("drop table if exists gold.tmp_pessoa_nova_visitante")
spark.sql("drop table if exists gold.tmp_pessoa_final_visitante")

CREATE EXTERNAL TABLE gold.sinp_ent_pessoa 
                        (id_preso char(6),
    id_pessoa varchar(24),
    origem char(16),
    cod_documento_referencia integer,
    desc_documento_referencia varchar(27),
    documento varchar(20),
    nome_pessoa varchar(99),
    sexo_pessoa varchar(21),
    flag_presidiario integer,
    flag_advogado integer,
    flag_servidor integer,
    flag_visitante integer,
    flag_ocorrencia_10d integer,
    flag_ocorrencia_30d integer,
    flag_ocorrencia_60d integer,
    data_nascimento_pessoa date,
    data_ultima_prisao date,
    nome_mae varchar(90),
    nome_pai varchar(98),
    etnia char(9))
                        STORED AS PARQUET LOCATION '/data_lake/gold/intlpris/sinp_ent_pessoa'
                        
DROP TABLE gold.sinp_ent_pessoa 
Estatísticas atualizadas para gold.sinp_ent_pessoa


NameError: name 'F' is not defined

In [ ]:
#CORRECAO DEVIDO ERRO DE TIPO DE DADO EM PRESO_CELA - APOS CORRIGIDO REMOVER

from pyspark.sql import functions as F

origem = "hdfs://hahdfsprod/data_lake/bronze/INFOPEN/PRESO_CELA.parquet"
destino = "hdfs://hahdfsprod/data_lake/bronze/INFOPEN/PRESO_CELA_CORR.parquet"

df_corr = (
    spark.read.parquet(origem)
    .withColumn("dt_entrada_cela", F.col("dt_entrada_cela").cast("timestamp"))
    .withColumn("dt_registro", F.col("dt_registro").cast("timestamp"))
)

df_corr.write.mode("overwrite").parquet(destino)
df_valid = spark.read.parquet(destino)
df_valid.createOrReplaceTempView("tmp_preso_cela_corr")
spark.sql("""
DROP TABLE IF EXISTS bronze.infopen_preso_cela_corr
""")
spark.sql(f"""
CREATE TABLE bronze.infopen_preso_cela_corr
USING PARQUET
LOCATION '{destino}'
""")

# CORRECAO DEVIDO ERRO DE TIPO DE DADO EM UP_CELA - APOS CORRIGIDO REMOVER

from pyspark.sql import functions as F

origem = "hdfs://hahdfsprod/data_lake/bronze/INFOPEN/UP_CELA.parquet"
destino = "hdfs://hahdfsprod/data_lake/bronze/INFOPEN/UP_CELA_CORR.parquet"

df_corr = (
    spark.read.parquet(origem)
    .withColumn("dt_desativacao", F.col("dt_desativacao").cast("timestamp"))
)

df_corr.write.mode("overwrite").parquet(destino)

df_valid = spark.read.parquet(destino)
df_valid.createOrReplaceTempView("tmp_up_cela_corr")

spark.sql("""
DROP TABLE IF EXISTS bronze.infopen_up_cela_corr
""")

spark.sql(f"""
CREATE TABLE bronze.infopen_up_cela_corr
USING PARQUET
LOCATION '{destino}'
""")


path_origem = "hdfs://hahdfsprod/data_lake/bronze/INFOPEN/UP_GALERIA.parquet"
path_destino = "hdfs://hahdfsprod/data_lake/bronze/INFOPEN/UP_GALERIA_CORR.parquet"

df_gal = (
    spark.read
    .parquet(path_origem)
    .withColumn(
        "dt_desativacao",
        F.to_timestamp(F.col("dt_desativacao").cast("string"), "yyyyMMdd")
    )
)

df_gal.write.mode("overwrite").parquet(path_destino)


spark.sql("drop table if exists bronze.infopen_up_galeria_corr")

spark.sql(f"""
create table bronze.infopen_up_galeria_corr
using parquet
location '{path_destino}'
""")

In [ ]:
spark.sql("""
create table if not exists bronze.sinp_estab_pris (
    sigla       string,
    unidade     string,
    municipio   string,
    endereco    string,
    cep         string,
    uf          string,
    pais        string,
    latitude    double,
    longitude   double
)
stored as parquet
location '/data_lake/bronze/intlpris/sinp_estab_pris'
""")

spark.sql("""
insert overwrite table bronze.sinp_estab_pris values
('CDPM' ,'CENTRO DE DETENÇÃO PROVISÓRIA DE MARATAÍZES' ,'Marataízes' ,'Rua Espinha de Peixe, s/n, Bairro Acapulco' ,'29345-000' ,'ES','Brasil',-21.021596,-40.820643),
('CDPCI' ,'CENTRO DE DETENÇÃO PROVISÓRIA DE CACHOEIRO DE ITAPEMIRIM' ,'Cachoeiro de Itapemirim' ,'Rodovia do Governador Lacerda de Aguiar, Km 01, Bairro Coronel Borges' ,'29306-095' ,'ES','Brasil',-20.8542956,-41.0902809),
('APACF' ,'ASSOCIAÇÃO PROTEÇÃO E ASSIST CONDENADOS CACH ITAPEMIRIM - CRS MASC' ,'Cachoeiro de Itapemirim' ,'Rodovia Cachoeiro-Monte Líbano, Village da Luz' ,'29309-500' ,'ES','Brasil',-20.82597,-41.11094),
('CPFCI' ,'CENTRO PRISIONAL FEMININO DE CACHOEIRO DE ITAPEMIRIM' ,'Cachoeiro de Itapemirim' ,'Fazenda Monte Líbano, s/n, Zona Rural' ,'29300-970' ,'ES','Brasil',-20.73284,-41.10819),
('PRCI' ,'PENITENCIÁRIA REGIONAL DE CACHOEIRO DE ITAPEMIRIM' ,'Cachoeiro de Itapemirim' ,'Fazenda Monte Líbano, s/n, Zona Rural' ,'29300-970' ,'ES','Brasil',-20.7327951,-41.10819),
('CDPG' ,'CENTRO DE DETENÇÃO PROVISÓRIA DE GUARAPARI' ,'Guarapari' ,'Rodovia do Sol, Contorno Argilino Dario, Km 51,3, Maxinda' ,'29200-970' ,'ES','Brasil',-20.6777411,-40.509638),
('CDPFVV' ,'CENTRO DE DETENÇÃO PROVISÓRIA FEMININO DE VILA VELHA' ,'Vila Velha' ,'Rodovia BR-101 Sul, Km 313, Xuri' ,'29100-000' ,'ES','Brasil',-20.4688193,-40.4644965),
('PEVV IV' ,'PENITENCIÁRIA ESTADUAL DE VILA VELHA IV' ,'Vila Velha' ,'Rodovia Governador Mario Covas, s/n, Xuri' ,'29100-000' ,'ES','Brasil',-20.4694866,-40.4674836),
('PEVV V' ,'PENITENCIÁRIA ESTADUAL DE VILA VELHA V' ,'Vila Velha' ,'Rodovia BR-101 Sul, Km 313, Fazenda Santa Fé, Xuri' ,'29100-000' ,'ES','Brasil',-20.4694866,-40.4675164),
('PEVV III' ,'PENITENCIÁRIA ESTADUAL DE VILA VELHA III' ,'Vila Velha' ,'Rodovia BR-101 Sul, Km 313, Fazenda Santa Fé, Xuri' ,'29100-000' ,'ES','Brasil',-20.4694669,-40.4674585),
('PEVV VI' ,'PENITENCIÁRIA ESTADUAL DE VILA VELHA VI' ,'Vila Velha' ,'Rodovia BR-101 Sul, Km 313, Fazenda Santa Fé, Xuri' ,'29100-000' ,'ES','Brasil',-20.4694669,-40.4675415),
('PEVV II' ,'PENITENCIÁRIA ESTADUAL DE VILA VELHA II' ,'Vila Velha' ,'Rodovia BR-101 Sul, Km 313, Fazenda Santa Fé, Xuri' ,'29100-000' ,'ES','Brasil',-20.4694366,-40.4674528),
('PSVV' ,'PENITENCIÁRIA SEMIABERTA DE VILA VELHA' ,'Vila Velha' ,'Rodovia BR-101 Sul, Km 313, Fazenda Santa Fé, Xuri' ,'29100-000' ,'ES','Brasil',-20.4694366,-40.4675472),
('CDPVV' ,'CENTRO DE DETENÇÃO PROVISÓRIA DE VILA VELHA' ,'Vila Velha' ,'Rodovia BR-101 Sul, Km 313, Fazenda Santa Fé, Xuri' ,'29100-000' ,'ES','Brasil',-20.4694444,-40.4675),
('CDPVV 2' ,'CENTRO DE DETENÇÃO PROVISÓRIA DE VILA VELHA II' ,'Vila Velha' ,'Rodovia BR-101 Sul, Km 313, Fazenda Santa Fé, Xuri' ,'29100-000' ,'ES','Brasil',-20.4693995,-40.4675),
('PEVV I' ,'PENITENCIÁRIA ESTADUAL DE VILA VELHA I' ,'Vila Velha' ,'Rodovia BR-101 Sul, Km 313, Fazenda Santa Fé, Xuri' ,'29100-000' ,'ES','Brasil',-20.46941,-40.4674692),
('PSVV II' ,'PENITENCIÁRIA SEMIABERTA DE VILA VELHA II' ,'Vila Velha' ,'Rodovia BR-101 Sul, Km 313, Fazenda Santa Fé, Xuri' ,'29100-000' ,'ES','Brasil',-20.46941,-40.4675308),
('CASCUVI' ,'CASA DE CUSTÓDIA DE VIANA' ,'Viana' ,'BR 262 KM 18 5 S N' ,'29130-010' ,'ES','Brasil',-20.382185,-40.4632381),
('CDPFV' ,'CENTRO DE DETENÇÃO PROVISÓRIA FEMININO DE VIANA' ,'Viana' ,'Rodovia BR-262, Km 18,5' ,'29130-055' ,'ES','Brasil',-20.3821401,-40.4632381),
('CDPV II' ,'CENTRO DE DETENÇÃO PROVISÓRIA DE VIANA II' ,'Viana' ,'Rodovia BR-262, Km 18,5' ,'29130-055' ,'ES','Brasil',-20.3821472,-40.4632122),
('USP' ,'UNIDADE DE SAÚDE PRISIONAL' ,'Viana' ,'Rodovia BR-262, Km 18,5' ,'29130-055' ,'ES','Brasil',-20.3821472,-40.463264),
('CTV' ,'CENTRO DE TRIAGEM DE VIANA' ,'Viana' ,'Rodovia BR-262, Km 18,5' ,'29130-055' ,'ES','Brasil',-20.3821663,-40.4631945),
('DIMCME' ,'DIMCME - SERVIÇO PENAL' ,'Viana' ,'Rodovia BR-262, Km 18,5' ,'29130-055' ,'ES','Brasil',-20.3821914,-40.4631907),
('PSME II' ,'PENITENCIÁRIA DE SEGURANÇA MÉDIA II' ,'Viana' ,'Rodovia BR-262, Km 18,5' ,'29130-055' ,'ES','Brasil',-20.3821663,-40.4632817),
('EPEN' ,'ESCOLA PENITENCIÁRIA' ,'Viana' ,'Rodovia BR-262, Km 18,5' ,'29130-055' ,'ES','Brasil',-20.3822144,-40.4632019),
('PSMA II' ,'PENITENCIÁRIA DE SEGURANÇA MÁXIMA II' ,'Viana' ,'Rodovia BR-262, Km 18,5' ,'29130-055' ,'ES','Brasil',-20.3822144,-40.4632743),
('PAES' ,'PENITENCIÁRIA AGRÍCOLA DO ESPÍRITO SANTO' ,'Viana' ,'Rodovia BR-262, Km 18,5' ,'29130-055' ,'ES','Brasil',-20.3822281,-40.4632246),
('PSMA I' ,'PENITENCIÁRIA DE SEGURANÇA MÁXIMA I' ,'Viana' ,'Rodovia BR-262, Km 18,5' ,'29130-055' ,'ES','Brasil',-20.3822281,-40.4632516),
('PSME I' ,'PENITENCIÁRIA DE SEGURANÇA MÉDIA I' ,'Viana' ,'Rodovia BR-262, Km 18,5' ,'29130-055' ,'ES','Brasil',-20.3821914,-40.4632855),
('CASCUVV' ,'CASA DE CUSTÓDIA DE VILA VELHA' ,'Vila Velha' ,'Rua Mestre Gomes, s/n, Pedra D’Água, Glória' ,'29122-100' ,'ES','Brasil',-20.3314703,-40.3137423),
('APAC' ,'ASSOCIAÇÃO DE PROTEÇÃO E ASSISTÊNCIA AOS CONDENADOS' ,'Vila Velha' ,'Praça Almirante Tamandaré, 193' ,'29.100-310' ,'ES','Brasil',-20.3321936,-40.300015),
('IRS' ,'INSTITUTO DE READAPTAÇÃO SOCIAL' ,'Vila Velha' ,'Rua Mestre Gomes, Glória' ,'29122-100' ,'ES','Brasil',-20.3261111,-40.3141667),
('PSC' ,'PENITENCIÁRIA SEMIABERTA DE CARIACICA' ,'Cariacica' ,'Rodovia Governador José Sette, s/n, Tucum' ,'29152-500' ,'ES','Brasil',-20.32244,-40.36908),
('CDPC' ,'CENTRO DE DETENÇÃO PROVISÓRIA DE CARIACICA' ,'Cariacica' ,'Rua Mario Valentim, Santana' ,'29154-585' ,'ES','Brasil',-20.30972,-40.38562),
('QCGPMES' ,'QUARTEL DO COMANDO GERAL DA POLÍCIA MILITAR DO EST. DO ESPÍRITO SANTO' ,'Vitória' ,'Avenida Maruípe, 2111, São Cristóvão' ,'29048-463' ,'ES','Brasil',-20.2938889,-40.3122222),
('CPFC' ,'CENTRO PRISIONAL FEMININO DE CARIACICA' ,'Cariacica' ,'Rua Ofelino Meireles, Bairro Bubu' ,'29157-766' ,'ES','Brasil',-20.2836,-40.40944),
('PFC' ,'PENITENCIÁRIA FEMININA CARIACICA' ,'Cariacica' ,'Rua Armélio/Ofelino Meireles, Bubu' ,'29157-766' ,'ES','Brasil',-20.2835551,-40.40944),
('HCTP' ,'HOSPITAL DE CUSTÓDIA E TRATAMENTO PSIQUIÁTRICO' ,'Cariacica' ,'Av. José Sette, s/n, Bairro Roças Velhas' ,'29156-970' ,'ES','Brasil',-20.27756,-40.40907),
('UCTP' ,'UNIDADE DE CUSTÓDIA E TRATAMENTO PSIQUIÁTRICO' ,'Cariacica' ,'Av. José Sette, s/n, Bairro Roças Velhas' ,'29156-970' ,'ES','Brasil',-20.2776049,-40.40907),
('PEF' ,'PENITENCIÁRIA ESTADUAL FEMININA' ,'Cariacica' ,'Av. José Sette / região de Roças Velhas' ,'29156-970' ,'ES','Brasil',-20.2775151,-40.40907),
('CDPS' ,'CENTRO DE DETENÇÃO PROVISÓRIA DA SERRA' ,'Serra' ,'Rodovia do Contorno, BR-101, Km 278, Distrito de Queimados' ,'29160-000' ,'ES','Brasil',-20.1283333,-40.3077778),
('CDPA' ,'CENTRO DE DETENÇÃO PROVISÓRIA DE ARACRUZ' ,'Aracruz' ,'Estrada Aracruz-Coqueiral, s/n, Fátima' ,'29192-205' ,'ES','Brasil',-19.9285265,-40.1542386),
('PRL' ,'PENITENCIÁRIA REGIONAL DE LINHARES' ,'Linhares' ,'Rua Projetada, s/n, Jardim Laguna' ,'29900-970' ,'ES','Brasil',-19.3768823,-40.0552718),
('CRL' ,'CENTRO DE RESSOCIALIZAÇÃO DE LINHARES' ,'Linhares' ,'Rodovia ES-440, Km 02, Bebedouro' ,'29900-970' ,'ES','Brasil',-19.4059,-40.0499),
('CDPCOL' ,'CENTRO DE DETENÇÃO PROVISÓRIA DE COLATINA' ,'Colatina' ,'Córrego Santa Fé, s/n' ,'29700-970' ,'ES','Brasil',-19.5498025,-40.6272813),
('CPFCOL' ,'CENTRO PRISIONAL FEMININO DE COLATINA' ,'Colatina' ,'Córrego Santa Fé, s/n' ,'29700-970' ,'ES','Brasil',-19.5497576,-40.6272813),
('PRCOL' ,'PENITENCIÁRIA REGIONAL DE COLATINA' ,'Colatina' ,'Córrego Santa Fé, s/n' ,'29700-970' ,'ES','Brasil',-19.549825,-40.62724),
('PSMECOL' ,'PENITENCIÁRIA DE SEGURANÇA MÉDIA DE COLATINA' ,'Colatina' ,'Córrego Santa Fé, s/n' ,'29700-970' ,'ES','Brasil',-19.549825,-40.6273226),
('PSMCOL' ,'PENITENCIÁRIA SEMIABERTA MASCULINA DE COLATINA' ,'Colatina' ,'Avenida das Nações, s/n, Bairro Benjamin Carlos dos Santos (IBC)' ,'29712-408' ,'ES','Brasil',-19.517611,-40.61206),
('CDPSDN' ,'CENTRO DE DETENÇÃO PROVISÓRIA DE SÃO DOMINGOS DO NORTE' ,'São Domingos do Norte' ,'Córrego Braço do Sul, Km 80, s/n' ,'29745-000' ,'ES','Brasil',-19.0169,-40.5358),
('PRBSF' ,'PENITENCIÁRIA REGIONAL DE BARRA DE SÃO FRANCISCO' ,'Barra de São Francisco' ,'Rodovia ES-320, Km 02' ,'29800-000' ,'ES','Brasil',-18.7525,-40.893),
('PSSM' ,'PENITENCIÁRIA SEMIABERTA DE SÃO MATEUS' ,'São Mateus' ,'Rodovia Governador Mario Covas, BR-101 Norte, Km 72,5, s/n, Rio Preto da Rodovia' ,'29940-800' ,'ES','Brasil',-18.7167449,-39.8594),
('CDPSM' ,'CENTRO DE DETENÇÃO PROVISÓRIA DE SÃO MATEUS' ,'São Mateus' ,'BR-101 Norte, Km 72,5, Fazenda Rancho das Telhas, Zona Rural' ,'29930-000' ,'ES','Brasil',-18.7167,-39.8594),
('PRSM' ,'PENITENCIÁRIA REGIONAL DE SÃO MATEUS' ,'São Mateus' ,'Rodovia Governador Mario Covas, BR-101 Norte, Km 72,5, s/n, Rio Preto da Rodovia' ,'29940-800' ,'ES','Brasil',-18.7166551,-39.8594);
""")

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# BASE DE LOCALIZAÇÃO DO PRESO
# ADAPTADA PARA A NOVA VISÃO:
#   - CELA + GALERIA + ESTABELECIMENTO
# SEM PERDER A LÓGICA JÁ ESTABELECIDA
# ============================================================

df_base_loc = spark.sql("""
with loc as (
    select
        e.id_estabelecimento,
        c.id_galeria,
        c.id_cela,

        c.nm_cela,
        c.id_tipo_cela,
        c.qt_capacidadeprojetada,
        c.qt_capacidadeadaptada,
        c.qt_metros_cela,
        c.observacao_cela,
        c.sexo_cela,
        c.dt_desativacao as dt_desativacao_cela,

        g.descricao_galeria,
        g.dt_desativacao as dt_desativacao_galeria,

        e.id_orgao,
        e.id_estabelecimentotipo,
        e.id_estabelecimentotiposeguranca,
        e.estabelecimento_nome,
        e.estabelecimento_sigla,
        e.estabelecimento_cnpj,
        e.estabelecimento_email,
        e.estabelecimento_capacidade,
        e.estabelecimento_macrorregiao,
        e.estabelecimento_situacaoregistro,

        row_number() over (
            partition by c.id_estabelecimento, c.id_galeria, c.id_cela
            order by
                c.dt_desativacao desc nulls last,
                g.dt_desativacao desc nulls last,
                c.nm_cela
        ) as rn_cela
    from bronze.infopen_estabelecimentos e
    left join 
    bronze.infopen_up_cela_corr c on c.id_estabelecimento = e.id_estabelecimento
    left join bronze.infopen_up_galeria_corr g
        on c.id_galeria = g.id_galeria
       and c.id_estabelecimento = g.id_estabelecimento
       ),

base as (
    select
        c.id_presocela,

        l.id_estabelecimento,
        cast(c.id_preso as string) as id_preso,
        p.id_pessoa,
        p.nome_pessoa,
        l.id_galeria,
        l.id_cela,

        l.nm_cela,
        l.id_tipo_cela,
        l.qt_capacidadeprojetada,
        l.qt_capacidadeadaptada,
        l.qt_metros_cela,
        l.observacao_cela,
        l.sexo_cela,
        l.dt_desativacao_cela,

        l.descricao_galeria,
        l.dt_desativacao_galeria,

        l.id_orgao,
        l.id_estabelecimentotipo,
        l.id_estabelecimentotiposeguranca,
        l.estabelecimento_nome,
        l.estabelecimento_sigla,
        l.estabelecimento_cnpj,
        l.estabelecimento_email,
        l.estabelecimento_capacidade,
        l.estabelecimento_macrorregiao,
        l.estabelecimento_situacaoregistro,

        c.st_cela_ativa,
        c.dt_entrada_cela,
        c.id_usuario_registro,
        c.dt_registro,
        c.observacao_presocela,

        r.municipio,
        r.endereco,
        r.cep,
        r.uf,
        r.pais,
        r.latitude,
        r.longitude,

        row_number() over (
            partition by coalesce(
                cast(c.id_preso as string),
                concat(
                    'SEM_PRESO_',
                    cast(l.id_estabelecimento as string), '_',
                    coalesce(cast(l.id_galeria as string), '0'), '_',
                    coalesce(cast(l.id_cela as string), '0')
                )
            )
            order by
                c.dt_entrada_cela desc nulls last,
                c.dt_registro desc nulls last,
                c.id_presocela desc nulls last
        ) as rn
    from bronze.infopen_estabelecimentos e
    left join loc l
        on l.id_estabelecimento = e.id_estabelecimento
       and l.rn_cela = 1
    left join tmp_preso_cela_corr c
        on c.id_estabelecimento = l.id_estabelecimento
       and c.id_galeria = l.id_galeria
       and c.id_cela = l.id_cela
    left join gold.sinp_pnt_pessoa_preso p
        on cast(c.id_preso as string) = cast(p.id_preso as string)
    left join bronze.sinp_estab_pris r
        on upper(trim(e.estabelecimento_sigla)) = upper(trim(r.sigla))

)

select *
from base
""")

# ============================================================
# ENTIDADE: ESTABELECIMENTO
# ============================================================

df_ent_estabelecimento = spark.sql("""
    SELECT 
        cast(e.id_estabelecimento as string) as id_estabelecimento,
        cast(e.id_orgao as string) as id_orgao,
        cast(e.id_estabelecimentotipo as string) as id_estabelecimentotipo,
        case
            when upper(trim(e.estabelecimento_nome)) like '%HOSPITAL DE CUSTÓDIA%' then 'UNIDADE DE SAÚDE / CUSTÓDIA PSIQUIÁTRICA'
            when upper(trim(e.estabelecimento_nome)) like '%TRATAMENTO PSIQUIÁTRICO%' then 'UNIDADE DE SAÚDE / CUSTÓDIA PSIQUIÁTRICA'
            when upper(trim(e.estabelecimento_nome)) like '%UNIDADE DE SAÚDE PRISIONAL%' then 'UNIDADE DE SAÚDE / CUSTÓDIA PSIQUIÁTRICA'
            when upper(trim(e.estabelecimento_nome)) like '%UNIDADE DE CUSTÓDIA E TRATAMENTO PSIQUIÁTRICO%' then 'UNIDADE DE SAÚDE / CUSTÓDIA PSIQUIÁTRICA'
            when upper(trim(e.estabelecimento_nome)) like '%ESCOLA PENITENCIÁRIA%' then 'UNIDADE ESPECIAL / APOIO / FORMAÇÃO'
            when upper(trim(e.estabelecimento_nome)) like '%ASSOCIAÇÃO DE PROTEÇÃO E ASSISTÊNCIA AOS CONDENADOS%' then 'APAC / ASSOCIAÇÃO DE PROTEÇÃO E ASSISTÊNCIA'
            when upper(trim(e.estabelecimento_nome)) like '%ASSOCIAÇÃO PROTEÇÃO E ASSIST CONDENADOS%' then 'APAC / ASSOCIAÇÃO DE PROTEÇÃO E ASSISTÊNCIA'
            when upper(trim(e.estabelecimento_nome)) like '%INSTITUTO DE READAPTAÇÃO SOCIAL%' then 'READAPTAÇÃO / RESSOCIALIZAÇÃO'
            when upper(trim(e.estabelecimento_nome)) like '%CENTRO DE RESSOCIALIZAÇÃO%' then 'READAPTAÇÃO / RESSOCIALIZAÇÃO'
            when upper(trim(e.estabelecimento_nome)) like '%CASA DE CUSTÓDIA%' then 'CASA DE CUSTÓDIA'
            when upper(trim(e.estabelecimento_nome)) like '%CENTRO DE TRIAGEM%' then 'CENTRO DE DETENÇÃO / TRIAGEM / CENTRO PRISIONAL'
            when upper(trim(e.estabelecimento_nome)) like '%CENTRO DE DETENÇÃO PROVISÓRIA%' then 'CENTRO DE DETENÇÃO / TRIAGEM / CENTRO PRISIONAL'
            when upper(trim(e.estabelecimento_nome)) like '%CENTRO PRISIONAL%' then 'CENTRO DE DETENÇÃO / TRIAGEM / CENTRO PRISIONAL'
            when upper(trim(e.estabelecimento_nome)) like '%QUARTEL DO COMANDO GERAL%' then 'UNIDADE MILITAR / POLICIAL'
            when upper(trim(e.estabelecimento_nome)) like '%PENITENCIÁRIA AGRÍCOLA%' then 'PENITENCIÁRIA AGRÍCOLA'
            when upper(trim(e.estabelecimento_nome)) like '%PENITENCIÁRIA%' then 'PENITENCIÁRIA'
            when upper(trim(e.estabelecimento_nome)) like '%SERVIÇO PENAL%' then 'SERVIÇO PENAL / ADMINISTRATIVO'
            when e.id_estabelecimentotipo = 1  then 'SERVIÇO PENAL / ADMINISTRATIVO'
            when e.id_estabelecimentotipo = 4  then 'PENITENCIÁRIA'
            when e.id_estabelecimentotipo = 5  then 'PENITENCIÁRIA AGRÍCOLA'
            when e.id_estabelecimentotipo = 6  then 'UNIDADE DE SAÚDE / CUSTÓDIA PSIQUIÁTRICA'
            when e.id_estabelecimentotipo = 8  then 'UNIDADE ESPECIAL / APOIO / FORMAÇÃO / APAC'
            when e.id_estabelecimentotipo = 12 then 'READAPTAÇÃO / RESSOCIALIZAÇÃO'
            when e.id_estabelecimentotipo = 14 then 'CENTRO DE DETENÇÃO / TRIAGEM / CENTRO PRISIONAL'
            when e.id_estabelecimentotipo = 15 then 'CASA DE CUSTÓDIA'
            when e.id_estabelecimentotipo = 17 then 'UNIDADE MILITAR / POLICIAL'
            else 'NÃO CLASSIFICADO'
        end as ds_estabelecimentotipo,
        cast(e.id_estabelecimentotiposeguranca as string) as id_estabelecimentotiposeguranca,
        case
            when upper(trim(e.estabelecimento_nome)) like '%SEMIABERTA%' then 'SEMIABERTO / BAIXA CONTENÇÃO / ALTERNATIVO'
            when upper(trim(e.estabelecimento_nome)) like '%APAC%' then 'SEMIABERTO / BAIXA CONTENÇÃO / ALTERNATIVO'
            when upper(trim(e.estabelecimento_nome)) like '%ASSOCIAÇÃO DE PROTEÇÃO E ASSISTÊNCIA AOS CONDENADOS%' then 'SEMIABERTO / BAIXA CONTENÇÃO / ALTERNATIVO'
            when upper(trim(e.estabelecimento_nome)) like '%ASSOCIAÇÃO PROTEÇÃO E ASSIST CONDENADOS%' then 'SEMIABERTO / BAIXA CONTENÇÃO / ALTERNATIVO'
            when upper(trim(e.estabelecimento_nome)) like '%SEGURANÇA MÉDIA%' then 'SEGURANÇA MÉDIA'
            when upper(trim(e.estabelecimento_nome)) like '%SEGURANCA MEDIA%' then 'SEGURANÇA MÉDIA'
            when upper(trim(e.estabelecimento_nome)) like '%SEGURANÇA MÁXIMA%' then 'SEGURANÇA MÁXIMA'
            when upper(trim(e.estabelecimento_nome)) like '%SEGURANCA MAXIMA%' then 'SEGURANÇA MÁXIMA'
            when upper(trim(e.estabelecimento_nome)) like '%VILA VELHA I%' then 'SEGURANÇA MÁXIMA'
            when upper(trim(e.estabelecimento_nome)) like '%VILA VELHA II%' then 'SEGURANÇA MÁXIMA'
            when upper(trim(e.estabelecimento_nome)) like '%VILA VELHA III%' then 'SEGURANÇA MÁXIMA'
            when upper(trim(e.estabelecimento_nome)) like '%VILA VELHA VI%' then 'SEGURANÇA MÁXIMA'
            when upper(trim(e.estabelecimento_nome)) like '%QUARTEL DO COMANDO GERAL%' then 'SEGURANÇA MÁXIMA'
            when upper(trim(e.estabelecimento_nome)) like '%HOSPITAL DE CUSTÓDIA%' then 'PADRÃO / OUTROS / NÃO ESPECIALIZADO'
            when upper(trim(e.estabelecimento_nome)) like '%TRATAMENTO PSIQUIÁTRICO%' then 'PADRÃO / OUTROS / NÃO ESPECIALIZADO'
            when upper(trim(e.estabelecimento_nome)) like '%UNIDADE DE SAÚDE PRISIONAL%' then 'PADRÃO / OUTROS / NÃO ESPECIALIZADO'
            when upper(trim(e.estabelecimento_nome)) like '%UNIDADE DE CUSTÓDIA E TRATAMENTO PSIQUIÁTRICO%' then 'PADRÃO / OUTROS / NÃO ESPECIALIZADO'
            when upper(trim(e.estabelecimento_nome)) like '%CENTRO DE DETENÇÃO PROVISÓRIA%' then 'PADRÃO / OUTROS / NÃO ESPECIALIZADO'
            when upper(trim(e.estabelecimento_nome)) like '%CENTRO DE TRIAGEM%' then 'PADRÃO / OUTROS / NÃO ESPECIALIZADO'
            when upper(trim(e.estabelecimento_nome)) like '%CENTRO PRISIONAL%' then 'PADRÃO / OUTROS / NÃO ESPECIALIZADO'
            when upper(trim(e.estabelecimento_nome)) like '%CASA DE CUSTÓDIA%' then 'PADRÃO / OUTROS / NÃO ESPECIALIZADO'
            when upper(trim(e.estabelecimento_nome)) like '%PENITENCIÁRIA REGIONAL%' then 'PADRÃO / OUTROS / NÃO ESPECIALIZADO'
            when upper(trim(e.estabelecimento_nome)) like '%PENITENCIÁRIA ESTADUAL FEMININA%' then 'PADRÃO / OUTROS / NÃO ESPECIALIZADO'
            when upper(trim(e.estabelecimento_nome)) like '%PENITENCIÁRIA FEMININA%' then 'PADRÃO / OUTROS / NÃO ESPECIALIZADO'
            when upper(trim(e.estabelecimento_nome)) like '%PENITENCIÁRIA AGRÍCOLA%' then 'PADRÃO / OUTROS / NÃO ESPECIALIZADO'
            when e.id_estabelecimentotiposeguranca = 1 then 'SEMIABERTO / BAIXA CONTENÇÃO / ALTERNATIVO'
            when e.id_estabelecimentotiposeguranca = 2 then 'SEGURANÇA MÉDIA'
            when e.id_estabelecimentotiposeguranca = 3 then 'SEGURANÇA MÁXIMA'
            when e.id_estabelecimentotiposeguranca = 4 then 'PADRÃO / OUTROS / NÃO ESPECIALIZADO'
            else 'NÃO CLASSIFICADO'
        end as ds_estabelecimentotiposeguranca,
        e.estabelecimento_nome,
        e.estabelecimento_sigla,
        e.estabelecimento_cnpj,
        e.estabelecimento_email,
        e.estabelecimento_capacidade,
        e.estabelecimento_macrorregiao,
        e.estabelecimento_situacaoregistro,
        p.municipio,
        p.endereco,
        p.cep,
        p.uf,
        p.pais,
        p.latitude,
        p.longitude
    FROM bronze.infopen_estabelecimentos e
    inner join bronze.sinp_estab_pris p
        on upper(trim(e.estabelecimento_sigla)) = upper(trim(p.sigla))
""")

# ============================================================
# ENTIDADE: GALERIA
# ============================================================

df_ent_galeria = (
    df_base_loc
    .filter(F.col("id_galeria").isNotNull())
    .select(
        F.md5(
            F.concat(
                F.lit("E"),
                F.coalesce(F.col("id_estabelecimento").cast("string"), F.lit("")),
                F.lit("G"),
                F.coalesce(F.col("id_galeria").cast("string"), F.lit(""))
            )
        ).alias("id_galeria"),

        F.col("id_estabelecimento").cast("string").alias("id_estabelecimento"),
        F.col("id_galeria").cast("string").alias("id_galeria_original"),

        F.when(
            F.col("descricao_galeria").isNotNull() & (F.length(F.trim(F.col("descricao_galeria"))) > 0),
            F.trim(F.col("descricao_galeria"))
        ).otherwise(
            F.concat(
                F.lit("Galeria "),
                F.when(
                    F.col("id_galeria").cast("string").rlike("^[0-9]+$"),
                    F.lpad(F.col("id_galeria").cast("string"), 3, "0")
                ).otherwise(F.col("id_galeria").cast("string"))
            )
        ).alias("nome_galeria"),

        F.trim(F.col("descricao_galeria")).alias("descricao_galeria"),
        F.col("dt_desativacao_galeria").alias("dt_desativacao_galeria")
    )
    .distinct()
)

# ============================================================
# ENTIDADE: CELA
# ============================================================

df_ent_cela = (
    df_base_loc
    .filter(
        F.col("id_galeria").isNotNull() &
        F.col("id_cela").isNotNull()
    )
    .select(
        F.md5(
            F.concat(
                F.lit("E"),
                F.coalesce(F.col("id_estabelecimento").cast("string"), F.lit("")),
                F.lit("G"),
                F.coalesce(F.col("id_galeria").cast("string"), F.lit("")),
                F.lit("C"),
                F.coalesce(F.col("id_cela").cast("string"), F.lit(""))
            )
        ).alias("id_cela"),

        F.md5(
            F.concat(
                F.lit("E"),
                F.coalesce(F.col("id_estabelecimento").cast("string"), F.lit("")),
                F.lit("G"),
                F.coalesce(F.col("id_galeria").cast("string"), F.lit(""))
            )
        ).alias("id_galeria"),

        F.col("id_estabelecimento").cast("string").alias("id_estabelecimento_origem"),
        F.col("id_galeria").cast("string").alias("id_galeria_origem"),
        F.col("id_cela").cast("string").alias("id_cela_origem"),

        F.when(
            F.col("nm_cela").isNotNull() & (F.length(F.trim(F.col("nm_cela"))) > 0),
            F.trim(F.col("nm_cela"))
        ).otherwise(
            F.concat(
                F.lit("Cela "),
                F.when(
                    F.col("id_cela").cast("string").rlike("^[0-9]+$"),
                    F.lpad(F.col("id_cela").cast("string"), 3, "0")
                ).otherwise(F.col("id_cela").cast("string"))
            )
        ).alias("nome_cela"),

        F.col("id_tipo_cela").cast("int").alias("id_tipo_cela"),
        F.col("qt_capacidadeprojetada").cast("int").alias("qt_capacidade_projetada"),
        F.col("qt_capacidadeadaptada").cast("int").alias("qt_capacidade_adaptada"),
        F.col("qt_metros_cela").cast("int").alias("qt_metros_cela"),
        F.trim(F.col("observacao_cela")).alias("observacao_cela"),
        F.col("sexo_cela").alias("sexo_cela"),
        F.col("dt_desativacao_cela").alias("dt_desativacao_cela"),
        F.trim(F.col("descricao_galeria")).alias("descricao_galeria"),
        F.col("estabelecimento_nome").alias("estabelecimento_nome"),

        F.concat_ws(
            " > ",
            F.col("estabelecimento_nome"),
            F.col("descricao_galeria"),
            F.when(
                F.col("nm_cela").isNotNull() & (F.length(F.trim(F.col("nm_cela"))) > 0),
                F.trim(F.col("nm_cela"))
            ).otherwise(
                F.concat(
                    F.lit("Cela "),
                    F.when(
                        F.col("id_cela").cast("string").rlike("^[0-9]+$"),
                        F.lpad(F.col("id_cela").cast("string"), 3, "0")
                    ).otherwise(F.col("id_cela").cast("string"))
                )
            )
        ).alias("ds_localizacao_completa")
    )
    .dropDuplicates(["id_cela"])
)

In [ ]:
# ============================================================
# ESCRITA GOLD
# ============================================================

tabela = "sinp_ent_estabelecimento"
df_ent_estabelecimento.write.mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_ent_estabelecimento, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres(f"gold.{tabela}", "id_estabelecimento")

tabela = "sinp_ent_galeria"
df_ent_galeria.write.mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_ent_galeria, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres(f"gold.{tabela}", "id_galeria")

tabela = "sinp_ent_cela"
df_ent_cela.write.mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_ent_cela, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres(f"gold.{tabela}", "id_cela")

In [ ]:
rel_passo = spark.sql("""
select
        id as id_mov,
        trim(cast(infopen as string)) as id_preso,
        nome as nome_livro,
        trim(cast(cela_atual as string)) as cela_atual,
        trim(cast(cela_destino as string)) as cela_destino,
        autorizacao,
        motivo,
        cast(data_registro as timestamp) as dt_movimentacao,
        equipe_id,
        presidio_id
    from bronze.livros_acesso_unidade_trocacela
    where trim(cast(infopen as string)) is not null
      and trim(cast(infopen as string)) <> ''
      and cast(data_registro as timestamp) is not null
      and (
            (trim(cast(cela_atual as string)) is not null and trim(cast(cela_atual as string)) <> '')
         or (trim(cast(cela_destino as string)) is not null and trim(cast(cela_destino as string)) <> '')
      )

""")

tabela = "sinp_preso_cela_p1"
rel_passo.write.mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(rel_passo, "gold", tabela, f"{path}{tabela}")

#rel_passo.show(30,False)

spark.catalog.clearCache()
spark.sql("refresh table gold.sinp_preso_cela_p1")

rel_passo = spark.sql("""
select
        s.id_mov as id_evento_origem,
        id_preso,
        s.cela_atual as id_cela_origem_livro,
        s.dt_movimentacao as dt_entrada_uso_cela,
        s.dt_movimentacao as dt_evento_referencia,
        'PRIMEIRA_ORIGEM_INFERIDA' as origem_regra,
        1 as prioridade_regra,
        s.autorizacao as autorizacao,
        s.motivo as motivo,
        s.nome_livro as nome_livro,
        s.equipe_id as equipe_id,
        s.presidio_id as presidio_id
    from (
        select
            id_preso,
            min(
                named_struct(
                    'dt_movimentacao', dt_movimentacao,
                    'id_mov', id_mov,
                    'cela_atual', cela_atual,
                    'autorizacao', autorizacao,
                    'motivo', motivo,
                    'nome_livro', nome_livro,
                    'equipe_id', equipe_id,
                    'presidio_id', presidio_id
                )
            ) as s
        from gold.sinp_preso_cela_p1
        where cela_atual is not null
          and cela_atual <> ''
        group by id_preso
    ) a
""")

tabela = "sinp_preso_cela_p2"
rel_passo.write.mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(rel_passo, "gold", tabela, f"{path}{tabela}")

#rel_passo.show(30,False)

rel_passo = spark.sql("""
    select
        id_mov as id_evento_origem,
        id_preso,
        cela_destino as id_cela_origem_livro,
        dt_movimentacao as dt_entrada_uso_cela,
        dt_movimentacao as dt_evento_referencia,
        'DESTINO_MOVIMENTACAO' as origem_regra,
        2 as prioridade_regra,
        autorizacao,
        motivo,
        nome_livro,
        equipe_id,
        presidio_id
    from gold.sinp_preso_cela_p1
    where cela_destino is not null
      and cela_destino <> ''
""")

tabela = "sinp_preso_cela_p3"
rel_passo.write.mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(rel_passo, "gold", tabela, f"{path}{tabela}")


spark.catalog.clearCache()
spark.sql("refresh table gold.sinp_preso_cela_p2")
spark.sql("refresh table gold.sinp_preso_cela_p3")


#rel_passo.show(30,False)

rel_passo = spark.sql("""
    select * from gold.sinp_preso_cela_p2
    union all
    select * from gold.sinp_preso_cela_p3
""")

tabela = "sinp_preso_cela_p4"
rel_passo.write.mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(rel_passo, "gold", tabela, f"{path}{tabela}")
    



In [ ]:
spark.sql("drop table if exists gold.sinp_preso_cela_p1")
spark.sql("drop table if exists gold.sinp_preso_cela_p2")
spark.sql("drop table if exists gold.sinp_preso_cela_p3")

spark.catalog.clearCache()
spark.sql("refresh table gold.sinp_preso_cela_p4")

rel_passo = spark.sql("""
select
    md5(concat_ws('||',
        trim(cast(id_preso as string)),
        cast(dt_entrada_uso_cela as string),
        trim(cast(id_cela_origem_livro as string))
    )) as id_preso_data_cela,
    id_evento_origem as id_mov,
    id_preso,
    id_cela_origem_livro,
    dt_entrada_uso_cela,
    dt_evento_referencia,
    origem_regra,
    prioridade_regra,
    autorizacao,
    motivo,
    nome_livro,
    equipe_id,
    presidio_id,
    row_number() over (
        partition by md5(concat_ws('||',
            trim(cast(id_preso as string)),
            cast(dt_entrada_uso_cela as string),
            trim(cast(id_cela_origem_livro as string))
        ))
        order by
            prioridade_regra asc,
            dt_evento_referencia asc,
            origem_regra asc
    ) as rn
from gold.sinp_preso_cela_p4
where id_preso is not null
  and trim(cast(id_preso as string)) <> ''
  and dt_entrada_uso_cela is not null
  and id_cela_origem_livro is not null
  and trim(cast(id_cela_origem_livro as string)) <> ''
""")

#rel_passo.show(30,False)

tabela = "sinp_preso_cela_p5"
rel_passo.write.mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(rel_passo, "gold", tabela, f"{path}{tabela}")

In [ ]:
df_ultima_data = spark.sql("""
select id_preso, max(dt_entrada_uso_cela) as dataRef from gold.sinp_preso_cela_p5 group by 1
""")
tabela = "ref_maxData"
df_ultima_data.write.mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_ultima_data, "gold", tabela, f"{path}{tabela}")

df_base_preso_cela = spark.sql("""
select
    p5.id_preso_data_cela as id_rel_preso_cela,
    c.id_cela,
    p.id_pessoa,p.nome_pessoa,
    p5.id_mov,
    p5.id_preso,
    p5.id_cela_origem_livro as numero_cela,
    p5.dt_entrada_uso_cela,
    p5.dt_evento_referencia,
    p5.origem_regra,
    p5.prioridade_regra,
    p5.autorizacao,
    p5.motivo,
    p5.nome_livro,
    p5.equipe_id,
    p5.presidio_id,
    case when m.dataRef is null then 'Anterior' else 'Atual' end as situacao
from gold.sinp_preso_cela_p5 p5
    inner join gold.sinp_ent_cela c on p5.id_cela_origem_livro = c.id_cela_origem
    inner join gold.sinp_pnt_pessoa_preso p on p5.id_preso = p.id_preso
    left join gold.ref_maxData m on p5.id_preso = m.id_preso and p5.dt_entrada_uso_cela = m.dataRef
where rn= 1
""")

tabela = "sinp_rel_preso_cela"
df_base_preso_cela.write.mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_base_preso_cela, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres(f"gold.{tabela}", "id_rel_preso_cela")


df_base_preso_cela_exc = spark.sql("""
select
    p5.id_preso_data_cela as id_rel_preso_cela,
    c.id_cela,
    p.id_pessoa,p.nome_pessoa,
    p5.id_mov,
    p5.id_preso,
    p5.id_cela_origem_livro as numero_cela,
    p5.dt_entrada_uso_cela,
    p5.dt_evento_referencia,
    p5.origem_regra,
    p5.prioridade_regra,
    p5.autorizacao,
    p5.motivo,
    p5.nome_livro,
    p5.equipe_id,
    p5.presidio_id,
    case when m.dataRef is null then 'Anterior' else 'Atual' end as situacao
from gold.sinp_preso_cela_p5 p5
    inner join gold.sinp_ent_cela c on p5.id_cela_origem_livro = c.id_cela_origem
    inner join gold.sinp_pnt_pessoa_preso p on p5.id_preso = p.id_preso
    left join gold.ref_maxData m on p5.id_preso = m.id_preso and p5.dt_entrada_uso_cela = m.dataRef
where rn> 1
""")

tabela = "sinp_rel_preso_cela_exc"
df_base_preso_cela_exc.write.mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_base_preso_cela_exc, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres(f"gold.{tabela}", "id_rel_preso_cela")


In [ ]:
# ============================================================
# CORREÇÃO DEVIDO ERRO DE TIPO DE DADO EM PRESO_PRONTUARIO_SOCIAL
# APÓS CORRIGIDO REMOVER
# ============================================================

origem = "hdfs://hahdfsprod/data_lake/bronze/INFOPEN/PRESO_PRONTUARIO_SOCIAL.parquet"
destino = "hdfs://hahdfsprod/data_lake/bronze/INFOPEN/PRESO_PRONTUARIO_SOCIAL_CORR.parquet"

df_corr = (
    spark.read.parquet(origem)
    .withColumn("dt_cadastro_ficha", F.col("dt_cadastro_ficha").cast("timestamp"))
)

df_corr.write.mode("overwrite").parquet(destino)

df_valid = spark.read.parquet(destino)
df_valid.createOrReplaceTempView("tmp_preso_prontuario_social_corr")

spark.sql("""
DROP TABLE IF EXISTS bronze.infopen_preso_prontuario_social_corr
""")

spark.sql(f"""
CREATE TABLE bronze.infopen_preso_prontuario_social_corr
USING PARQUET
LOCATION '{destino}'
""")

# ============================================================
# CORRECAO DEVIDO ERRO DE TIPO DE DADO EM PRESO_PRONTUARIO_PSICO
# APOS CORRIGIDO REMOVER
# ============================================================

origem = "hdfs://hahdfsprod/data_lake/bronze/INFOPEN/PRESO_PRONTUARIO_PSICO.parquet"
destino = "hdfs://hahdfsprod/data_lake/bronze/INFOPEN/PRESO_PRONTUARIO_PSICO_CORR.parquet"

df_corr = (
    spark.read.parquet(origem)
    .withColumn("dt_cadastro_ficha", F.col("dt_cadastro_ficha").cast("timestamp"))
    .withColumn("dt_ultima_alteracao", F.col("dt_ultima_alteracao").cast("timestamp"))
)

df_corr.write.mode("overwrite").parquet(destino)

df_valid = spark.read.parquet(destino)
df_valid.createOrReplaceTempView("tmp_preso_prontuario_psico_corr")

spark.sql("""
DROP TABLE IF EXISTS bronze.infopen_preso_prontuario_psico_corr
""")

spark.sql(f"""
CREATE TABLE bronze.infopen_preso_prontuario_psico_corr
USING PARQUET
LOCATION '{destino}'
""")

# ============================================================
# CORRECAO DEVIDO ERRO DE TIPO DE DADO EM PRESO_PRONTUARIO_CRIMINAL
# APOS CORRIGIDO REMOVER
# ============================================================

origem = "hdfs://hahdfsprod/data_lake/bronze/INFOPEN/PRESO_PRONTUARIO_CRIMINAL.parquet"
destino = "hdfs://hahdfsprod/data_lake/bronze/INFOPEN/PRESO_PRONTUARIO_CRIMINAL_CORR.parquet"

df_corr = (
    spark.read.parquet(origem)
    .withColumn("dt_cadastro_ficha", F.col("dt_cadastro_ficha").cast("timestamp"))
    .withColumn("dt_ultima_alteracao", F.col("dt_ultima_alteracao").cast("timestamp"))
)

df_corr.write.mode("overwrite").parquet(destino)

df_valid = spark.read.parquet(destino)
df_valid.createOrReplaceTempView("tmp_preso_prontuario_criminal_corr")

spark.sql("""
DROP TABLE IF EXISTS bronze.infopen_preso_prontuario_criminal_corr
""")

spark.sql(f"""
CREATE TABLE bronze.infopen_preso_prontuario_criminal_corr
USING PARQUET
LOCATION '{destino}'
""")


# ============================================================
# CORRECAO DEVIDO ERRO DE TIPO DE DADO EM PRESO_PRONTUARIO_PROFISSIONAL
# APOS CORRIGIDO REMOVER
# ============================================================

origem = "hdfs://hahdfsprod/data_lake/bronze/INFOPEN/PRESO_PRONTUARIO_PROFISSIONAL.parquet"
destino = "hdfs://hahdfsprod/data_lake/bronze/INFOPEN/PRESO_PRONTUARIO_PROFISSIONAL_CORR.parquet"

df_corr = (
    spark.read.parquet(origem)
    .withColumn("dt_cadastro_ficha", F.col("dt_cadastro_ficha").cast("timestamp"))
    .withColumn("dt_ultima_alteracao", F.col("dt_ultima_alteracao").cast("timestamp"))
)

df_corr.write.mode("overwrite").parquet(destino)

df_valid = spark.read.parquet(destino)
df_valid.createOrReplaceTempView("tmp_preso_prontuario_profissional_corr")

spark.sql("""
DROP TABLE IF EXISTS bronze.infopen_preso_prontuario_profissional_corr
""")

spark.sql(f"""
CREATE TABLE bronze.infopen_preso_prontuario_profissional_corr
USING PARQUET
LOCATION '{destino}'
""")

In [ ]:
prontsoc = spark.sql("""
SELECT
    md5(concat(cast(ps.id_prontuario_social as string), cast(p.id_pessoa as string))) as id_pessoa_prontsoc,
    p.id_pessoa,p.nome_pessoa,
    ps.*
FROM bronze.infopen_preso_prontuario_social_corr ps
inner join gold.sinp_pnt_pessoa_preso p
    on cast(ps.id_preso as string) = cast(p.id_preso as string)
""")

tabela = "sinp_fil_pront_soc"
prontsoc.write.mode("overwrite").option("maxRecordsPerFile", 1_000_000).option("compression", "snappy").parquet(f"{path}{tabela}")
write_impala_table_partioned(prontsoc, "gold", tabela, f"{path}{tabela}")

enviar_gold_para_postgres("gold.sinp_fil_pront_soc", "id_pessoa_prontsoc")

tabela = "sinp_fil_pront_soc"

# ============================================================
# PRONTUARIO CRIMINAL
# ============================================================

prontcrim = spark.sql("""
SELECT
    md5(concat(cast(pc.id_prontuario_criminal as string), cast(p.id_pessoa as string))) as id_pessoa_prontcrim,
    p.id_pessoa,p.nome_pessoa,
    pc.*
FROM bronze.infopen_preso_prontuario_criminal_corr pc
inner join gold.sinp_pnt_pessoa_preso p
    on cast(pc.id_preso as string) = cast(p.id_preso as string)
""")

tabela = "sinp_fil_pront_crim"

prontcrim.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(prontcrim, "gold", tabela, f"{path}{tabela}")

enviar_gold_para_postgres("gold.sinp_fil_pront_crim", "id_pessoa_prontcrim")


# ============================================================
# PRONTUARIO PROFISSIONAL
# ============================================================

prontprof = spark.sql("""
SELECT
    md5(concat(cast(pp.id_prontuario_profissional as string), cast(p.id_pessoa as string))) as id_pessoa_prontprof,
    p.id_pessoa,p.nome_pessoa,
    pp.*
FROM bronze.infopen_preso_prontuario_profissional_corr pp
inner join gold.sinp_pnt_pessoa_preso p
    on cast(pp.id_preso as string) = cast(p.id_preso as string)
""")

tabela = "sinp_fil_pront_prof"

prontprof.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(prontprof, "gold", tabela, f"{path}{tabela}")

enviar_gold_para_postgres("gold.sinp_fil_pront_prof", "id_pessoa_prontprof")

# ============================================================
# PRONTUARIO PSICO
# ============================================================

prontpsico = spark.sql("""
SELECT
    md5(concat(cast(pps.id_prontuario_psico as string), cast(p.id_pessoa as string))) as id_pessoa_prontpsico,
    p.id_pessoa,p.nome_pessoa,
    pps.*
FROM bronze.infopen_preso_prontuario_psico_corr pps
inner join gold.sinp_pnt_pessoa_preso p
    on cast(pps.id_preso as string) = cast(p.id_preso as string)
""")

tabela = "sinp_fil_pront_psico"

prontpsico.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(prontpsico, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres("gold.sinp_fil_pront_psico", "id_pessoa_prontpsico")

In [ ]:
import os
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# REFRESH
# ============================================================

spark.sql("REFRESH TABLE bronze.infopen_vw_alvaras")
spark.sql("REFRESH TABLE bronze.infopen_minio_beneficios")
spark.sql("REFRESH TABLE gold.sinp_pnt_pessoa_preso")
spark.sql("REFRESH TABLE gold.sinp_ent_pessoa")
spark.catalog.clearCache()


# ============================================================
# HELPERS
# ============================================================

def first_existing_col(cols, candidates):
    mapa = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in mapa:
            return mapa[cand.lower()]
    return None

def add_ts_from_candidates(df, new_col, candidates):
    col = first_existing_col(df.columns, candidates)
    if col:
        return df.withColumn(new_col, F.to_timestamp(F.col(col))), col
    return df.withColumn(new_col, F.lit(None).cast("timestamp")), None

def add_dt_from_ts(df, new_col, source_ts_col):
    return df.withColumn(new_col, F.to_date(F.col(source_ts_col)))

def lit_str_or_null(valor):
    return F.lit(valor).cast("string")


# ============================================================
# REFERENCIA DE PESSOA
# ============================================================

df_ref_pessoa_alvara = spark.sql("""
    select distinct
        cast(pp.id_preso as string) as id_preso_str,
        pp.id_pessoa,
        coalesce(ep.nome_pessoa, pp.nome_pessoa) as nome_pessoa,
        ep.documento as documento_pessoa,
        ep.origem as origem_pessoa,
        ep.sexo_pessoa,
        ep.data_nascimento_pessoa,
        ep.nome_mae,
        ep.nome_pai,
        ep.etnia
    from gold.sinp_pnt_pessoa_preso pp
    left join gold.sinp_ent_pessoa ep
        on pp.id_pessoa = ep.id_pessoa
""")

tabela = "tmp_ref_pessoa_alvara"

spark.sql(f"drop table if exists gold.{tabela}")
os.system(f"hdfs dfs -rm -r -skipTrash {path}{tabela} >/dev/null 2>&1")

df_ref_pessoa_alvara.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_ref_pessoa_alvara, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_ref_pessoa_alvara")


# ============================================================
# BASE ALVARAS
# ============================================================

df_alvaras_base = spark.sql("""
    with alvaras as (
        select
            *,
            cast(id_preso as string) as id_preso_str
        from bronze.infopen_vw_alvaras
    )
    select
        a.*,
        p.id_pessoa,
        b.beneficio_nome,
        b.beneficio_descricao,
        p.nome_pessoa,
        p.documento_pessoa,
        p.origem_pessoa,
        p.sexo_pessoa,
        p.data_nascimento_pessoa,
        p.nome_mae,
        p.nome_pai,
        p.etnia
    from alvaras a
    left join gold.tmp_ref_pessoa_alvara p
        on a.id_preso_str = p.id_preso_str
    inner join bronze.infopen_minio_beneficios b
        on a.id_beneficio = b.id_beneficio
""")

tabela = "tmp_base_alvaras"

spark.sql(f"drop table if exists gold.{tabela}")
os.system(f"hdfs dfs -rm -r -skipTrash {path}{tabela} >/dev/null 2>&1")

df_alvaras_base.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_alvaras_base, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_alvaras")


# ============================================================
# ENRIQUECIMENTO DINAMICO
# ============================================================

df_alvaras = spark.table("gold.tmp_base_alvaras")

# Datas canônicas
df_alvaras, src_dt_emissao = add_ts_from_candidates(df_alvaras, "dt_emissao_alvara", [
    "dt_emissao", "data_emissao", "dt_alvara", "data_alvara", "dt_expedicao", "data_expedicao"
])

df_alvaras, src_dt_cadastro = add_ts_from_candidates(df_alvaras, "dt_cadastro_alvara", [
    "dt_cadastro", "data_cadastro", "dt_registro", "data_registro", "dt_inclusao", "data_inclusao"
])

df_alvaras, src_dt_cumprimento = add_ts_from_candidates(df_alvaras, "dt_cumprimento_alvara", [
    "dt_cumprimento", "data_cumprimento", "dt_baixa", "data_baixa"
])

df_alvaras, src_dt_revogacao = add_ts_from_candidates(df_alvaras, "dt_revogacao_alvara", [
    "dt_revogacao", "data_revogacao", "dt_cancelamento", "data_cancelamento"
])

df_alvaras, src_dt_validade = add_ts_from_candidates(df_alvaras, "dt_validade_alvara", [
    "dt_validade", "data_validade", "dt_vencimento", "data_vencimento", "dt_prazo", "data_prazo"
])

df_alvaras = df_alvaras.withColumn(
    "dt_referencia_alvara",
    F.coalesce(
        F.col("dt_emissao_alvara"),
        F.col("dt_cadastro_alvara"),
        F.col("dt_cumprimento_alvara")
    )
)

df_alvaras = add_dt_from_ts(df_alvaras, "dt_referencia_alvara_ref", "dt_referencia_alvara")
df_alvaras = add_dt_from_ts(df_alvaras, "dt_emissao_alvara_ref", "dt_emissao_alvara")
df_alvaras = add_dt_from_ts(df_alvaras, "dt_cadastro_alvara_ref", "dt_cadastro_alvara")
df_alvaras = add_dt_from_ts(df_alvaras, "dt_cumprimento_alvara_ref", "dt_cumprimento_alvara")
df_alvaras = add_dt_from_ts(df_alvaras, "dt_revogacao_alvara_ref", "dt_revogacao_alvara")
df_alvaras = add_dt_from_ts(df_alvaras, "dt_validade_alvara_ref", "dt_validade_alvara")

# Texto consolidado do benefício
df_alvaras = df_alvaras.withColumn(
    "txt_beneficio",
    F.trim(
        F.concat_ws(
            " - ",
            F.coalesce(F.col("beneficio_nome"), F.lit("")),
            F.coalesce(F.col("beneficio_descricao"), F.lit(""))
        )
    )
)

df_alvaras = df_alvaras.withColumn(
    "txt_beneficio_upper",
    F.upper(F.coalesce(F.col("txt_beneficio"), F.lit("")))
)

# Flags básicas
df_alvaras = df_alvaras.withColumn(
    "flag_tem_pessoa",
    F.when(F.col("id_pessoa").isNotNull(), F.lit(1)).otherwise(F.lit(0))
)

df_alvaras = df_alvaras.withColumn(
    "flag_tem_beneficio",
    F.when(F.col("id_beneficio").isNotNull(), F.lit(1)).otherwise(F.lit(0))
)

df_alvaras = df_alvaras.withColumn(
    "flag_tem_documento_pessoa",
    F.when(F.col("documento_pessoa").isNotNull(), F.lit(1)).otherwise(F.lit(0))
)

# Tipificação do benefício
df_alvaras = df_alvaras.withColumn(
    "flag_beneficio_liberatorio",
    F.when(
        F.col("txt_beneficio_upper").rlike(
            "SOLTURA|LIBERDADE|ALVARA DE SOLTURA|ALVARÁ DE SOLTURA|RELAXAMENTO|REVOGA.?.? DE PRISAO|REVOGA.?.? DE PRISÃO|PRISAO DOMICILIAR|PRISÃO DOMICILIAR"
        ),
        F.lit(1)
    ).otherwise(F.lit(0))
)

df_alvaras = df_alvaras.withColumn(
    "flag_beneficio_temporario",
    F.when(
        F.col("txt_beneficio_upper").rlike(
            "SAIDA TEMPORARIA|SAÍDA TEMPORÁRIA|VISITA PERIODICA|VISITA PERIÓDICA|TRABALHO EXTERNO"
        ),
        F.lit(1)
    ).otherwise(F.lit(0))
)

df_alvaras = df_alvaras.withColumn(
    "flag_beneficio_regime",
    F.when(
        F.col("txt_beneficio_upper").rlike("REGIME|PROGRESSAO|PROGRESSÃO"),
        F.lit(1)
    ).otherwise(F.lit(0))
)

# Status operacional
df_alvaras = df_alvaras.withColumn(
    "status_alvara",
    F.when(F.col("dt_revogacao_alvara").isNotNull(), F.lit("REVOGADO"))
     .when(F.col("dt_cumprimento_alvara").isNotNull(), F.lit("CUMPRIDO"))
     .when(
         F.col("dt_validade_alvara_ref").isNotNull() &
         (F.col("dt_validade_alvara_ref") < F.current_date()),
         F.lit("VENCIDO")
     )
     .when(F.col("dt_referencia_alvara").isNotNull(), F.lit("ATIVO_OU_EM_ABERTO"))
     .otherwise(F.lit("SEM_STATUS"))
)

df_alvaras = df_alvaras.withColumn(
    "flag_alvara_ativo",
    F.when(F.col("status_alvara") == "ATIVO_OU_EM_ABERTO", F.lit(1)).otherwise(F.lit(0))
)

df_alvaras = df_alvaras.withColumn(
    "flag_alvara_cumprido",
    F.when(F.col("status_alvara") == "CUMPRIDO", F.lit(1)).otherwise(F.lit(0))
)

df_alvaras = df_alvaras.withColumn(
    "flag_alvara_revogado",
    F.when(F.col("status_alvara") == "REVOGADO", F.lit(1)).otherwise(F.lit(0))
)

df_alvaras = df_alvaras.withColumn(
    "flag_alvara_vencido",
    F.when(F.col("status_alvara") == "VENCIDO", F.lit(1)).otherwise(F.lit(0))
)

# Métricas temporais
df_alvaras = df_alvaras.withColumn(
    "idade_alvara_dias",
    F.when(
        F.col("dt_referencia_alvara_ref").isNotNull(),
        F.datediff(F.current_date(), F.col("dt_referencia_alvara_ref"))
    ).otherwise(F.lit(None).cast("int"))
)

df_alvaras = df_alvaras.withColumn(
    "flag_alvara_recente_30d",
    F.when(
        F.col("idade_alvara_dias").between(0, 30),
        F.lit(1)
    ).otherwise(F.lit(0))
)

# Métricas por preso/pessoa
w_preso = Window.partitionBy("id_preso_str")
w_pessoa = Window.partitionBy("id_pessoa")

df_alvaras = df_alvaras.withColumn(
    "qtd_alvaras_preso",
    F.count(F.lit(1)).over(w_preso)
)

df_alvaras = df_alvaras.withColumn(
    "qtd_alvaras_pessoa",
    F.when(
        F.col("id_pessoa").isNotNull(),
        F.count(F.lit(1)).over(w_pessoa)
    ).otherwise(F.lit(None).cast("int"))
)

df_alvaras = df_alvaras.withColumn(
    "dt_primeiro_alvara_preso",
    F.min("dt_referencia_alvara").over(w_preso)
)

df_alvaras = df_alvaras.withColumn(
    "dt_ultimo_alvara_preso",
    F.max("dt_referencia_alvara").over(w_preso)
)

df_alvaras = df_alvaras.withColumn(
    "flag_multiplos_alvaras_preso",
    F.when(F.col("qtd_alvaras_preso") > 1, F.lit(1)).otherwise(F.lit(0))
)

# Colunas de rastreabilidade - CORRIGIDAS PARA NAO GERAR VOID
df_alvaras = df_alvaras.withColumn("src_dt_emissao_alvara", lit_str_or_null(src_dt_emissao))
df_alvaras = df_alvaras.withColumn("src_dt_cadastro_alvara", lit_str_or_null(src_dt_cadastro))
df_alvaras = df_alvaras.withColumn("src_dt_cumprimento_alvara", lit_str_or_null(src_dt_cumprimento))
df_alvaras = df_alvaras.withColumn("src_dt_revogacao_alvara", lit_str_or_null(src_dt_revogacao))
df_alvaras = df_alvaras.withColumn("src_dt_validade_alvara", lit_str_or_null(src_dt_validade))


# ============================================================
# PERSISTENCIA INTERMEDIARIA
# ============================================================

tabela = "tmp_base_alvaras_enriquecida"

spark.sql(f"drop table if exists gold.{tabela}")
os.system(f"hdfs dfs -rm -r -skipTrash {path}{tabela} >/dev/null 2>&1")

df_alvaras.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_alvaras, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_alvaras_enriquecida")


# ============================================================
# ENTIDADE FINAL
# ============================================================

df_alvaras_final = spark.table("gold.tmp_base_alvaras_enriquecida")

tabela = "sinp_ent_alvaras"

spark.sql(f"drop table if exists gold.{tabela}")
os.system(f"hdfs dfs -rm -r -skipTrash {path}{tabela} >/dev/null 2>&1")

df_alvaras_final.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_alvaras_final, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres(f"gold.{tabela}", "id_alvara")

In [ ]:
from pyspark.sql import functions as F, types as T, Row
import hashlib
from datetime import datetime, timedelta, date

# ============================================================
# FUNCOES AUXILIARES
# ============================================================

def _hash(*vals):
    txt = "||".join("" if v is None else str(v) for v in vals)
    return hashlib.md5(txt.encode("utf-8")).hexdigest()

def _to_python_dt(v):
    if v is None:
        return None
    return v

def _today_midnight():
    now = datetime.now()
    return datetime(now.year, now.month, now.day)

# ============================================================
# 1. BASE UNIFICADA DE EVENTOS
# ============================================================

df_eventos_macro = spark.sql("""
select
    cast(id_preso as string) as id_preso,
    cast(id_pessoa as string) as id_pessoa,
    nome_pessoa,
    cast(id_movimentacao as string) as id_movimentacao,
    cast(id_tipomovimentacao as string) as id_tipomovimentacao,
    ds_tipo_mov,
    cast(movimentacao_data as timestamp) as dt_evento,
    cast(id_estabelecimentosecurity as string) as id_estabelecimento,
    observacao,
    categoria_movimentacao,
    cast(null as string) as subcategoria_evento,
    cast(ids_alvara as string) as ids_alvara,
    cast(qtd_alvaras as bigint) as qtd_alvaras,
    cast(ids_artigo as string) as ids_artigo,
    cast(ds_tipificacao_penal as string) as ds_tipificacao_penal,
    cast(ds_tipificacao_penal_principal as string) as ds_tipificacao_penal_principal,
    cast(qtd_tipificacoes_penais as bigint) as qtd_tipificacoes_penais,
    cast(ids_estabelecimento_externo as string) as ids_estabelecimento_externo,
    cast(ds_estabelecimento_externo as string) as ds_estabelecimento_externo,
    cast(ids_estabelecimento_security as string) as ids_estabelecimento_security,
    cast(ids_estabelecimento_anterior as string) as ids_estabelecimento_anterior,
    cast(ids_tipo_obito as string) as ids_tipo_obito,
    cast(ds_tipo_obito as string) as ds_tipo_obito,
    cast(ids_tipo_saida_temporaria as string) as ids_tipo_saida_temporaria,
    cast(ds_tipo_saida_temporaria as string) as ds_tipo_saida_temporaria,
    cast(dt_retorno_saida_temporaria as timestamp) as dt_retorno_saida_temporaria,
    1 as prioridade_categoria
from gold.sinp_ent_mov_entrada
where id_preso is not null
  and movimentacao_data is not null

union all

select
    cast(id_preso as string) as id_preso,
    cast(id_pessoa as string) as id_pessoa,
    nome_pessoa,
    cast(id_movimentacao as string) as id_movimentacao,
    cast(id_tipomovimentacao as string) as id_tipomovimentacao,
    ds_tipo_mov,
    cast(movimentacao_data as timestamp) as dt_evento,
    cast(id_estabelecimentosecurity as string) as id_estabelecimento,
    observacao,
    categoria_movimentacao,
    cast(null as string) as subcategoria_evento,
    cast(ids_alvara as string) as ids_alvara,
    cast(qtd_alvaras as bigint) as qtd_alvaras,
    cast(ids_artigo as string) as ids_artigo,
    cast(ds_tipificacao_penal as string) as ds_tipificacao_penal,
    cast(ds_tipificacao_penal_principal as string) as ds_tipificacao_penal_principal,
    cast(qtd_tipificacoes_penais as bigint) as qtd_tipificacoes_penais,
    cast(ids_estabelecimento_externo as string) as ids_estabelecimento_externo,
    cast(ds_estabelecimento_externo as string) as ds_estabelecimento_externo,
    cast(ids_estabelecimento_security as string) as ids_estabelecimento_security,
    cast(ids_estabelecimento_anterior as string) as ids_estabelecimento_anterior,
    cast(ids_tipo_obito as string) as ids_tipo_obito,
    cast(ds_tipo_obito as string) as ds_tipo_obito,
    cast(ids_tipo_saida_temporaria as string) as ids_tipo_saida_temporaria,
    cast(ds_tipo_saida_temporaria as string) as ds_tipo_saida_temporaria,
    cast(dt_retorno_saida_temporaria as timestamp) as dt_retorno_saida_temporaria,
    2 as prioridade_categoria
from gold.sinp_ent_mov_saida
where id_preso is not null
  and movimentacao_data is not null
""")

df_eventos_menores = spark.sql("""
select
    cast(id_preso as string) as id_preso,
    cast(id_pessoa as string) as id_pessoa,
    nome_pessoa,
    cast(id_movimentacao as string) as id_movimentacao,
    cast(id_tipomovimentacao as string) as id_tipomovimentacao,
    ds_tipo_mov,
    cast(movimentacao_data as timestamp) as dt_evento,
    cast(id_estabelecimentosecurity as string) as id_estabelecimento,
    observacao,
    categoria_movimentacao,
    cast(subcategoria_saidinha as string) as subcategoria_evento,
    cast(ids_alvara as string) as ids_alvara,
    cast(qtd_alvaras as bigint) as qtd_alvaras,
    cast(ids_artigo as string) as ids_artigo,
    cast(ds_tipificacao_penal as string) as ds_tipificacao_penal,
    cast(ds_tipificacao_penal_principal as string) as ds_tipificacao_penal_principal,
    cast(qtd_tipificacoes_penais as bigint) as qtd_tipificacoes_penais,
    cast(ids_estabelecimento_externo as string) as ids_estabelecimento_externo,
    cast(ds_estabelecimento_externo as string) as ds_estabelecimento_externo,
    cast(ids_estabelecimento_security as string) as ids_estabelecimento_security,
    cast(ids_estabelecimento_anterior as string) as ids_estabelecimento_anterior,
    cast(ids_tipo_obito as string) as ids_tipo_obito,
    cast(ds_tipo_obito as string) as ds_tipo_obito,
    cast(ids_tipo_saida_temporaria as string) as ids_tipo_saida_temporaria,
    cast(ds_tipo_saida_temporaria as string) as ds_tipo_saida_temporaria,
    cast(dt_retorno_saida_temporaria as timestamp) as dt_retorno_saida_temporaria,
    case
        when categoria_movimentacao = 'MOVIMENTACOES_INTERNAS' then 3
        when categoria_movimentacao = 'MOVIMENTACOES_EXTERNAS' then 4
        when categoria_movimentacao = 'MOVIMENTACOES_SAIDINHA' then 5
        else 9
    end as prioridade_categoria
from (
    select * from gold.sinp_ent_mov_interna
    union all
    select * from gold.sinp_ent_mov_externa
    union all
    select * from gold.sinp_ent_mov_saidinha
) x
where id_preso is not null
  and movimentacao_data is not null
""")

df_eventos_todos = (
    df_eventos_macro
    .unionByName(df_eventos_menores)
    .withColumn(
        "ord_tp_evento",
        F.when(F.col("categoria_movimentacao") == "ENTRADA", F.lit(1))
         .when(F.col("categoria_movimentacao") == "SAIDA", F.lit(2))
         .when(F.col("categoria_movimentacao") == "MOVIMENTACOES_INTERNAS", F.lit(3))
         .when(F.col("categoria_movimentacao") == "MOVIMENTACOES_EXTERNAS", F.lit(4))
         .when(F.col("categoria_movimentacao") == "MOVIMENTACOES_SAIDINHA", F.lit(5))
         .otherwise(F.lit(9))
    )
    .repartition("id_preso")
    .sortWithinPartitions("id_preso", "dt_evento", "ord_tp_evento", "prioridade_categoria", "id_movimentacao")
)

# ============================================================
# 2. SCHEMAS
# ============================================================

schema_encarceramento = T.StructType([
    T.StructField("id_encarceramento", T.StringType(), False),
    T.StructField("id_preso", T.StringType(), True),
    T.StructField("id_pessoa", T.StringType(), True),
    T.StructField("nome_pessoa", T.StringType(), True),
    T.StructField("nr_periodo_encarceramento", T.IntegerType(), True),

    T.StructField("id_movimentacao_entrada", T.StringType(), True),
    T.StructField("id_tipomovimentacao_entrada", T.StringType(), True),
    T.StructField("ds_tipo_mov_entrada", T.StringType(), True),
    T.StructField("dt_entrada", T.TimestampType(), True),
    T.StructField("id_estabelecimento_entrada", T.StringType(), True),
    T.StructField("observacao_entrada", T.StringType(), True),
    T.StructField("ids_artigo_entrada", T.StringType(), True),
    T.StructField("ds_tipificacao_penal_entrada", T.StringType(), True),
    T.StructField("ds_tipificacao_penal_principal_entrada", T.StringType(), True),

    T.StructField("id_movimentacao_saida", T.StringType(), True),
    T.StructField("id_tipomovimentacao_saida", T.StringType(), True),
    T.StructField("ds_tipo_mov_saida", T.StringType(), True),
    T.StructField("dt_saida", T.TimestampType(), True),
    T.StructField("id_estabelecimento_saida", T.StringType(), True),
    T.StructField("observacao_saida", T.StringType(), True),
    T.StructField("tp_fechamento", T.StringType(), True),

    T.StructField("st_encarceramento", T.StringType(), True),
    T.StructField("qt_dias_encarceramento", T.IntegerType(), True),

    T.StructField("qtd_mov_internas", T.IntegerType(), True),
    T.StructField("qtd_mov_externas", T.IntegerType(), True),
    T.StructField("qtd_mov_saidinha", T.IntegerType(), True),
    T.StructField("qtd_saida_saidinha", T.IntegerType(), True),
    T.StructField("qtd_retorno_saidinha", T.IntegerType(), True),
    T.StructField("qtd_alvaras_periodo", T.IntegerType(), True),

    T.StructField("dt_primeira_mov_interna", T.TimestampType(), True),
    T.StructField("dt_ultima_mov_interna", T.TimestampType(), True),
    T.StructField("dt_primeira_mov_externa", T.TimestampType(), True),
    T.StructField("dt_ultima_mov_externa", T.TimestampType(), True),
    T.StructField("dt_primeira_saidinha", T.TimestampType(), True),
    T.StructField("dt_ultima_saidinha", T.TimestampType(), True),

    T.StructField("ds_eventos_internos", T.StringType(), True),
    T.StructField("ds_eventos_externos", T.StringType(), True),
    T.StructField("ds_eventos_saidinha", T.StringType(), True),

    T.StructField("score_comportamental", T.DoubleType(), True),
    T.StructField("perfil_comportamental", T.StringType(), True),

    T.StructField("fl_inconsistente", T.StringType(), True)
])

schema_evento_periodo = T.StructType([
    T.StructField("id_encarceramento_evento", T.StringType(), False),
    T.StructField("id_encarceramento", T.StringType(), True),
    T.StructField("id_preso", T.StringType(), True),
    T.StructField("id_pessoa", T.StringType(), True),
    T.StructField("nome_pessoa", T.StringType(), True),
    T.StructField("nr_periodo_encarceramento", T.IntegerType(), True),
    T.StructField("categoria_movimentacao", T.StringType(), True),
    T.StructField("subcategoria_evento", T.StringType(), True),
    T.StructField("id_movimentacao", T.StringType(), True),
    T.StructField("id_tipomovimentacao", T.StringType(), True),
    T.StructField("ds_tipo_mov", T.StringType(), True),
    T.StructField("dt_evento", T.TimestampType(), True),
    T.StructField("id_estabelecimento", T.StringType(), True),
    T.StructField("observacao", T.StringType(), True),
    T.StructField("ids_alvara", T.StringType(), True),
    T.StructField("qtd_alvaras", T.LongType(), True),
    T.StructField("ids_artigo", T.StringType(), True),
    T.StructField("ds_tipificacao_penal", T.StringType(), True),
    T.StructField("ds_tipificacao_penal_principal", T.StringType(), True),
    T.StructField("qtd_tipificacoes_penais", T.LongType(), True),
    T.StructField("ids_estabelecimento_externo", T.StringType(), True),
    T.StructField("ds_estabelecimento_externo", T.StringType(), True),
    T.StructField("ids_estabelecimento_security", T.StringType(), True),
    T.StructField("ids_estabelecimento_anterior", T.StringType(), True),
    T.StructField("ids_tipo_obito", T.StringType(), True),
    T.StructField("ds_tipo_obito", T.StringType(), True),
    T.StructField("ids_tipo_saida_temporaria", T.StringType(), True),
    T.StructField("ds_tipo_saida_temporaria", T.StringType(), True),
    T.StructField("dt_retorno_saida_temporaria", T.TimestampType(), True)
])

schema_inconsistencia = T.StructType([
    T.StructField("id_encarceramento_inconsistencia", T.StringType(), False),
    T.StructField("id_preso", T.StringType(), True),
    T.StructField("id_pessoa", T.StringType(), True),
    T.StructField("nome_pessoa", T.StringType(), True),
    T.StructField("nr_periodo_encarceramento", T.IntegerType(), True),
    T.StructField("id_movimentacao_ref", T.StringType(), True),
    T.StructField("categoria_movimentacao_ref", T.StringType(), True),
    T.StructField("ds_tipo_mov_ref", T.StringType(), True),
    T.StructField("dt_evento_ref", T.TimestampType(), True),
    T.StructField("tp_inconsistencia", T.StringType(), True),
    T.StructField("detalhe_inconsistencia", T.StringType(), True)
])

# ============================================================
# 3. PROCESSAMENTO PURO PYTHON POR PRESO
# ============================================================

def processar_lista_eventos(eventos):
    eventos = sorted(
        eventos,
        key=lambda r: (
            r["dt_evento"],
            r["ord_tp_evento"],
            r["prioridade_categoria"],
            r["id_movimentacao"] if r["id_movimentacao"] is not None else ""
        )
    )

    encarceramentos = []
    eventos_periodo = []
    inconsistencias = []

    aberto = None
    nr_periodo = 0
    hoje = _today_midnight()

    def novo_periodo(r, nr):
        return {
            "id_preso": r["id_preso"],
            "id_pessoa": r["id_pessoa"],
            "nome_pessoa": r["nome_pessoa"],
            "nr_periodo_encarceramento": int(nr),

            "id_movimentacao_entrada": r["id_movimentacao"],
            "id_tipomovimentacao_entrada": r["id_tipomovimentacao"],
            "ds_tipo_mov_entrada": r["ds_tipo_mov"],
            "dt_entrada": _to_python_dt(r["dt_evento"]),
            "id_estabelecimento_entrada": r["id_estabelecimento"],
            "observacao_entrada": r["observacao"],
            "ids_artigo_entrada": r["ids_artigo"],
            "ds_tipificacao_penal_entrada": r["ds_tipificacao_penal"],
            "ds_tipificacao_penal_principal_entrada": r["ds_tipificacao_penal_principal"],

            "id_movimentacao_saida": None,
            "id_tipomovimentacao_saida": None,
            "ds_tipo_mov_saida": None,
            "dt_saida": None,
            "id_estabelecimento_saida": None,
            "observacao_saida": None,
            "tp_fechamento": None,

            "eventos": [],
            "fl_inconsistente": "N"
        }

    def adicionar_evento(periodo, r):
        periodo["eventos"].append({
            "categoria_movimentacao": r["categoria_movimentacao"],
            "subcategoria_evento": r["subcategoria_evento"],
            "id_movimentacao": r["id_movimentacao"],
            "id_tipomovimentacao": r["id_tipomovimentacao"],
            "ds_tipo_mov": r["ds_tipo_mov"],
            "dt_evento": _to_python_dt(r["dt_evento"]),
            "id_estabelecimento": r["id_estabelecimento"],
            "observacao": r["observacao"],
            "ids_alvara": r["ids_alvara"],
            "qtd_alvaras": int(r["qtd_alvaras"]) if r["qtd_alvaras"] is not None else None,
            "ids_artigo": r["ids_artigo"],
            "ds_tipificacao_penal": r["ds_tipificacao_penal"],
            "ds_tipificacao_penal_principal": r["ds_tipificacao_penal_principal"],
            "qtd_tipificacoes_penais": int(r["qtd_tipificacoes_penais"]) if r["qtd_tipificacoes_penais"] is not None else None,
            "ids_estabelecimento_externo": r["ids_estabelecimento_externo"],
            "ds_estabelecimento_externo": r["ds_estabelecimento_externo"],
            "ids_estabelecimento_security": r["ids_estabelecimento_security"],
            "ids_estabelecimento_anterior": r["ids_estabelecimento_anterior"],
            "ids_tipo_obito": r["ids_tipo_obito"],
            "ds_tipo_obito": r["ds_tipo_obito"],
            "ids_tipo_saida_temporaria": r["ids_tipo_saida_temporaria"],
            "ds_tipo_saida_temporaria": r["ds_tipo_saida_temporaria"],
            "dt_retorno_saida_temporaria": _to_python_dt(r["dt_retorno_saida_temporaria"])
        })

    def fechar_periodo(periodo, r_saida=None, dt_fechamento_forcado=None, tp_fechamento="SAIDA"):
        nonlocal encarceramentos, eventos_periodo

        if r_saida is not None:
            periodo["id_movimentacao_saida"] = r_saida["id_movimentacao"]
            periodo["id_tipomovimentacao_saida"] = r_saida["id_tipomovimentacao"]
            periodo["ds_tipo_mov_saida"] = r_saida["ds_tipo_mov"]
            periodo["dt_saida"] = _to_python_dt(r_saida["dt_evento"])
            periodo["id_estabelecimento_saida"] = r_saida["id_estabelecimento"]
            periodo["observacao_saida"] = r_saida["observacao"]
            periodo["tp_fechamento"] = tp_fechamento
        else:
            periodo["dt_saida"] = dt_fechamento_forcado
            periodo["tp_fechamento"] = tp_fechamento

        id_enc = _hash(
            periodo["id_preso"],
            periodo["nr_periodo_encarceramento"],
            periodo["id_movimentacao_entrada"],
            periodo["dt_entrada"]
        )

        eventos = periodo["eventos"]
        ev_interna = [e for e in eventos if e["categoria_movimentacao"] == "MOVIMENTACOES_INTERNAS"]
        ev_externa = [e for e in eventos if e["categoria_movimentacao"] == "MOVIMENTACOES_EXTERNAS"]
        ev_saidinha = [e for e in eventos if e["categoria_movimentacao"] == "MOVIMENTACOES_SAIDINHA"]

        def min_dt(lst):
            return min([e["dt_evento"] for e in lst]) if lst else None

        def max_dt(lst):
            return max([e["dt_evento"] for e in lst]) if lst else None

        def txt_eventos(lst):
            vals = sorted(set([str(e["ds_tipo_mov"]) for e in lst if e["ds_tipo_mov"] is not None]))
            return ",".join(vals) if vals else None

        qtd_saida_saidinha = sum(1 for e in ev_saidinha if e["subcategoria_evento"] == "SAIDA_SAIDINHA")
        qtd_retorno_saidinha = sum(1 for e in ev_saidinha if e["subcategoria_evento"] == "RETORNO_SAIDINHA")
        qtd_alvaras_periodo = sum(int(e["qtd_alvaras"]) for e in eventos if e["qtd_alvaras"] is not None)

        dt_fim = periodo["dt_saida"] if periodo["dt_saida"] is not None else hoje
        qt_dias = int((dt_fim.date() - periodo["dt_entrada"].date()).days)

        score = (
            len(ev_interna) * 1.0
            + len(ev_externa) * 2.0
            + len(ev_saidinha) * 1.5
            + qtd_alvaras_periodo * 2.0
        )

        if score >= 20:
            perfil = "ALTA_DINAMICA"
        elif score >= 8:
            perfil = "MEDIA_DINAMICA"
        else:
            perfil = "BAIXA_DINAMICA"

        encarceramentos.append((
            id_enc,
            periodo["id_preso"],
            periodo["id_pessoa"],
            periodo["nome_pessoa"],
            int(periodo["nr_periodo_encarceramento"]),

            periodo["id_movimentacao_entrada"],
            periodo["id_tipomovimentacao_entrada"],
            periodo["ds_tipo_mov_entrada"],
            periodo["dt_entrada"],
            periodo["id_estabelecimento_entrada"],
            periodo["observacao_entrada"],
            periodo["ids_artigo_entrada"],
            periodo["ds_tipificacao_penal_entrada"],
            periodo["ds_tipificacao_penal_principal_entrada"],

            periodo["id_movimentacao_saida"],
            periodo["id_tipomovimentacao_saida"],
            periodo["ds_tipo_mov_saida"],
            periodo["dt_saida"],
            periodo["id_estabelecimento_saida"],
            periodo["observacao_saida"],
            periodo["tp_fechamento"],

            "FECHADO" if periodo["dt_saida"] is not None else "ABERTO",
            qt_dias,

            len(ev_interna),
            len(ev_externa),
            len(ev_saidinha),
            qtd_saida_saidinha,
            qtd_retorno_saidinha,
            qtd_alvaras_periodo,

            min_dt(ev_interna),
            max_dt(ev_interna),
            min_dt(ev_externa),
            max_dt(ev_externa),
            min_dt(ev_saidinha),
            max_dt(ev_saidinha),

            txt_eventos(ev_interna),
            txt_eventos(ev_externa),
            txt_eventos(ev_saidinha),

            float(score),
            perfil,

            periodo["fl_inconsistente"]
        ))

        for e in eventos:
            eventos_periodo.append((
                _hash(id_enc, e["id_movimentacao"], e["categoria_movimentacao"]),
                id_enc,
                periodo["id_preso"],
                periodo["id_pessoa"],
                periodo["nome_pessoa"],
                int(periodo["nr_periodo_encarceramento"]),
                e["categoria_movimentacao"],
                e["subcategoria_evento"],
                e["id_movimentacao"],
                e["id_tipomovimentacao"],
                e["ds_tipo_mov"],
                e["dt_evento"],
                e["id_estabelecimento"],
                e["observacao"],
                e["ids_alvara"],
                e["qtd_alvaras"],
                e["ids_artigo"],
                e["ds_tipificacao_penal"],
                e["ds_tipificacao_penal_principal"],
                e["qtd_tipificacoes_penais"],
                e["ids_estabelecimento_externo"],
                e["ds_estabelecimento_externo"],
                e["ids_estabelecimento_security"],
                e["ids_estabelecimento_anterior"],
                e["ids_tipo_obito"],
                e["ds_tipo_obito"],
                e["ids_tipo_saida_temporaria"],
                e["ds_tipo_saida_temporaria"],
                e["dt_retorno_saida_temporaria"]
            ))

    for r in eventos:
        cat = r["categoria_movimentacao"]

        if cat == "ENTRADA":
            if aberto is None:
                nr_periodo += 1
                aberto = novo_periodo(r, nr_periodo)
            else:
                dt_forcada = _to_python_dt(r["dt_evento"]) - timedelta(seconds=1)
                fechar_periodo(
                    aberto,
                    r_saida=None,
                    dt_fechamento_forcado=dt_forcada,
                    tp_fechamento="AJUSTE_NOVA_ENTRADA"
                )

                inconsistencias.append((
                    _hash(r["id_preso"], r["id_movimentacao"], "ENTRADA_SEM_SAIDA_ANTERIOR"),
                    r["id_preso"],
                    r["id_pessoa"],
                    r["nome_pessoa"],
                    int(aberto["nr_periodo_encarceramento"]),
                    r["id_movimentacao"],
                    cat,
                    r["ds_tipo_mov"],
                    _to_python_dt(r["dt_evento"]),
                    "ENTRADA_SEM_SAIDA_ANTERIOR",
                    "Periodo anterior fechado artificialmente em nova entrada - 1 segundo"
                ))

                nr_periodo += 1
                aberto = novo_periodo(r, nr_periodo)

        elif cat == "SAIDA":
            if aberto is None:
                inconsistencias.append((
                    _hash(r["id_preso"], r["id_movimentacao"], "SAIDA_SEM_ENTRADA"),
                    r["id_preso"],
                    r["id_pessoa"],
                    r["nome_pessoa"],
                    None,
                    r["id_movimentacao"],
                    cat,
                    r["ds_tipo_mov"],
                    _to_python_dt(r["dt_evento"]),
                    "SAIDA_SEM_ENTRADA",
                    "Saida encontrada sem periodo aberto"
                ))
            else:
                if _to_python_dt(r["dt_evento"]) < aberto["dt_entrada"]:
                    inconsistencias.append((
                        _hash(r["id_preso"], r["id_movimentacao"], "SAIDA_ANTERIOR_ENTRADA"),
                        r["id_preso"],
                        r["id_pessoa"],
                        r["nome_pessoa"],
                        int(aberto["nr_periodo_encarceramento"]),
                        r["id_movimentacao"],
                        cat,
                        r["ds_tipo_mov"],
                        _to_python_dt(r["dt_evento"]),
                        "SAIDA_ANTERIOR_ENTRADA",
                        "Saida com data anterior a entrada do periodo"
                    ))
                else:
                    fechar_periodo(aberto, r_saida=r, tp_fechamento="SAIDA")
                    aberto = None

        else:
            if aberto is None:
                inconsistencias.append((
                    _hash(r["id_preso"], r["id_movimentacao"], "EVENTO_MENOR_FORA_PERIODO"),
                    r["id_preso"],
                    r["id_pessoa"],
                    r["nome_pessoa"],
                    None,
                    r["id_movimentacao"],
                    cat,
                    r["ds_tipo_mov"],
                    _to_python_dt(r["dt_evento"]),
                    "EVENTO_MENOR_FORA_PERIODO",
                    "Evento menor sem encarceramento aberto para agrupamento"
                ))
            else:
                adicionar_evento(aberto, r)

    if aberto is not None:
        fechar_periodo(aberto, r_saida=None, dt_fechamento_forcado=None, tp_fechamento=None)

    return encarceramentos, eventos_periodo, inconsistencias

def iter_grouped_by_preso(rows_iter):
    current_id = None
    bucket = []

    for row in rows_iter:
        d = row.asDict(recursive=True)
        k = d["id_preso"]

        if current_id is None:
            current_id = k

        if k != current_id:
            yield current_id, bucket
            current_id = k
            bucket = [d]
        else:
            bucket.append(d)

    if current_id is not None:
        yield current_id, bucket

def map_encarceramento(rows_iter):
    for _, eventos in iter_grouped_by_preso(rows_iter):
        encs, _, _ = processar_lista_eventos(eventos)
        for x in encs:
            yield x

def map_eventos(rows_iter):
    for _, eventos in iter_grouped_by_preso(rows_iter):
        _, evs, _ = processar_lista_eventos(eventos)
        for x in evs:
            yield x

def map_inconsistencias(rows_iter):
    for _, eventos in iter_grouped_by_preso(rows_iter):
        _, _, incs = processar_lista_eventos(eventos)
        for x in incs:
            yield x

# ============================================================
# 4. EXECUCAO SEM PANDAS
# ============================================================

base_rdd = df_eventos_todos.rdd

df_fat_encarceramento = spark.createDataFrame(
    base_rdd.mapPartitions(map_encarceramento),
    schema=schema_encarceramento
)

df_fat_encarceramento_evento = spark.createDataFrame(
    base_rdd.mapPartitions(map_eventos),
    schema=schema_evento_periodo
)

df_fat_encarceramento_inconsistencia = spark.createDataFrame(
    base_rdd.mapPartitions(map_inconsistencias),
    schema=schema_inconsistencia
)

# ============================================================
# 5. AJUSTES FINAIS
# ============================================================

df_fat_encarceramento = df_fat_encarceramento.dropDuplicates(["id_encarceramento"])
df_fat_encarceramento_evento = df_fat_encarceramento_evento.dropDuplicates(["id_encarceramento_evento"])
df_fat_encarceramento_inconsistencia = df_fat_encarceramento_inconsistencia.dropDuplicates(["id_encarceramento_inconsistencia"])

# ============================================================
# 6. PERSISTENCIA
# ============================================================

tabela = "sinp_fat_encarceramento"

df_fat_encarceramento.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_fat_encarceramento, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres(f"gold.{tabela}", "id_encarceramento")

tabela = "sinp_fat_encarceramento_evento"

df_fat_encarceramento_evento.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_fat_encarceramento_evento, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres(f"gold.{tabela}", "id_encarceramento_evento")

tabela = "sinp_fat_encarceramento_inconsistencia"

df_fat_encarceramento_inconsistencia.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_fat_encarceramento_inconsistencia, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres(f"gold.{tabela}", "id_encarceramento_inconsistencia")

In [ ]:
# ============================================================
# LIMPEZA DEFENSIVA DAS TABELAS TEMPORARIAS DA FATO
# ============================================================

spark.sql("drop table if exists gold.tmp_base_controle_advogado")
spark.sql("drop table if exists gold.tmp_base_vinculo_advogado")
spark.sql("drop table if exists gold.tmp_base_cadastro_advogado")
spark.sql("drop table if exists gold.tmp_base_interno_livro")
spark.sql("drop table if exists gold.tmp_base_pessoa_preso_ponte")
spark.sql("drop table if exists gold.tmp_base_presidiario_resolvido")
spark.sql("drop table if exists gold.tmp_base_restricao_advogado")
spark.sql("drop table if exists gold.tmp_base_restricao_advogado_interno")
spark.sql("drop table if exists gold.tmp_base_restricao_advogado_presidio")
spark.sql("drop table if exists gold.tmp_base_restricao_advogado_global")
spark.sql("drop table if exists gold.tmp_base_pessoa_advogado")
spark.sql("drop table if exists gold.tmp_evento_base_visita_advogado")
spark.sql("drop table if exists gold.tmp_evento_enriquecido_visita_advogado")
spark.sql("drop table if exists gold.tmp_evento_calculado_visita_advogado")
spark.sql("drop table if exists gold.tmp_evento_rank_visita_advogado")
spark.sql("drop table if exists gold.sinp_fat_visita_advogado")

import os
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_base_controle_advogado >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_base_vinculo_advogado >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_base_cadastro_advogado >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_base_interno_livro >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_base_pessoa_preso_ponte >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_base_presidiario_resolvido >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_base_restricao_advogado >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_base_restricao_advogado_interno >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_base_restricao_advogado_presidio >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_base_restricao_advogado_global >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_base_pessoa_advogado >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_evento_base_visita_advogado >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_evento_enriquecido_visita_advogado >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_evento_calculado_visita_advogado >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_evento_rank_visita_advogado >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}sinp_fat_visita_advogado >/dev/null 2>&1")

spark.catalog.clearCache()
spark.sql("refresh table gold.sinp_ent_pessoa")
spark.sql("refresh table gold.sinp_pnt_pessoa_preso")


# ============================================================
# BASE CONTROLE ADVOGADO
# ============================================================

df_base_controle_advogado = spark.sql("""
    select
        cast(id as string) as id_evento_origem,
        cast(vinculos_id as string) as id_vinculo_origem,
        cast(presidio_id as string) as id_presidio_controle,
        cast(equipe_id as string) as id_equipe_origem,
        to_timestamp(hr_entrada) as dt_hr_entrada,
        to_timestamp(hr_saida) as dt_hr_saida,
        to_timestamp(data_registro) as dt_registro,
        case
            when hr_entrada is not null then date_format(hr_entrada, 'HH:mm:ss')
            else null
        end as hr_entrada,
        case
            when hr_saida is not null then date_format(hr_saida, 'HH:mm:ss')
            else null
        end as hr_saida,
        trim(regexp_replace(coalesce(autorizacao, ''), '\\\\s+', ' ')) as autorizacao,
        trim(regexp_replace(coalesce(motivo, ''), '\\\\s+', ' ')) as motivo
    from bronze.livros_acesso_unidade_controleadvogado
""")

tabela = "tmp_base_controle_advogado"

df_base_controle_advogado.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_base_controle_advogado, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_controle_advogado")


# ============================================================
# BASE VINCULO ADVOGADO
# ============================================================

df_base_vinculo_advogado = spark.sql("""
    select
        cast(id as string) as id_vinculo_origem,
        cast(advogado_id_id as string) as id_advogado_origem,
        cast(interno_id_id as string) as id_interno_origem,
        cast(presidio_id as string) as id_presidio_vinculo,
        coalesce(inativo, false) as inativo,
        coalesce(procuracao, false) as procuracao,
        to_timestamp(data_registro) as dt_registro_vinculo,
        trim(regexp_replace(coalesce(advogado, ''), '\\\\s+', ' ')) as nome_advogado_vinculo,
        trim(regexp_replace(coalesce(interno, ''), '\\\\s+', ' ')) as nome_interno_vinculo,
        trim(regexp_replace(coalesce(arquivo, ''), '\\\\s+', ' ')) as arquivo
    from bronze.livros_acesso_unidade_vinculavisitaadvogado
""")

tabela = "tmp_base_vinculo_advogado"

df_base_vinculo_advogado.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_base_vinculo_advogado, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_vinculo_advogado")


# ============================================================
# BASE CADASTRO ADVOGADO
# ============================================================

df_base_cadastro_advogado = spark.sql("""
    select
        cast(id as string) as id_advogado_origem,
        trim(regexp_replace(coalesce(nome, ''), '\\\\s+', ' ')) as nome_advogado_cadastro,
        upper(trim(regexp_replace(coalesce(estado, ''), '\\\\s+', ' '))) as estado_oab,
        upper(trim(regexp_replace(coalesce(oab, ''), '\\\\s+', ' '))) as oab_bruta,
        upper(regexp_replace(coalesce(oab, ''), '[^0-9A-Za-z]', '')) as oab_normalizada,
        concat(
            'OAB_',
            upper(trim(regexp_replace(coalesce(estado, ''), '\\\\s+', ' '))),
            '_',
            upper(regexp_replace(coalesce(oab, ''), '[^0-9A-Za-z]', ''))
        ) as id_pessoa_advogado_chave,
        concat(
            upper(regexp_replace(coalesce(oab, ''), '[^0-9A-Za-z]', '')),
            '/',
            upper(trim(regexp_replace(coalesce(estado, ''), '\\\\s+', ' ')))
        ) as documento_advogado_oab
    from bronze.livros_acesso_unidade_advogado
""")

tabela = "tmp_base_cadastro_advogado"

df_base_cadastro_advogado.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_base_cadastro_advogado, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_cadastro_advogado")


# ============================================================
# BASE INTERNO LIVRO
# ============================================================

df_base_interno_livro = spark.sql("""
    select
        cast(id as string) as id_interno_origem,
        cast(infopen as string) as id_preso_infopen
    from bronze.livros_acesso_unidade_interno
    where infopen is not null
""")

tabela = "tmp_base_interno_livro"

df_base_interno_livro.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_base_interno_livro, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_interno_livro")


# ============================================================
# BASE PESSOA PRESO PONTE
# ============================================================

df_base_pessoa_preso_ponte = spark.sql("""
    select distinct
        cast(id_preso as string) as id_preso_infopen,
        cast(id_preso as string) as id_preso_presidiario,
        id_pessoa as id_pessoa_presidiario
    from gold.sinp_pnt_pessoa_preso
    where id_preso is not null
      and id_pessoa is not null
""")

tabela = "tmp_base_pessoa_preso_ponte"

df_base_pessoa_preso_ponte.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_base_pessoa_preso_ponte, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_pessoa_preso_ponte")


# ============================================================
# BASE PRESIDIARIO RESOLVIDO
# ============================================================

df_base_presidiario_resolvido = spark.sql("""
    select
        trim(i.id_interno_origem) as id_interno_origem,
        trim(i.id_preso_infopen) as id_preso_infopen,
        trim(p.id_preso_presidiario) as id_preso_presidiario,
        p.id_pessoa_presidiario,
        e.documento as documento_presidiario,
        e.nome_pessoa as nome_presidiario_pessoa
    from gold.tmp_base_interno_livro i
    inner join gold.tmp_base_pessoa_preso_ponte p
        on trim(i.id_preso_infopen) = trim(p.id_preso_infopen)
    left join gold.sinp_ent_pessoa e
        on p.id_pessoa_presidiario = e.id_pessoa
""")

tabela = "tmp_base_presidiario_resolvido"

df_base_presidiario_resolvido.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_base_presidiario_resolvido, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_presidiario_resolvido")


# ============================================================
# BASE RESTRICAO ADVOGADO
# ============================================================

df_base_restricao_advogado = spark.sql("""
    select
        cast(advogado_id as string) as id_advogado_origem,
        cast(interno_id as string) as id_interno_origem,
        cast(presidio_id as string) as id_presidio_origem,
        max(case when coalesce(bloquear_todos_internos, false) = true then 1 else 0 end) as flag_bloqueia_todos_internos,
        max(case when coalesce(bloquear_todos_presidios, false) = true then 1 else 0 end) as flag_bloqueia_todos_presidios,
        count(*) as qtd_restricoes
    from bronze.livros_acesso_unidade_restricaoadvogado
    group by
        cast(advogado_id as string),
        cast(interno_id as string),
        cast(presidio_id as string)
""")

tabela = "tmp_base_restricao_advogado"

df_base_restricao_advogado.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_base_restricao_advogado, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_restricao_advogado")


# ============================================================
# RESTRICAO POR ADVOGADO + INTERNO
# ============================================================

df_base_restricao_advogado_interno = spark.sql("""
    select
        trim(id_advogado_origem) as id_advogado_origem,
        trim(id_interno_origem) as id_interno_origem,
        max(flag_bloqueia_todos_internos) as flag_bloqueia_todos_internos_interno,
        max(flag_bloqueia_todos_presidios) as flag_bloqueia_todos_presidios_interno,
        sum(qtd_restricoes) as qtd_restricoes_interno
    from gold.tmp_base_restricao_advogado
    group by
        trim(id_advogado_origem),
        trim(id_interno_origem)
""")

tabela = "tmp_base_restricao_advogado_interno"

df_base_restricao_advogado_interno.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_base_restricao_advogado_interno, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_restricao_advogado_interno")


# ============================================================
# RESTRICAO POR ADVOGADO + PRESIDIO
# ============================================================

df_base_restricao_advogado_presidio = spark.sql("""
    select
        trim(id_advogado_origem) as id_advogado_origem,
        trim(id_presidio_origem) as id_presidio_origem,
        max(flag_bloqueia_todos_internos) as flag_bloqueia_todos_internos_presidio,
        max(flag_bloqueia_todos_presidios) as flag_bloqueia_todos_presidios_presidio,
        sum(qtd_restricoes) as qtd_restricoes_presidio
    from gold.tmp_base_restricao_advogado
    group by
        trim(id_advogado_origem),
        trim(id_presidio_origem)
""")

tabela = "tmp_base_restricao_advogado_presidio"

df_base_restricao_advogado_presidio.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_base_restricao_advogado_presidio, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_restricao_advogado_presidio")


# ============================================================
# RESTRICAO GLOBAL POR ADVOGADO
# ============================================================

df_base_restricao_advogado_global = spark.sql("""
    select
        trim(id_advogado_origem) as id_advogado_origem,
        max(flag_bloqueia_todos_internos) as flag_bloqueia_todos_internos_global,
        max(flag_bloqueia_todos_presidios) as flag_bloqueia_todos_presidios_global,
        sum(qtd_restricoes) as qtd_restricoes_global
    from gold.tmp_base_restricao_advogado
    group by trim(id_advogado_origem)
""")

tabela = "tmp_base_restricao_advogado_global"

df_base_restricao_advogado_global.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_base_restricao_advogado_global, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_restricao_advogado_global")


# ============================================================
# BASE PESSOA ADVOGADO
# ============================================================

df_base_pessoa_advogado = spark.sql("""
    select distinct
        id_pessoa as id_pessoa_advogado,
        documento as documento_advogado,
        nome_pessoa as nome_advogado_pessoa
    from gold.sinp_ent_pessoa
    where coalesce(flag_advogado, 0) = 1
""")

tabela = "tmp_base_pessoa_advogado"

df_base_pessoa_advogado.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_base_pessoa_advogado, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_pessoa_advogado")


# ============================================================
# EVENTO BASE VISITA ADVOGADO
# ============================================================

df_evento_base_visita_advogado = spark.sql("""
    select
        v.id_vinculo_origem as id_vinculo_origem_raw,
        v.id_vinculo_origem as id_vinculo_origem,
        c.id_evento_origem,
        c.id_presidio_controle,
        c.id_equipe_origem,
        c.dt_hr_entrada,
        c.dt_hr_saida,
        c.dt_registro,
        c.hr_entrada,
        c.hr_saida,
        c.autorizacao,
        c.motivo,
        v.id_advogado_origem,
        v.id_interno_origem,
        v.id_presidio_vinculo,
        v.inativo,
        v.procuracao,
        v.dt_registro_vinculo,
        v.nome_advogado_vinculo,
        v.nome_interno_vinculo,
        v.arquivo,
        pr.id_preso_presidiario,
        pr.id_pessoa_presidiario,
        pr.documento_presidiario,
        pr.nome_presidiario_pessoa
    from gold.tmp_base_vinculo_advogado v
    left join gold.tmp_base_controle_advogado c
        on trim(v.id_vinculo_origem) = trim(c.id_vinculo_origem)
    left join gold.tmp_base_presidiario_resolvido pr
        on trim(v.id_interno_origem) = trim(pr.id_interno_origem)
""")

tabela = "tmp_evento_base_visita_advogado"

df_evento_base_visita_advogado.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_evento_base_visita_advogado, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_evento_base_visita_advogado")


# ============================================================
# EVENTO ENRIQUECIDO VISITA ADVOGADO
# ============================================================

df_evento_enriquecido_visita_advogado = spark.sql("""
    select
        e.id_evento_origem,
        e.id_vinculo_origem,
        e.id_advogado_origem,
        e.id_interno_origem,
        e.id_preso_presidiario,
        e.id_presidio_controle,
        e.id_presidio_vinculo,
        e.id_equipe_origem,
        cad.id_pessoa_advogado_chave,
        pa.id_pessoa_advogado,
        e.id_pessoa_presidiario,
        cad.documento_advogado_oab,
        coalesce(pa.documento_advogado, cad.documento_advogado_oab) as documento_advogado,
        coalesce(pa.nome_advogado_pessoa, cad.nome_advogado_cadastro, e.nome_advogado_vinculo) as nome_advogado,
        e.documento_presidiario,
        coalesce(e.nome_presidiario_pessoa, e.nome_interno_vinculo) as nome_presidiario,
        e.dt_hr_entrada,
        e.dt_hr_saida,
        e.hr_entrada,
        e.hr_saida,
        e.dt_registro,
        coalesce(e.dt_hr_entrada, e.dt_hr_saida, e.dt_registro, e.dt_registro_vinculo) as dt_evento_referencia,
        e.autorizacao,
        e.motivo,
        e.inativo,
        e.procuracao,
        e.dt_registro_vinculo,
        e.arquivo,
        case
            when e.id_pessoa_presidiario is not null then 'INTERNO_INFOOPEN_PONTE'
            else 'NAO_RESOLVIDO'
        end as origem_resolucao_presidiario,
        coalesce(ri.qtd_restricoes_interno, 0) as qtd_restricoes_interno,
        coalesce(rp.qtd_restricoes_presidio, 0) as qtd_restricoes_presidio,
        coalesce(rg.qtd_restricoes_global, 0) as qtd_restricoes_global,
        coalesce(ri.flag_bloqueia_todos_internos_interno, 0) as flag_bloqueia_todos_internos_interno,
        coalesce(ri.flag_bloqueia_todos_presidios_interno, 0) as flag_bloqueia_todos_presidios_interno,
        coalesce(rp.flag_bloqueia_todos_internos_presidio, 0) as flag_bloqueia_todos_internos_presidio,
        coalesce(rp.flag_bloqueia_todos_presidios_presidio, 0) as flag_bloqueia_todos_presidios_presidio,
        coalesce(rg.flag_bloqueia_todos_internos_global, 0) as flag_bloqueia_todos_internos_global,
        coalesce(rg.flag_bloqueia_todos_presidios_global, 0) as flag_bloqueia_todos_presidios_global
    from gold.tmp_evento_base_visita_advogado e
    left join gold.tmp_base_cadastro_advogado cad
        on trim(e.id_advogado_origem) = trim(cad.id_advogado_origem)
    left join gold.tmp_base_pessoa_advogado pa
        on cad.id_pessoa_advogado_chave = pa.id_pessoa_advogado
    left join gold.tmp_base_restricao_advogado_interno ri
        on trim(e.id_advogado_origem) = trim(ri.id_advogado_origem)
       and trim(e.id_interno_origem) = trim(ri.id_interno_origem)
    left join gold.tmp_base_restricao_advogado_presidio rp
        on trim(e.id_advogado_origem) = trim(rp.id_advogado_origem)
       and trim(coalesce(e.id_presidio_controle, e.id_presidio_vinculo)) = trim(rp.id_presidio_origem)
    left join gold.tmp_base_restricao_advogado_global rg
        on trim(e.id_advogado_origem) = trim(rg.id_advogado_origem)
""")

tabela = "tmp_evento_enriquecido_visita_advogado"

df_evento_enriquecido_visita_advogado.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_evento_enriquecido_visita_advogado, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_evento_enriquecido_visita_advogado")


# ============================================================
# EVENTO CALCULADO VISITA ADVOGADO
# ============================================================

df_evento_calculado_visita_advogado = spark.sql("""
    select
        concat(
            'FVA_',
            md5(
                concat_ws(
                    '|',
                    coalesce(id_evento_origem, ''),
                    coalesce(id_vinculo_origem, ''),
                    coalesce(id_advogado_origem, ''),
                    coalesce(id_interno_origem, ''),
                    coalesce(id_pessoa_presidiario, '')
                )
            )
        ) as id_fato_visita_advogado,
        id_evento_origem,
        id_vinculo_origem,
        id_advogado_origem,
        id_interno_origem,
        id_preso_presidiario,
        coalesce(id_presidio_controle, id_presidio_vinculo) as id_presidio_origem,
        id_equipe_origem,
        id_pessoa_advogado,
        id_pessoa_presidiario,
        documento_advogado,
        nome_advogado,
        documento_presidiario,
        nome_presidiario,
        origem_resolucao_presidiario,
        dt_hr_entrada,
        dt_hr_saida,
        hr_entrada,
        hr_saida,
        dt_registro,
        dt_evento_referencia,
        to_date(dt_evento_referencia) as dt_evento,
        autorizacao,
        motivo,
        coalesce(inativo, false) as inativo,
        coalesce(procuracao, false) as procuracao,
        dt_registro_vinculo,
        arquivo,
        case when dt_hr_entrada is not null then 1 else 0 end as flag_tem_entrada,
        case when dt_hr_saida is not null then 1 else 0 end as flag_tem_saida,
        case
            when dt_hr_entrada is not null and dt_hr_saida is not null and dt_hr_saida >= dt_hr_entrada then 1
            else 0
        end as flag_duracao_valida,
        case
            when coalesce(inativo, false) = false then 1
            else 0
        end as flag_vinculo_ativo,
        case
            when coalesce(procuracao, false) = true then 1
            else 0
        end as flag_procuracao,
        case
            when coalesce(autorizacao, '') <> '' then 1
            else 0
        end as flag_autorizacao_preenchida,
        case
            when coalesce(motivo, '') <> '' then 1
            else 0
        end as flag_motivo_preenchido,
        case
            when coalesce(qtd_restricoes_interno, 0) > 0 then 1
            else 0
        end as flag_restricao_mesmo_interno,
        case
            when coalesce(qtd_restricoes_presidio, 0) > 0 then 1
            else 0
        end as flag_restricao_mesmo_presidio,
        greatest(
            coalesce(flag_bloqueia_todos_internos_interno, 0),
            coalesce(flag_bloqueia_todos_internos_presidio, 0),
            coalesce(flag_bloqueia_todos_internos_global, 0)
        ) as flag_bloqueia_todos_internos,
        greatest(
            coalesce(flag_bloqueia_todos_presidios_interno, 0),
            coalesce(flag_bloqueia_todos_presidios_presidio, 0),
            coalesce(flag_bloqueia_todos_presidios_global, 0)
        ) as flag_bloqueia_todos_presidios,
        case
            when coalesce(qtd_restricoes_interno, 0) > 0
              or coalesce(qtd_restricoes_presidio, 0) > 0
              or coalesce(qtd_restricoes_global, 0) > 0
              or coalesce(flag_bloqueia_todos_internos_interno, 0) = 1
              or coalesce(flag_bloqueia_todos_presidios_interno, 0) = 1
              or coalesce(flag_bloqueia_todos_internos_presidio, 0) = 1
              or coalesce(flag_bloqueia_todos_presidios_presidio, 0) = 1
              or coalesce(flag_bloqueia_todos_internos_global, 0) = 1
              or coalesce(flag_bloqueia_todos_presidios_global, 0) = 1
            then 1
            else 0
        end as flag_visita_com_restricao,
        case
            when dt_hr_entrada is not null and dt_hr_saida is not null and dt_hr_saida >= dt_hr_entrada
            then cast((unix_timestamp(dt_hr_saida) - unix_timestamp(dt_hr_entrada)) / 60.0 as decimal(18,2))
            else cast(null as decimal(18,2))
        end as duracao_minutos
    from gold.tmp_evento_enriquecido_visita_advogado
""")

tabela = "tmp_evento_calculado_visita_advogado"

df_evento_calculado_visita_advogado.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_evento_calculado_visita_advogado, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_evento_calculado_visita_advogado")


# ============================================================
# EVENTO RANKEADO VISITA ADVOGADO
# ============================================================

df_evento_rank_visita_advogado = spark.sql("""
    select
        *,
        row_number() over (
            partition by coalesce(id_evento_origem, id_vinculo_origem)
            order by
                case when id_pessoa_advogado is not null then 1 else 2 end,
                case when id_pessoa_presidiario is not null then 1 else 2 end,
                case when id_preso_presidiario is not null then 1 else 2 end,
                case when dt_evento_referencia is not null then 1 else 2 end,
                dt_evento_referencia desc,
                id_vinculo_origem desc
        ) as rn
    from gold.tmp_evento_calculado_visita_advogado
""")

tabela = "tmp_evento_rank_visita_advogado"

df_evento_rank_visita_advogado.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_evento_rank_visita_advogado, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_evento_rank_visita_advogado")


# ============================================================
# FATO VISITA ADVOGADO
# ============================================================

df_fat_visita_advogado = spark.sql("""
    select
        id_fato_visita_advogado,
        id_evento_origem,
        id_vinculo_origem,
        id_advogado_origem,
        id_interno_origem,
        id_preso_presidiario,
        id_presidio_origem,
        id_equipe_origem,
        id_pessoa_advogado,
        id_pessoa_presidiario,
        documento_advogado,
        nome_advogado,
        documento_presidiario,
        nome_presidiario,
        origem_resolucao_presidiario,
        dt_hr_entrada,
        dt_hr_saida,
        hr_entrada,
        hr_saida,
        dt_registro,
        dt_evento_referencia,
        dt_evento,
        autorizacao,
        motivo,
        inativo,
        procuracao,
        dt_registro_vinculo,
        arquivo,
        flag_tem_entrada,
        flag_tem_saida,
        flag_duracao_valida,
        flag_vinculo_ativo,
        flag_procuracao,
        flag_autorizacao_preenchida,
        flag_motivo_preenchido,
        flag_restricao_mesmo_interno,
        flag_restricao_mesmo_presidio,
        flag_bloqueia_todos_internos,
        flag_bloqueia_todos_presidios,
        flag_visita_com_restricao,
        duracao_minutos
    from gold.tmp_evento_rank_visita_advogado
    where rn = 1
""")

tabela = "sinp_fat_visita_advogado"

df_fat_visita_advogado.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_fat_visita_advogado, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres(f"gold.{tabela}", "id_fato_visita_advogado")

In [ ]:
spark.sql("show tables in bronze like '*familiar*'").show(200, False)
spark.sql("show tables in bronze like '*famil*'").show(200, False)
spark.sql("show tables in bronze like '*vinc*'").show(200, False)

spark.sql("describe bronze.livros_acesso_unidade_visitafamiliar").show(200, False)
spark.sql("describe bronze.livros_acesso_unidade_controlefamiliares").show(200, False)
spark.sql("describe bronze.livros_acesso_unidade_interno").show(200, False)

In [ ]:
import os

# ============================================================
# LIMPEZA DEFENSIVA DAS TABELAS TEMPORARIAS - FATO VISITA FAMILIAR
# ============================================================

spark.sql("drop table if exists gold.tmp_base_controle_familiar")
spark.sql("drop table if exists gold.tmp_base_historicalvisitafamiliar_evento")
spark.sql("drop table if exists gold.tmp_base_pessoa_familiar")
spark.sql("drop table if exists gold.tmp_base_interno_livro_familiar")
spark.sql("drop table if exists gold.tmp_base_pessoa_preso_ponte_familiar")
spark.sql("drop table if exists gold.tmp_base_presidiario_resolvido_familiar")
spark.sql("drop table if exists gold.tmp_evento_base_visita_familiar")
spark.sql("drop table if exists gold.tmp_evento_enriquecido_visita_familiar")
spark.sql("drop table if exists gold.tmp_evento_calculado_visita_familiar")
spark.sql("drop table if exists gold.tmp_evento_rank_visita_familiar")
spark.sql("drop table if exists gold.sinp_fat_visita_familiar")

os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_base_controle_familiar >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_base_historicalvisitafamiliar_evento >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_base_pessoa_familiar >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_base_interno_livro_familiar >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_base_pessoa_preso_ponte_familiar >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_base_presidiario_resolvido_familiar >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_evento_base_visita_familiar >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_evento_enriquecido_visita_familiar >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_evento_calculado_visita_familiar >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_evento_rank_visita_familiar >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}sinp_fat_visita_familiar >/dev/null 2>&1")

spark.catalog.clearCache()
spark.sql("refresh table gold.df_pessoa_final_familiar")
spark.sql("refresh table gold.sinp_pnt_pessoa_preso")
spark.sql("refresh table gold.sinp_ent_pessoa")
spark.sql("refresh table bronze.livros_acesso_unidade_controlefamiliares")
spark.sql("refresh table bronze.livros_acesso_unidade_historicalvisitafamiliar")
spark.sql("refresh table bronze.livros_acesso_unidade_interno")


# ============================================================
# BASE CONTROLE FAMILIAR
# ============================================================

df_base_controle_familiar = spark.sql("""
    select
        cast(id as string) as id_evento_origem,
        cast(vinculo_id as string) as id_vinculo_origem,
        cast(vinculo_id as string) as id_interno_origem,
        cast(equipe_id as string) as id_equipe_origem,
        cast(presidio_id as string) as id_presidio_origem,
        to_timestamp(hr_entrada) as dt_hr_entrada,
        to_timestamp(hr_saida) as dt_hr_saida,
        to_timestamp(data_registro) as dt_registro,
        case
            when hr_entrada is not null then date_format(hr_entrada, 'HH:mm:ss')
            else null
        end as hr_entrada,
        case
            when hr_saida is not null then date_format(hr_saida, 'HH:mm:ss')
            else null
        end as hr_saida,
        trim(regexp_replace(coalesce(tipo, ''), '\\\\s+', ' ')) as tipo_visita
    from bronze.livros_acesso_unidade_controlefamiliares
""")

tabela = "tmp_base_controle_familiar"

df_base_controle_familiar.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_controle_familiar, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_controle_familiar")


# ============================================================
# BASE HISTORICAL VISITA FAMILIAR EVENTO
# history_id = id do controle
# ============================================================

df_base_historicalvisitafamiliar_evento = spark.sql("""
    select
        id_visitante_origem,
        id_evento_origem,
        nome_visitante,
        documento_original,
        telefone,
        history_date,
        history_type,
        history_user_id,
        chave_documento
    from (
        select
            cast(id as string) as id_visitante_origem,
            cast(history_id as string) as id_evento_origem,
            trim(regexp_replace(coalesce(nome, ''), '\\\\s+', ' ')) as nome_visitante,
            trim(regexp_replace(coalesce(documento, ''), '\\\\s+', ' ')) as documento_original,
            trim(regexp_replace(coalesce(telefone, ''), '\\\\s+', ' ')) as telefone,
            to_timestamp(history_date) as history_date,
            trim(regexp_replace(coalesce(history_type, ''), '\\\\s+', ' ')) as history_type,
            cast(history_user_id as string) as history_user_id,
            case
                when length(regexp_replace(coalesce(documento, ''), '[^0-9]', '')) = 11 then
                    concat(
                        'CPF_',
                        lpad(regexp_replace(coalesce(documento, ''), '[^0-9]', ''), 11, '0')
                    )
                else
                    concat(
                        'DOC_',
                        upper(regexp_replace(coalesce(documento, ''), '[^0-9A-Za-z]', ''))
                    )
            end as chave_documento,
            row_number() over (
                partition by cast(history_id as string)
                order by
                    case when history_date is not null then 1 else 2 end,
                    history_date desc,
                    id desc
            ) as rn
        from bronze.livros_acesso_unidade_historicalvisitafamiliar
    ) x
    where rn = 1
""")

tabela = "tmp_base_historicalvisitafamiliar_evento"

df_base_historicalvisitafamiliar_evento.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_historicalvisitafamiliar_evento, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_historicalvisitafamiliar_evento")


# ============================================================
# BASE PESSOA FAMILIAR
# ============================================================

df_base_pessoa_familiar = spark.sql("""
    select
        id_pessoa as id_pessoa_visitante,
        documento as documento_visitante,
        nome_pessoa as nome_visitante_pessoa,
        case
            when cod_documento_referencia = 19 then
                concat(
                    'CPF_',
                    lpad(regexp_replace(coalesce(documento, ''), '[^0-9]', ''), 11, '0')
                )
            else
                concat(
                    'DOC_',
                    upper(regexp_replace(coalesce(documento, ''), '[^0-9A-Za-z]', ''))
                )
        end as chave_documento
    from gold.df_pessoa_final_familiar
    where coalesce(flag_visitante, 0) = 1
""")

tabela = "tmp_base_pessoa_familiar"

df_base_pessoa_familiar.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_pessoa_familiar, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_pessoa_familiar")


# ============================================================
# BASE INTERNO LIVRO FAMILIAR
# controle.vinculo_id = interno.id
# ============================================================

df_base_interno_livro_familiar = spark.sql("""
    select
        cast(id as string) as id_interno_origem,
        cast(infopen as string) as id_preso_infopen,
        trim(regexp_replace(coalesce(nome, ''), '\\\\s+', ' ')) as nome_presidiario_livro,
        trim(regexp_replace(coalesce(galeria, ''), '\\\\s+', ' ')) as galeria_presidiario_livro,
        trim(regexp_replace(coalesce(cela, ''), '\\\\s+', ' ')) as cela_presidiario_livro,
        cast(status as string) as status_presidiario_livro,
        cast(presidio_id as string) as id_presidio_interno,
        cast(regime_id as string) as id_regime_interno,
        cast(grupo_visita_intima_id as string) as grupo_visita_intima_id,
        cast(grupo_visita_social_id as string) as grupo_visita_social_id,
        cast(grupo_saida_id as string) as grupo_saida_id,
        cast(situacao_id as string) as situacao_id,
        cast(n_uniforme as string) as n_uniforme
    from bronze.livros_acesso_unidade_interno
    where infopen is not null
""")

tabela = "tmp_base_interno_livro_familiar"

df_base_interno_livro_familiar.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_interno_livro_familiar, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_interno_livro_familiar")


# ============================================================
# BASE PONTE PESSOA/PRESO
# ============================================================

df_base_pessoa_preso_ponte_familiar = spark.sql("""
    select distinct
        trim(cast(id_preso as string)) as id_preso_infopen,
        trim(cast(id_preso as string)) as id_preso_presidiario,
        id_pessoa as id_pessoa_presidiario
    from gold.sinp_pnt_pessoa_preso
    where id_preso is not null
      and id_pessoa is not null
""")

tabela = "tmp_base_pessoa_preso_ponte_familiar"

df_base_pessoa_preso_ponte_familiar.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_pessoa_preso_ponte_familiar, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_pessoa_preso_ponte_familiar")


# ============================================================
# BASE PRESIDIARIO RESOLVIDO FAMILIAR
# MODELO IGUAL AO ADVOGADO:
# controle -> interno -> infopen -> ponte -> ent_pessoa
# ============================================================

df_base_presidiario_resolvido_familiar = spark.sql("""
    select
        c.id_evento_origem,
        c.id_vinculo_origem,
        c.id_interno_origem,
        trim(i.id_preso_infopen) as id_preso_infopen,
        trim(p.id_preso_presidiario) as id_preso_presidiario,
        p.id_pessoa_presidiario,
        e.documento as documento_presidiario,
        e.nome_pessoa as nome_presidiario_pessoa,
        i.nome_presidiario_livro,
        i.galeria_presidiario_livro,
        i.cela_presidiario_livro,
        i.status_presidiario_livro,
        i.id_presidio_interno,
        i.id_regime_interno,
        i.grupo_visita_intima_id,
        i.grupo_visita_social_id,
        i.grupo_saida_id,
        i.situacao_id,
        i.n_uniforme
    from gold.tmp_base_controle_familiar c
    inner join gold.tmp_base_interno_livro_familiar i
        on trim(c.id_interno_origem) = trim(i.id_interno_origem)
    inner join gold.tmp_base_pessoa_preso_ponte_familiar p
        on trim(i.id_preso_infopen) = trim(p.id_preso_infopen)
    left join gold.sinp_ent_pessoa e
        on p.id_pessoa_presidiario = e.id_pessoa
""")

tabela = "tmp_base_presidiario_resolvido_familiar"

df_base_presidiario_resolvido_familiar.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_presidiario_resolvido_familiar, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_presidiario_resolvido_familiar")


# ============================================================
# EVENTO BASE VISITA FAMILIAR
# controle.id = historicalvisitafamiliar.history_id
# ============================================================

df_evento_base_visita_familiar = spark.sql("""
    select
        c.id_evento_origem,
        c.id_vinculo_origem,
        c.id_interno_origem,
        c.id_equipe_origem,
        c.id_presidio_origem,
        c.dt_hr_entrada,
        c.dt_hr_saida,
        c.hr_entrada,
        c.hr_saida,
        c.dt_registro,
        c.tipo_visita,

        h.id_visitante_origem,
        h.nome_visitante,
        h.documento_original,
        h.telefone,
        h.history_date,
        h.history_type,
        h.history_user_id,
        h.chave_documento

    from gold.tmp_base_controle_familiar c
    inner join gold.tmp_base_historicalvisitafamiliar_evento h
        on trim(c.id_evento_origem) = trim(h.id_evento_origem)
""")

tabela = "tmp_evento_base_visita_familiar"

df_evento_base_visita_familiar.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_evento_base_visita_familiar, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_evento_base_visita_familiar")


# ============================================================
# EVENTO ENRIQUECIDO VISITA FAMILIAR
# ============================================================

df_evento_enriquecido_visita_familiar = spark.sql("""
    select
        e.id_evento_origem,
        e.id_vinculo_origem,
        e.id_interno_origem,
        e.id_equipe_origem,
        e.id_presidio_origem,

        e.id_visitante_origem,
        p.id_pessoa_visitante,

        pr.id_pessoa_presidiario,
        pr.id_preso_presidiario,
        pr.id_preso_infopen,

        coalesce(p.nome_visitante_pessoa, e.nome_visitante) as nome_visitante,
        coalesce(p.documento_visitante, e.documento_original) as documento_visitante,
        e.telefone,

        e.dt_hr_entrada,
        e.dt_hr_saida,
        e.hr_entrada,
        e.hr_saida,
        e.dt_registro,
        coalesce(e.dt_hr_entrada, e.dt_hr_saida, e.dt_registro, e.history_date) as dt_evento_referencia,
        e.tipo_visita,

        case
            when p.id_pessoa_visitante is not null then 'VISITANTE_RESOLVIDO_POR_DOCUMENTO'
            else 'VISITANTE_SEM_RESOLUCAO_POR_DOCUMENTO'
        end as origem_resolucao_visitante,

        case
            when pr.id_pessoa_presidiario is not null then 'CONTROLE_INTERNO_INFOPEN_PESSOA'
            when pr.id_preso_infopen is not null then 'CONTROLE_INTERNO_INFOPEN_SEM_PESSOA'
            else 'SEM_RESOLUCAO'
        end as origem_resolucao_presidiario,

        e.nome_visitante as nome_visitante_historico,
        e.documento_original as documento_visitante_historico,
        e.history_date,
        e.history_type,
        e.history_user_id,

        pr.nome_presidiario_livro,
        pr.nome_presidiario_pessoa,
        pr.documento_presidiario,
        pr.galeria_presidiario_livro,
        pr.cela_presidiario_livro,
        pr.status_presidiario_livro,
        pr.id_presidio_interno,
        pr.id_regime_interno,
        pr.grupo_visita_intima_id,
        pr.grupo_visita_social_id,
        pr.grupo_saida_id,
        pr.situacao_id,
        pr.n_uniforme

    from gold.tmp_evento_base_visita_familiar e
    left join gold.tmp_base_pessoa_familiar p
        on e.chave_documento = p.chave_documento
    left join gold.tmp_base_presidiario_resolvido_familiar pr
        on trim(e.id_evento_origem) = trim(pr.id_evento_origem)
""")

tabela = "tmp_evento_enriquecido_visita_familiar"

df_evento_enriquecido_visita_familiar.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_evento_enriquecido_visita_familiar, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_evento_enriquecido_visita_familiar")


# ============================================================
# EVENTO CALCULADO VISITA FAMILIAR
# ============================================================

df_evento_calculado_visita_familiar = spark.sql("""
    select
        concat(
            'FVF_',
            md5(
                concat_ws(
                    '|',
                    coalesce(id_evento_origem, ''),
                    coalesce(id_vinculo_origem, ''),
                    coalesce(id_visitante_origem, ''),
                    coalesce(id_pessoa_visitante, ''),
                    coalesce(cast(dt_evento_referencia as string), '')
                )
            )
        ) as id_fato_visita_familiar,

        -- CAMPOS JA EXISTENTES
        id_evento_origem,
        id_vinculo_origem,
        id_equipe_origem,
        id_presidio_origem,
        id_pessoa_visitante,
        id_pessoa_presidiario,
        id_preso_presidiario,
        nome_visitante,
        documento_visitante,
        telefone,
        dt_hr_entrada,
        dt_hr_saida,
        hr_entrada,
        hr_saida,
        dt_registro,
        dt_evento_referencia,
        to_date(dt_evento_referencia) as dt_evento,
        tipo_visita,
        origem_resolucao_presidiario,
        case when dt_hr_entrada is not null then 1 else 0 end as flag_tem_entrada,
        case when dt_hr_saida is not null then 1 else 0 end as flag_tem_saida,
        case
            when dt_hr_entrada is not null and dt_hr_saida is not null and dt_hr_saida >= dt_hr_entrada then 1
            else 0
        end as flag_duracao_valida,
        case
            when dt_hr_entrada is not null and dt_hr_saida is not null and dt_hr_saida >= dt_hr_entrada
            then cast((unix_timestamp(dt_hr_saida) - unix_timestamp(dt_hr_entrada)) / 60.0 as decimal(18,2))
            else cast(null as decimal(18,2))
        end as duracao_minutos,

        -- NOVOS CAMPOS
        id_visitante_origem,
        id_interno_origem,
        id_preso_infopen,
        origem_resolucao_visitante,

        nome_visitante_historico,
        documento_visitante_historico,
        history_date,
        history_type,
        history_user_id,

        nome_presidiario_livro,
        nome_presidiario_pessoa,
        documento_presidiario,
        galeria_presidiario_livro,
        cela_presidiario_livro,
        status_presidiario_livro,
        id_presidio_interno,
        id_regime_interno,
        grupo_visita_intima_id,
        grupo_visita_social_id,
        grupo_saida_id,
        situacao_id,
        n_uniforme,

        case when id_pessoa_visitante is not null then 1 else 0 end as flag_visitante_resolvido,
        case when id_pessoa_presidiario is not null then 1 else 0 end as flag_presidiario_resolvido
    from gold.tmp_evento_enriquecido_visita_familiar
""")

tabela = "tmp_evento_calculado_visita_familiar"

df_evento_calculado_visita_familiar.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_evento_calculado_visita_familiar, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_evento_calculado_visita_familiar")


# ============================================================
# EVENTO RANKEADO VISITA FAMILIAR
# ============================================================

df_evento_rank_visita_familiar = spark.sql("""
    select
        *,
        row_number() over (
            partition by coalesce(id_evento_origem, id_vinculo_origem, id_visitante_origem)
            order by
                case when id_pessoa_visitante is not null then 1 else 2 end,
                case when id_pessoa_presidiario is not null then 1 else 2 end,
                case when dt_evento_referencia is not null then 1 else 2 end,
                dt_evento_referencia desc,
                history_date desc,
                id_visitante_origem desc
        ) as rn
    from gold.tmp_evento_calculado_visita_familiar
""")

tabela = "tmp_evento_rank_visita_familiar"

df_evento_rank_visita_familiar.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_evento_rank_visita_familiar, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_evento_rank_visita_familiar")


# ============================================================
# FATO VISITA FAMILIAR
# ============================================================

df_fat_visita_familiar = spark.sql("""
    select
        -- CAMPOS JA EXISTENTES
        id_fato_visita_familiar,
        id_evento_origem,
        id_vinculo_origem,
        id_equipe_origem,
        id_presidio_origem,
        id_pessoa_visitante,
        id_pessoa_presidiario,
        id_preso_presidiario,
        nome_visitante,
        documento_visitante,
        telefone,
        dt_hr_entrada,
        dt_hr_saida,
        hr_entrada,
        hr_saida,
        dt_registro,
        dt_evento_referencia,
        dt_evento,
        tipo_visita,
        origem_resolucao_presidiario,
        flag_tem_entrada,
        flag_tem_saida,
        flag_duracao_valida,
        duracao_minutos,

        -- NOVOS CAMPOS
        id_visitante_origem,
        id_interno_origem,
        id_preso_infopen,
        origem_resolucao_visitante,

        nome_visitante_historico,
        documento_visitante_historico,
        history_date,
        history_type,
        history_user_id,

        nome_presidiario_livro,
        nome_presidiario_pessoa,
        documento_presidiario,
        galeria_presidiario_livro,
        cela_presidiario_livro,
        status_presidiario_livro,
        id_presidio_interno,
        id_regime_interno,
        grupo_visita_intima_id,
        grupo_visita_social_id,
        grupo_saida_id,
        situacao_id,
        n_uniforme,

        flag_visitante_resolvido,
        flag_presidiario_resolvido
    from gold.tmp_evento_rank_visita_familiar
    where rn = 1
""")

tabela = "sinp_fat_visita_familiar"

df_fat_visita_familiar.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_fat_visita_familiar, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres(f"gold.{tabela}", "id_fato_visita_familiar")


# ============================================================
# VALIDACAO RAPIDA
# ============================================================

spark.sql("""
select
    count(*) as total_eventos,
    sum(case when id_pessoa_visitante is not null then 1 else 0 end) as eventos_com_visitante_resolvido,
    sum(case when id_preso_presidiario is not null then 1 else 0 end) as eventos_com_id_preso_presidiario,
    sum(case when id_pessoa_presidiario is not null then 1 else 0 end) as eventos_com_id_pessoa_presidiario
from gold.sinp_fat_visita_familiar
""").show(truncate=False)

spark.sql("""
select
    origem_resolucao_presidiario,
    count(*) as qtd
from gold.sinp_fat_visita_familiar
group by origem_resolucao_presidiario
order by qtd desc
""").show(truncate=False)

spark.sql("""
select
    id_fato_visita_familiar,
    id_evento_origem,
    id_vinculo_origem,
    id_interno_origem,
    id_preso_infopen,
    id_preso_presidiario,
    id_pessoa_presidiario,
    nome_visitante,
    nome_presidiario_pessoa,
    tipo_visita,
    dt_evento_referencia
from gold.sinp_fat_visita_familiar
order by dt_evento_referencia desc
""").show(100, False)

In [ ]:
import os

# ============================================================
# DEFINE BASE DE PESSOAS
# ============================================================

tbl_religiosa = spark.sql("show tables in gold like 'df_pessoa_final_religiosa'").count()
tbl_familiar = spark.sql("show tables in gold like 'df_pessoa_final_familiar'").count()

if tbl_religiosa > 0:
    tabela_base_pessoas = "gold.df_pessoa_final_religiosa"
elif tbl_familiar > 0:
    tabela_base_pessoas = "gold.df_pessoa_final_familiar"
else:
    raise Exception("Nenhuma base de pessoas encontrada: nem gold.df_pessoa_final_religiosa nem gold.df_pessoa_final_familiar.")

print(f"Base de pessoas utilizada: {tabela_base_pessoas}")

# ============================================================
# LIMPEZA DEFENSIVA DAS TABELAS TEMPORARIAS - FATO VISITA RELIGIOSA
# ============================================================

spark.sql("drop table if exists gold.tmp_base_controle_visitareligiosa")
spark.sql("drop table if exists gold.tmp_base_visitareligiosa_evento")
spark.sql("drop table if exists gold.tmp_base_pessoa_visitareligiosa")
spark.sql("drop table if exists gold.tmp_base_restricao_visitareligiosa")
spark.sql("drop table if exists gold.tmp_base_restricao_visitareligiosa_presidio")
spark.sql("drop table if exists gold.tmp_base_restricao_visitareligiosa_global")
spark.sql("drop table if exists gold.tmp_evento_base_visita_religiosa")
spark.sql("drop table if exists gold.tmp_evento_enriquecido_visita_religiosa")
spark.sql("drop table if exists gold.tmp_evento_calculado_visita_religiosa")
spark.sql("drop table if exists gold.tmp_evento_rank_visita_religiosa")
spark.sql("drop table if exists gold.sinp_fat_visita_religiosa")

os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_base_controle_visitareligiosa >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_base_visitareligiosa_evento >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_base_pessoa_visitareligiosa >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_base_restricao_visitareligiosa >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_base_restricao_visitareligiosa_presidio >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_base_restricao_visitareligiosa_global >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_evento_base_visita_religiosa >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_evento_enriquecido_visita_religiosa >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_evento_calculado_visita_religiosa >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}tmp_evento_rank_visita_religiosa >/dev/null 2>&1")
os.system(f"hdfs dfs -rm -r -skipTrash {path}sinp_fat_visita_religiosa >/dev/null 2>&1")

spark.catalog.clearCache()
spark.sql(f"refresh table {tabela_base_pessoas}")


# ============================================================
# BASE CONTROLE VISITA RELIGIOSA
# ============================================================

df_base_controle_visitareligiosa = spark.sql("""
    select
        cast(id as string) as id_evento_origem,
        cast(nome_id as string) as id_visitante_religioso_origem,
        cast(equipe_id as string) as id_equipe_origem,
        cast(presidio_id as string) as id_presidio_origem,
        to_timestamp(hr_entrada) as dt_hr_entrada,
        to_timestamp(hr_saida) as dt_hr_saida,
        to_timestamp(data_registro) as dt_registro,
        case
            when hr_entrada is not null then date_format(hr_entrada, 'HH:mm:ss')
            else null
        end as hr_entrada,
        case
            when hr_saida is not null then date_format(hr_saida, 'HH:mm:ss')
            else null
        end as hr_saida
    from bronze.livros_acesso_unidade_controlevisitareligiosa
""")

tabela = "tmp_base_controle_visitareligiosa"

df_base_controle_visitareligiosa.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_controle_visitareligiosa, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_controle_visitareligiosa")


# ============================================================
# BASE VISITA RELIGIOSA EVENTO
# ============================================================

df_base_visitareligiosa_evento = spark.sql("""
    select
        cast(id as string) as id_visitante_religioso_origem,
        trim(regexp_replace(coalesce(nome, ''), '\\\\s+', ' ')) as nome_visitante,
        trim(regexp_replace(coalesce(instituicao, ''), '\\\\s+', ' ')) as instituicao,
        trim(regexp_replace(coalesce(documento, ''), '\\\\s+', ' ')) as documento_original,
        cast(presidio_id as string) as presidio_id_cadastro,
        regexp_replace(
            regexp_replace(cast(documento as string), '\\\\.0+$', ''),
            '[^0-9]',
            ''
        ) as documento_digitos,
        case
            when length(
                regexp_replace(
                    regexp_replace(cast(documento as string), '\\\\.0+$', ''),
                    '[^0-9]',
                    ''
                )
            ) between 1 and 11 then
                concat(
                    'CPF_',
                    lpad(
                        regexp_replace(
                            regexp_replace(cast(documento as string), '\\\\.0+$', ''),
                            '[^0-9]',
                            ''
                        ),
                        11,
                        '0'
                    )
                )
            else
                concat(
                    'DOC_',
                    upper(regexp_replace(coalesce(documento, ''), '[^0-9A-Za-z]', ''))
                )
        end as chave_documento
    from bronze.livros_acesso_unidade_visitareligiosa
""")

tabela = "tmp_base_visitareligiosa_evento"

df_base_visitareligiosa_evento.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_visitareligiosa_evento, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_visitareligiosa_evento")


# ============================================================
# BASE PESSOA VISITA RELIGIOSA
# ============================================================

df_base_pessoa_visitareligiosa = spark.sql(f"""
    select
        id_pessoa as id_pessoa_visitante,
        documento as documento_visitante,
        nome_pessoa as nome_visitante_pessoa,
        case
            when regexp_replace(coalesce(documento, ''), '[^0-9]', '') <> ''
             and length(regexp_replace(coalesce(documento, ''), '[^0-9]', '')) between 1 and 11 then
                concat(
                    'CPF_',
                    lpad(regexp_replace(coalesce(documento, ''), '[^0-9]', ''), 11, '0')
                )
            else
                concat(
                    'DOC_',
                    upper(regexp_replace(coalesce(documento, ''), '[^0-9A-Za-z]', ''))
                )
        end as chave_documento
    from {tabela_base_pessoas}
    where coalesce(flag_visitante, 0) = 1
""")

tabela = "tmp_base_pessoa_visitareligiosa"

df_base_pessoa_visitareligiosa.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_pessoa_visitareligiosa, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_pessoa_visitareligiosa")


# ============================================================
# BASE RESTRICAO VISITA RELIGIOSA
# ============================================================

df_base_restricao_visitareligiosa = spark.sql("""
    select
        cast(visitante_id as string) as id_visitante_religioso_origem,
        cast(presidio_id as string) as id_presidio_origem,
        max(case when coalesce(bloquear_todos_presidios, false) = true then 1 else 0 end) as flag_bloquear_todos_presidios,
        count(*) as qtd_restricoes
    from bronze.livros_acesso_unidade_restricaovisitareligiosa
    group by
        cast(visitante_id as string),
        cast(presidio_id as string)
""")

tabela = "tmp_base_restricao_visitareligiosa"

df_base_restricao_visitareligiosa.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_restricao_visitareligiosa, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_restricao_visitareligiosa")


# ============================================================
# RESTRICAO POR VISITANTE + PRESIDIO
# ============================================================

df_base_restricao_visitareligiosa_presidio = spark.sql("""
    select
        trim(id_visitante_religioso_origem) as id_visitante_religioso_origem,
        trim(id_presidio_origem) as id_presidio_origem,
        max(flag_bloquear_todos_presidios) as flag_bloquear_todos_presidios,
        sum(qtd_restricoes) as qtd_restricoes_presidio
    from gold.tmp_base_restricao_visitareligiosa
    group by
        trim(id_visitante_religioso_origem),
        trim(id_presidio_origem)
""")

tabela = "tmp_base_restricao_visitareligiosa_presidio"

df_base_restricao_visitareligiosa_presidio.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_restricao_visitareligiosa_presidio, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_restricao_visitareligiosa_presidio")


# ============================================================
# RESTRICAO GLOBAL POR VISITANTE
# ============================================================

df_base_restricao_visitareligiosa_global = spark.sql("""
    select
        trim(id_visitante_religioso_origem) as id_visitante_religioso_origem,
        max(flag_bloquear_todos_presidios) as flag_bloquear_todos_presidios_global,
        sum(qtd_restricoes) as qtd_restricoes_global
    from gold.tmp_base_restricao_visitareligiosa
    group by trim(id_visitante_religioso_origem)
""")

tabela = "tmp_base_restricao_visitareligiosa_global"

df_base_restricao_visitareligiosa_global.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_restricao_visitareligiosa_global, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_restricao_visitareligiosa_global")


# ============================================================
# EVENTO BASE VISITA RELIGIOSA
# ============================================================

df_evento_base_visita_religiosa = spark.sql("""
    select
        c.id_evento_origem,
        c.id_visitante_religioso_origem,
        c.id_equipe_origem,
        c.id_presidio_origem,
        c.dt_hr_entrada,
        c.dt_hr_saida,
        c.hr_entrada,
        c.hr_saida,
        c.dt_registro,
        v.nome_visitante,
        v.instituicao,
        v.documento_original,
        v.presidio_id_cadastro,
        v.chave_documento
    from gold.tmp_base_controle_visitareligiosa c
    inner join gold.tmp_base_visitareligiosa_evento v
        on trim(c.id_visitante_religioso_origem) = trim(v.id_visitante_religioso_origem)
""")

tabela = "tmp_evento_base_visita_religiosa"

df_evento_base_visita_religiosa.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_evento_base_visita_religiosa, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_evento_base_visita_religiosa")


# ============================================================
# EVENTO ENRIQUECIDO VISITA RELIGIOSA
# ============================================================

df_evento_enriquecido_visita_religiosa = spark.sql("""
    select
        e.id_evento_origem,
        e.id_visitante_religioso_origem,
        e.id_equipe_origem,
        e.id_presidio_origem,
        p.id_pessoa_visitante,
        coalesce(p.nome_visitante_pessoa, e.nome_visitante) as nome_visitante,
        coalesce(p.documento_visitante, e.documento_original) as documento_visitante,
        e.instituicao,
        e.dt_hr_entrada,
        e.dt_hr_saida,
        e.hr_entrada,
        e.hr_saida,
        e.dt_registro,
        coalesce(e.dt_hr_entrada, e.dt_hr_saida, e.dt_registro) as dt_evento_referencia,
        coalesce(rp.qtd_restricoes_presidio, 0) as qtd_restricoes_presidio,
        coalesce(rg.qtd_restricoes_global, 0) as qtd_restricoes_global,
        coalesce(rp.flag_bloquear_todos_presidios, 0) as flag_bloquear_todos_presidios_presidio,
        coalesce(rg.flag_bloquear_todos_presidios_global, 0) as flag_bloquear_todos_presidios_global
    from gold.tmp_evento_base_visita_religiosa e
    left join gold.tmp_base_pessoa_visitareligiosa p
        on e.chave_documento = p.chave_documento
    left join gold.tmp_base_restricao_visitareligiosa_presidio rp
        on trim(e.id_visitante_religioso_origem) = trim(rp.id_visitante_religioso_origem)
       and trim(e.id_presidio_origem) = trim(rp.id_presidio_origem)
    left join gold.tmp_base_restricao_visitareligiosa_global rg
        on trim(e.id_visitante_religioso_origem) = trim(rg.id_visitante_religioso_origem)
""")

tabela = "tmp_evento_enriquecido_visita_religiosa"

df_evento_enriquecido_visita_religiosa.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_evento_enriquecido_visita_religiosa, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_evento_enriquecido_visita_religiosa")


# ============================================================
# EVENTO CALCULADO VISITA RELIGIOSA
# ============================================================

df_evento_calculado_visita_religiosa = spark.sql("""
    select
        concat(
            'FVR_',
            md5(
                concat_ws(
                    '|',
                    coalesce(id_evento_origem, ''),
                    coalesce(id_visitante_religioso_origem, ''),
                    coalesce(id_pessoa_visitante, ''),
                    coalesce(cast(dt_evento_referencia as string), '')
                )
            )
        ) as id_fato_visita_religiosa,
        id_evento_origem,
        id_visitante_religioso_origem,
        id_equipe_origem,
        id_presidio_origem,
        id_pessoa_visitante,
        nome_visitante,
        documento_visitante,
        instituicao,
        dt_hr_entrada,
        dt_hr_saida,
        hr_entrada,
        hr_saida,
        dt_registro,
        dt_evento_referencia,
        to_date(dt_evento_referencia) as dt_evento,
        case when dt_hr_entrada is not null then 1 else 0 end as flag_tem_entrada,
        case when dt_hr_saida is not null then 1 else 0 end as flag_tem_saida,
        case
            when dt_hr_entrada is not null and dt_hr_saida is not null and dt_hr_saida >= dt_hr_entrada then 1
            else 0
        end as flag_duracao_valida,
        case
            when dt_hr_entrada is not null and dt_hr_saida is not null and dt_hr_saida >= dt_hr_entrada
            then cast((unix_timestamp(dt_hr_saida) - unix_timestamp(dt_hr_entrada)) / 60.0 as decimal(18,2))
            else cast(null as decimal(18,2))
        end as duracao_minutos,
        case
            when coalesce(qtd_restricoes_presidio, 0) > 0 then 1
            else 0
        end as flag_restricao_mesmo_presidio,
        greatest(
            coalesce(flag_bloquear_todos_presidios_presidio, 0),
            coalesce(flag_bloquear_todos_presidios_global, 0)
        ) as flag_bloquear_todos_presidios,
        case
            when coalesce(qtd_restricoes_presidio, 0) > 0
              or coalesce(qtd_restricoes_global, 0) > 0
              or coalesce(flag_bloquear_todos_presidios_presidio, 0) = 1
              or coalesce(flag_bloquear_todos_presidios_global, 0) = 1
            then 1
            else 0
        end as flag_visita_com_restricao
    from gold.tmp_evento_enriquecido_visita_religiosa
""")

tabela = "tmp_evento_calculado_visita_religiosa"

df_evento_calculado_visita_religiosa.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_evento_calculado_visita_religiosa, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_evento_calculado_visita_religiosa")


# ============================================================
# EVENTO RANKEADO VISITA RELIGIOSA
# ============================================================

df_evento_rank_visita_religiosa = spark.sql("""
    select
        *,
        row_number() over (
            partition by coalesce(id_evento_origem, id_visitante_religioso_origem)
            order by
                case when id_pessoa_visitante is not null then 1 else 2 end,
                case when dt_evento_referencia is not null then 1 else 2 end,
                dt_evento_referencia desc,
                id_visitante_religioso_origem desc
        ) as rn
    from gold.tmp_evento_calculado_visita_religiosa
""")

tabela = "tmp_evento_rank_visita_religiosa"

df_evento_rank_visita_religiosa.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_evento_rank_visita_religiosa, "gold", tabela, f"{path}{tabela}")

spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_evento_rank_visita_religiosa")


# ============================================================
# FATO VISITA RELIGIOSA
# ============================================================

df_fat_visita_religiosa = spark.sql("""
    select
        id_fato_visita_religiosa,
        id_evento_origem,
        id_visitante_religioso_origem,
        id_equipe_origem,
        id_presidio_origem,
        id_pessoa_visitante,
        nome_visitante,
        documento_visitante,
        instituicao,
        dt_hr_entrada,
        dt_hr_saida,
        hr_entrada,
        hr_saida,
        dt_registro,
        dt_evento_referencia,
        dt_evento,
        flag_tem_entrada,
        flag_tem_saida,
        flag_duracao_valida,
        duracao_minutos,
        flag_restricao_mesmo_presidio,
        flag_bloquear_todos_presidios,
        flag_visita_com_restricao
    from gold.tmp_evento_rank_visita_religiosa
    where rn = 1
""")

tabela = "sinp_fat_visita_religiosa"

df_fat_visita_religiosa.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_fat_visita_religiosa, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres(f"gold.{tabela}", "id_fato_visita_religiosa")

In [ ]:
import os

# ============================================================
# LIMPEZA DEFENSIVA
# ============================================================

tabelas_drop = [
    "tmp_base_infopen_ocorrencias",
    "tmp_base_infopen_tipos_ocorrencia",
    "tmp_base_infopen_status_ocorrencia_raw",
    "tmp_base_infopen_status_ocorrencia_rank",
    "tmp_base_infopen_status_ocorrencia_ultimo",
    "tmp_base_infopen_status_ocorrencia_agg",
    "tmp_base_infopen_observacoes_raw",
    "tmp_base_infopen_observacoes_agg",
    "tmp_base_infopen_ocorrencia_processo_raw",
    "tmp_base_infopen_ocorrencia_processo_agg",
    "tmp_base_pessoa_preso_ponte_ocorrencia",
    "tmp_base_presidiario_catalogo_ocorrencia",
    "tmp_base_infopen_ocorrencia_preso_raw",
    "tmp_base_infopen_ocorrencia_preso_agg",
    "tmp_fat_ocorrencia_infopen",
    "sinp_fat_ocorrencia_infopen",
    "sinp_rl_ocorrencia_preso_infopen",
    "sinp_rl_ocorrencia_processo_infopen",
    "sinp_rl_ocorrencia_observacao_infopen",
    "sinp_rl_ocorrencia_status_infopen"
]

for t in tabelas_drop:
    spark.sql(f"drop table if exists gold.{t}")
    os.system(f"hdfs dfs -rm -r -skipTrash {path}{t} >/dev/null 2>&1")

spark.catalog.clearCache()
spark.sql("refresh table gold.sinp_pnt_pessoa_preso")
spark.sql("refresh table gold.sinp_ent_pessoa")


# ============================================================
# BASE INFOPEN OCORRENCIAS
# ============================================================

df_base_infopen_ocorrencias = spark.sql("""
    select
        cast(id_ocorrencia as string) as id_ocorrencia_origem,
        cast(id_tipoocorrencia as string) as id_tipo_ocorrencia,
        cast(id_municipio as string) as id_municipio,
        cast(id_status as string) as id_status_atual,
        cast(ocorrencia_numprontuario as string) as nr_prontuario,
        cast(ocorrencia_numero as string) as nr_ocorrencia,
        to_timestamp(ocorrencia_dtfato) as dt_fato,
        to_timestamp(ocorrencia_dtexpedicao) as dt_expedicao
    from bronze.infopen_ocorrencias
""")

tabela = "tmp_base_infopen_ocorrencias"

df_base_infopen_ocorrencias.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_infopen_ocorrencias, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_infopen_ocorrencias")


# ============================================================
# BASE INFOPEN TIPOS OCORRENCIA
# ============================================================

df_base_infopen_tipos_ocorrencia = spark.sql("""
    select
        cast(id_tipoocorrencia as string) as id_tipo_ocorrencia,
        trim(regexp_replace(coalesce(tipoocorrencia_descricao, ''), '\\\\s+', ' ')) as ds_tipo_ocorrencia
    from bronze.infopen_tipos_ocorrencia
""")

tabela = "tmp_base_infopen_tipos_ocorrencia"

df_base_infopen_tipos_ocorrencia.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_infopen_tipos_ocorrencia, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_infopen_tipos_ocorrencia")


# ============================================================
# RL STATUS INFOPEN RAW
# ============================================================

df_base_infopen_status_ocorrencia_raw = spark.sql("""
    select
        concat(
            'RLSTATUS_',
            md5(
                concat_ws(
                    '|',
                    cast(id_ocorrencia as string),
                    cast(id_status as string),
                    cast(statusocorrencia_data as string),
                    cast(coalesce(statusocorrencia_ultima, false) as string)
                )
            )
        ) as id_rl_ocorrencia_status,
        concat('OCR_INFOPEN_', cast(id_ocorrencia as string)) as id_fato_ocorrencia,
        cast(id_ocorrencia as string) as id_ocorrencia_origem,
        cast(id_status as string) as id_status_historico,
        to_timestamp(statusocorrencia_data) as dt_status_historico,
        case when coalesce(statusocorrencia_ultima, false) = true then 1 else 0 end as flag_status_marcado_ultimo
    from bronze.infopen_status_ocorrencias
""")

tabela = "tmp_base_infopen_status_ocorrencia_raw"

df_base_infopen_status_ocorrencia_raw.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_infopen_status_ocorrencia_raw, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_infopen_status_ocorrencia_raw")


# ============================================================
# STATUS INFOPEN RANKEADO
# ============================================================

df_base_infopen_status_ocorrencia_rank = spark.sql("""
    select
        *,
        row_number() over (
            partition by id_ocorrencia_origem
            order by
                case when flag_status_marcado_ultimo = 1 then 1 else 2 end,
                dt_status_historico desc,
                id_status_historico desc
        ) as rn
    from gold.tmp_base_infopen_status_ocorrencia_raw
""")

tabela = "tmp_base_infopen_status_ocorrencia_rank"

df_base_infopen_status_ocorrencia_rank.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_infopen_status_ocorrencia_rank, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_infopen_status_ocorrencia_rank")


# ============================================================
# STATUS INFOPEN ULTIMO
# ============================================================

df_base_infopen_status_ocorrencia_ultimo = spark.sql("""
    select
        id_ocorrencia_origem,
        id_status_historico as id_status_ultimo,
        dt_status_historico as dt_status_ultimo,
        flag_status_marcado_ultimo
    from gold.tmp_base_infopen_status_ocorrencia_rank
    where rn = 1
""")

tabela = "tmp_base_infopen_status_ocorrencia_ultimo"

df_base_infopen_status_ocorrencia_ultimo.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_infopen_status_ocorrencia_ultimo, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_infopen_status_ocorrencia_ultimo")


# ============================================================
# STATUS INFOPEN AGREGADO
# ============================================================

df_base_infopen_status_ocorrencia_agg = spark.sql("""
    select
        id_ocorrencia_origem,
        count(*) as qtd_status_historico,
        count(distinct id_status_historico) as qtd_status_distintos,
        min(dt_status_historico) as dt_primeiro_status_historico,
        max(dt_status_historico) as dt_ultimo_status_historico,
        max(flag_status_marcado_ultimo) as flag_possui_status_marcado_ultimo,
        concat_ws(
            ' | ',
            sort_array(collect_set(id_status_historico))
        ) as txt_ids_status_historico
    from gold.tmp_base_infopen_status_ocorrencia_raw
    group by id_ocorrencia_origem
""")

tabela = "tmp_base_infopen_status_ocorrencia_agg"

df_base_infopen_status_ocorrencia_agg.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_infopen_status_ocorrencia_agg, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_infopen_status_ocorrencia_agg")


# ============================================================
# RL OBSERVACAO INFOPEN RAW
# ============================================================

df_base_infopen_observacoes_raw = spark.sql("""
    select
        concat(
            'RLOBS_',
            md5(
                concat_ws(
                    '|',
                    cast(id_ocorrencia as string),
                    trim(regexp_replace(coalesce(ocorrenciaobservacao_descricao, ''), '\\\\s+', ' '))
                )
            )
        ) as id_rl_ocorrencia_observacao,
        concat('OCR_INFOPEN_', cast(id_ocorrencia as string)) as id_fato_ocorrencia,
        cast(id_ocorrencia as string) as id_ocorrencia_origem,
        trim(regexp_replace(coalesce(ocorrenciaobservacao_descricao, ''), '\\\\s+', ' ')) as ds_observacao
    from bronze.infopen_ocorrencias_observacoes
""")

tabela = "tmp_base_infopen_observacoes_raw"

df_base_infopen_observacoes_raw.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_infopen_observacoes_raw, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_infopen_observacoes_raw")


# ============================================================
# OBSERVACOES INFOPEN AGREGADAS
# ============================================================

df_base_infopen_observacoes_agg = spark.sql("""
    select
        id_ocorrencia_origem,
        count(*) as qtd_observacoes_infopen,
        concat_ws(
            ' | ',
            sort_array(collect_set(ds_observacao))
        ) as txt_observacoes_infopen
    from gold.tmp_base_infopen_observacoes_raw
    group by id_ocorrencia_origem
""")

tabela = "tmp_base_infopen_observacoes_agg"

df_base_infopen_observacoes_agg.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_infopen_observacoes_agg, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_infopen_observacoes_agg")


# ============================================================
# RL OCORRENCIA X PROCESSO INFOPEN RAW
# ============================================================

df_base_infopen_ocorrencia_processo_raw = spark.sql("""
    select
        concat(
            'RLPROC_',
            md5(
                concat_ws(
                    '|',
                    cast(id_ocorrencia as string),
                    cast(id_processo as string)
                )
            )
        ) as id_rl_ocorrencia_processo,
        concat('OCR_INFOPEN_', cast(id_ocorrencia as string)) as id_fato_ocorrencia,
        cast(id_ocorrencia as string) as id_ocorrencia_origem,
        cast(id_processo as string) as id_processo_origem
    from bronze.infopen_ocorrencias_processos
""")

tabela = "tmp_base_infopen_ocorrencia_processo_raw"

df_base_infopen_ocorrencia_processo_raw.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_infopen_ocorrencia_processo_raw, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_infopen_ocorrencia_processo_raw")


# ============================================================
# PROCESSOS INFOPEN AGREGADOS
# ============================================================

df_base_infopen_ocorrencia_processo_agg = spark.sql("""
    select
        id_ocorrencia_origem,
        count(*) as qtd_processos_infopen,
        concat_ws(
            ' | ',
            sort_array(collect_set(id_processo_origem))
        ) as txt_ids_processo_infopen
    from gold.tmp_base_infopen_ocorrencia_processo_raw
    group by id_ocorrencia_origem
""")

tabela = "tmp_base_infopen_ocorrencia_processo_agg"

df_base_infopen_ocorrencia_processo_agg.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_infopen_ocorrencia_processo_agg, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_infopen_ocorrencia_processo_agg")


# ============================================================
# BASE PONTE PESSOA X PRESO
# ============================================================

df_base_pessoa_preso_ponte_ocorrencia = spark.sql("""
    select distinct
        cast(id_preso as string) as id_preso_origem,
        id_pessoa as id_pessoa_presidiario
    from gold.sinp_pnt_pessoa_preso
    where id_preso is not null
      and id_pessoa is not null
""")

tabela = "tmp_base_pessoa_preso_ponte_ocorrencia"

df_base_pessoa_preso_ponte_ocorrencia.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_pessoa_preso_ponte_ocorrencia, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_pessoa_preso_ponte_ocorrencia")


# ============================================================
# CATALOGO PRESIDIARIO ENRIQUECIDO
# ============================================================

df_base_presidiario_catalogo_ocorrencia = spark.sql("""
    select
        p.id_preso_origem,
        p.id_pessoa_presidiario,
        e.nome_pessoa as nome_presidiario,
        e.documento as documento_presidiario
    from gold.tmp_base_pessoa_preso_ponte_ocorrencia p
    left join gold.sinp_ent_pessoa e
        on p.id_pessoa_presidiario = e.id_pessoa
""")

tabela = "tmp_base_presidiario_catalogo_ocorrencia"

df_base_presidiario_catalogo_ocorrencia.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_presidiario_catalogo_ocorrencia, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_presidiario_catalogo_ocorrencia")


# ============================================================
# RL OCORRENCIA X PRESO INFOPEN RAW
# ============================================================

df_base_infopen_ocorrencia_preso_raw = spark.sql("""
    select
        concat(
            'RLPRESO_',
            md5(
                concat_ws(
                    '|',
                    cast(po.id_ocorrencia as string),
                    cast(po.id_preso as string)
                )
            )
        ) as id_rl_ocorrencia_preso,
        concat('OCR_INFOPEN_', cast(po.id_ocorrencia as string)) as id_fato_ocorrencia,
        cast(po.id_ocorrencia as string) as id_ocorrencia_origem,
        cast(po.id_preso as string) as id_preso_origem,
        pc.id_pessoa_presidiario,
        pc.nome_presidiario,
        pc.documento_presidiario
    from bronze.infopen_presos_ocorrencias po
    left join gold.tmp_base_presidiario_catalogo_ocorrencia pc
        on cast(po.id_preso as string) = pc.id_preso_origem
""")

tabela = "tmp_base_infopen_ocorrencia_preso_raw"

df_base_infopen_ocorrencia_preso_raw.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_infopen_ocorrencia_preso_raw, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_infopen_ocorrencia_preso_raw")


# ============================================================
# PRESOS INFOPEN AGREGADOS
# ============================================================

df_base_infopen_ocorrencia_preso_agg = spark.sql("""
    select
        id_ocorrencia_origem,
        count(*) as qtd_presos_infopen,
        sum(case when id_pessoa_presidiario is not null then 1 else 0 end) as qtd_presos_resolvidos_infopen,
        concat_ws(
            ' | ',
            sort_array(collect_set(id_preso_origem))
        ) as txt_ids_preso_infopen,
        concat_ws(
            ' | ',
            sort_array(collect_set(cast(id_pessoa_presidiario as string)))
        ) as txt_ids_pessoa_presidiario
    from gold.tmp_base_infopen_ocorrencia_preso_raw
    group by id_ocorrencia_origem
""")

tabela = "tmp_base_infopen_ocorrencia_preso_agg"

df_base_infopen_ocorrencia_preso_agg.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_infopen_ocorrencia_preso_agg, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_infopen_ocorrencia_preso_agg")


# ============================================================
# FATO OCORRENCIA INFOPEN - MAXIMA
# ============================================================

df_fat_ocorrencia_infopen = spark.sql("""
    select
        concat('OCR_INFOPEN_', o.id_ocorrencia_origem) as id_fato_ocorrencia,
        'INFOPEN' as origem_sistema,

        o.id_ocorrencia_origem,

        o.id_tipo_ocorrencia,
        t.ds_tipo_ocorrencia,

        o.id_municipio,

        o.id_status_atual,
        su.id_status_ultimo,
        su.dt_status_ultimo,
        su.flag_status_marcado_ultimo,

        coalesce(sa.qtd_status_historico, 0) as qtd_status_historico,
        coalesce(sa.qtd_status_distintos, 0) as qtd_status_distintos,
        sa.dt_primeiro_status_historico,
        sa.dt_ultimo_status_historico,
        coalesce(sa.flag_possui_status_marcado_ultimo, 0) as flag_possui_status_marcado_ultimo,
        sa.txt_ids_status_historico,

        case
            when o.id_status_atual is not null
             and su.id_status_ultimo is not null
             and o.id_status_atual <> su.id_status_ultimo then 1
            else 0
        end as flag_status_divergente,

        case when coalesce(sa.qtd_status_distintos, 0) > 1 then 1 else 0 end as flag_multiplos_status,

        o.nr_prontuario,
        o.nr_ocorrencia,

        o.dt_fato,
        o.dt_expedicao,
        coalesce(o.dt_fato, o.dt_expedicao) as dt_evento_referencia,
        datediff(current_date(), to_date(coalesce(o.dt_fato, o.dt_expedicao))) as dias_desde_evento,

        case when o.nr_prontuario is not null then 1 else 0 end as flag_tem_prontuario,
        case when o.nr_ocorrencia is not null then 1 else 0 end as flag_tem_numero_ocorrencia,
        case when o.dt_fato is not null then 1 else 0 end as flag_tem_dt_fato,
        case when o.dt_expedicao is not null then 1 else 0 end as flag_tem_dt_expedicao,

        coalesce(obs.qtd_observacoes_infopen, 0) as qtd_observacoes_infopen,
        case when coalesce(obs.qtd_observacoes_infopen, 0) > 0 then 1 else 0 end as flag_tem_observacao,
        case when coalesce(obs.qtd_observacoes_infopen, 0) > 1 then 1 else 0 end as flag_multiplas_observacoes,
        obs.txt_observacoes_infopen,

        coalesce(pr.qtd_processos_infopen, 0) as qtd_processos_infopen,
        case when coalesce(pr.qtd_processos_infopen, 0) > 0 then 1 else 0 end as flag_tem_processo,
        case when coalesce(pr.qtd_processos_infopen, 0) > 1 then 1 else 0 end as flag_multiplos_processos,
        pr.txt_ids_processo_infopen,

        coalesce(p.qtd_presos_infopen, 0) as qtd_presos_infopen,
        coalesce(p.qtd_presos_resolvidos_infopen, 0) as qtd_presos_resolvidos_infopen,
        case when coalesce(p.qtd_presos_infopen, 0) > 0 then 1 else 0 end as flag_tem_preso,
        case when coalesce(p.qtd_presos_infopen, 0) > 1 then 1 else 0 end as flag_multiplos_presos,
        p.txt_ids_preso_infopen,
        p.txt_ids_pessoa_presidiario,

        (
            coalesce(obs.qtd_observacoes_infopen, 0) +
            coalesce(pr.qtd_processos_infopen, 0) +
            coalesce(p.qtd_presos_infopen, 0) +
            coalesce(sa.qtd_status_historico, 0)
        ) as score_complexidade_basica
    from gold.tmp_base_infopen_ocorrencias o
    left join gold.tmp_base_infopen_tipos_ocorrencia t
        on o.id_tipo_ocorrencia = t.id_tipo_ocorrencia
    left join gold.tmp_base_infopen_status_ocorrencia_ultimo su
        on o.id_ocorrencia_origem = su.id_ocorrencia_origem
    left join gold.tmp_base_infopen_status_ocorrencia_agg sa
        on o.id_ocorrencia_origem = sa.id_ocorrencia_origem
    left join gold.tmp_base_infopen_observacoes_agg obs
        on o.id_ocorrencia_origem = obs.id_ocorrencia_origem
    left join gold.tmp_base_infopen_ocorrencia_processo_agg pr
        on o.id_ocorrencia_origem = pr.id_ocorrencia_origem
    left join gold.tmp_base_infopen_ocorrencia_preso_agg p
        on o.id_ocorrencia_origem = p.id_ocorrencia_origem
""")

tabela = "tmp_fat_ocorrencia_infopen"

df_fat_ocorrencia_infopen.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_fat_ocorrencia_infopen, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_fat_ocorrencia_infopen")


# ============================================================
# FATO OCORRENCIA INFOPEN FINAL
# ============================================================

df_fat_ocorrencia_infopen_final = spark.sql("""
    select
        id_fato_ocorrencia,
        origem_sistema,
        id_ocorrencia_origem,

        id_tipo_ocorrencia,
        ds_tipo_ocorrencia,
        id_municipio,

        id_status_atual,
        id_status_ultimo,
        dt_status_ultimo,
        flag_status_marcado_ultimo,
        qtd_status_historico,
        qtd_status_distintos,
        dt_primeiro_status_historico,
        dt_ultimo_status_historico,
        flag_possui_status_marcado_ultimo,
        txt_ids_status_historico,
        flag_status_divergente,
        flag_multiplos_status,

        nr_prontuario,
        nr_ocorrencia,
        dt_fato,
        dt_expedicao,
        dt_evento_referencia,
        dias_desde_evento,
        flag_tem_prontuario,
        flag_tem_numero_ocorrencia,
        flag_tem_dt_fato,
        flag_tem_dt_expedicao,

        qtd_observacoes_infopen,
        flag_tem_observacao,
        flag_multiplas_observacoes,
        txt_observacoes_infopen,

        qtd_processos_infopen,
        flag_tem_processo,
        flag_multiplos_processos,
        txt_ids_processo_infopen,

        qtd_presos_infopen,
        qtd_presos_resolvidos_infopen,
        flag_tem_preso,
        flag_multiplos_presos,
        txt_ids_preso_infopen,
        txt_ids_pessoa_presidiario,

        score_complexidade_basica
    from gold.tmp_fat_ocorrencia_infopen
""")

tabela = "sinp_fat_ocorrencia_infopen"

df_fat_ocorrencia_infopen_final.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_fat_ocorrencia_infopen_final, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres(f"gold.{tabela}", "id_fato_ocorrencia")


# ============================================================
# RL OCORRENCIA X PRESO INFOPEN FINAL
# ============================================================

df_rl_ocorrencia_preso_infopen_final = spark.sql("""
    select
        id_rl_ocorrencia_preso,
        id_fato_ocorrencia,
        id_ocorrencia_origem,
        id_preso_origem,
        id_pessoa_presidiario,
        nome_presidiario,
        documento_presidiario
    from gold.tmp_base_infopen_ocorrencia_preso_raw
""")

tabela = "sinp_rl_ocorrencia_preso_infopen"

df_rl_ocorrencia_preso_infopen_final.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_rl_ocorrencia_preso_infopen_final, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres(f"gold.{tabela}", "id_rl_ocorrencia_preso")


# ============================================================
# RL OCORRENCIA X PROCESSO INFOPEN FINAL
# ============================================================

df_rl_ocorrencia_processo_infopen_final = spark.sql("""
    select
        id_rl_ocorrencia_processo,
        id_fato_ocorrencia,
        id_ocorrencia_origem,
        id_processo_origem
    from gold.tmp_base_infopen_ocorrencia_processo_raw
""")

tabela = "sinp_rl_ocorrencia_processo_infopen"

df_rl_ocorrencia_processo_infopen_final.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_rl_ocorrencia_processo_infopen_final, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres(f"gold.{tabela}", "id_rl_ocorrencia_processo")


# ============================================================
# RL OCORRENCIA X OBSERVACAO INFOPEN FINAL
# ============================================================

df_rl_ocorrencia_observacao_infopen_final = spark.sql("""
    select
        id_rl_ocorrencia_observacao,
        id_fato_ocorrencia,
        id_ocorrencia_origem,
        ds_observacao
    from gold.tmp_base_infopen_observacoes_raw
""")

tabela = "sinp_rl_ocorrencia_observacao_infopen"

df_rl_ocorrencia_observacao_infopen_final.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_rl_ocorrencia_observacao_infopen_final, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres(f"gold.{tabela}", "id_rl_ocorrencia_observacao")


# ============================================================
# RL OCORRENCIA X STATUS INFOPEN FINAL
# ============================================================

df_rl_ocorrencia_status_infopen_final = spark.sql("""
    select
        id_rl_ocorrencia_status,
        id_fato_ocorrencia,
        id_ocorrencia_origem,
        id_status_historico,
        dt_status_historico,
        flag_status_marcado_ultimo
    from gold.tmp_base_infopen_status_ocorrencia_raw
""")

tabela = "sinp_rl_ocorrencia_status_infopen"

df_rl_ocorrencia_status_infopen_final.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_rl_ocorrencia_status_infopen_final, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres(f"gold.{tabela}", "id_rl_ocorrencia_status")

In [ ]:
import os
import re

# ============================================================
# DESCOBERTA DA TABELA PONTE LIVRO -> INTERNO -> INFOPEN
# ============================================================

if spark.sql("show tables in bronze like 'livros_acesso_unidade_interno'").count() == 0:
    raise Exception("Tabela bronze.livros_acesso_unidade_interno não encontrada. Sem ela não é possível resolver a ponte do livro para o preso INFOPEN.")

cols_interno = spark.table("bronze.livros_acesso_unidade_interno").columns

def pick_col(cols, candidatos, obrigatoria=True, rotulo="coluna"):
    norm = {re.sub(r'[^a-z0-9]', '', c.lower()): c for c in cols}
    for cand in candidatos:
        k = re.sub(r'[^a-z0-9]', '', cand.lower())
        if k in norm:
            return norm[k]
    if obrigatoria:
        raise Exception(f"Não foi possível localizar {rotulo} em bronze.livros_acesso_unidade_interno. Colunas disponíveis: {cols}")
    return None

col_id_interno = pick_col(cols_interno, ["id"], True, "id do interno")
col_infopen = pick_col(cols_interno, ["infopen", "id_preso", "preso_id"], True, "chave INFOPEN do interno")
col_nome_interno = pick_col(cols_interno, ["nome", "interno", "nome_interno", "descricao"], True, "nome do interno")

print("Tabela ponte detectada:")
print(f"id interno   : {col_id_interno}")
print(f"id infopen   : {col_infopen}")
print(f"nome interno : {col_nome_interno}")


# ============================================================
# LIMPEZA DEFENSIVA
# ============================================================

tabelas_drop = [
    "tmp_raw_livro_ocorrencia",
    "tmp_raw_livro_registrovinculoocorrencia",
    "tmp_raw_livro_vinculacaoocorrencias",
    "tmp_raw_livro_historicalocorrencia",
    "tmp_raw_livro_historicalregistrovinculoocorrencia",
    "tmp_raw_livro_historicalvinculacaoocorrencias",
    "tmp_raw_livro_interno",
    "tmp_base_livro_ocorrencia",
    "tmp_base_livro_registrovinculoocorrencia_agg",
    "tmp_base_livro_vinculacaoocorrencias_raw",
    "tmp_base_livro_vinculacaoocorrencias_agg",
    "tmp_base_livro_historicalocorrencia_agg",
    "tmp_base_livro_historicalregistrovinculoocorrencia_agg",
    "tmp_base_livro_historicalvinculacaoocorrencias_agg",
    "tmp_base_livro_interno_catalogo",
    "tmp_base_livro_interno_catalogo_nome_unico",
    "tmp_base_pessoa_preso_ponte_livro",
    "tmp_base_presidiario_catalogo_livro",
    "tmp_base_livro_interno_tokenizado",
    "tmp_rl_ocorrencia_preso_livro",
    "tmp_base_livro_preso_agg",
    "tmp_fat_ocorrencia_livro",
    "sinp_fat_ocorrencia_livro",
    "sinp_rl_ocorrencia_entidade_livro_raw",
    "sinp_rl_ocorrencia_preso_livro"
]

for t in tabelas_drop:
    spark.sql(f"drop table if exists gold.{t}")
    os.system(f"hdfs dfs -rm -r -skipTrash {path}{t} >/dev/null 2>&1")

spark.catalog.clearCache()

spark.sql("refresh table bronze.livros_acesso_unidade_ocorrencia")
spark.sql("refresh table bronze.livros_acesso_unidade_registrovinculoocorrencia")
spark.sql("refresh table bronze.livros_acesso_unidade_vinculacaoocorrencias")
spark.sql("refresh table bronze.livros_acesso_unidade_historicalocorrencia")
spark.sql("refresh table bronze.livros_acesso_unidade_historicalregistrovinculoocorrencia")
spark.sql("refresh table bronze.livros_acesso_unidade_historicalvinculacaoocorrencias")
spark.sql("refresh table bronze.livros_acesso_unidade_interno")

spark.sql("refresh table gold.sinp_pnt_pessoa_preso")
spark.sql("refresh table gold.sinp_ent_pessoa")


# ============================================================
# RAW LIVRO OCORRENCIA
# ============================================================

df_raw_livro_ocorrencia = spark.sql("""
    select *
    from bronze.livros_acesso_unidade_ocorrencia
""")

tabela = "tmp_raw_livro_ocorrencia"

df_raw_livro_ocorrencia.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_raw_livro_ocorrencia, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_raw_livro_ocorrencia")


# ============================================================
# RAW LIVRO REGISTRO VINCULO OCORRENCIA
# ============================================================

df_raw_livro_registrovinculoocorrencia = spark.sql("""
    select *
    from bronze.livros_acesso_unidade_registrovinculoocorrencia
""")

tabela = "tmp_raw_livro_registrovinculoocorrencia"

df_raw_livro_registrovinculoocorrencia.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_raw_livro_registrovinculoocorrencia, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_raw_livro_registrovinculoocorrencia")


# ============================================================
# RAW LIVRO VINCULACAO OCORRENCIAS
# ============================================================

df_raw_livro_vinculacaoocorrencias = spark.sql("""
    select *
    from bronze.livros_acesso_unidade_vinculacaoocorrencias
""")

tabela = "tmp_raw_livro_vinculacaoocorrencias"

df_raw_livro_vinculacaoocorrencias.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_raw_livro_vinculacaoocorrencias, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_raw_livro_vinculacaoocorrencias")


# ============================================================
# RAW HISTORICOS LIVRO
# ============================================================

for origem, destino in [
    ("bronze.livros_acesso_unidade_historicalocorrencia", "tmp_raw_livro_historicalocorrencia"),
    ("bronze.livros_acesso_unidade_historicalregistrovinculoocorrencia", "tmp_raw_livro_historicalregistrovinculoocorrencia"),
    ("bronze.livros_acesso_unidade_historicalvinculacaoocorrencias", "tmp_raw_livro_historicalvinculacaoocorrencias")
]:
    df = spark.sql(f"select * from {origem}")
    df.write \
        .mode("overwrite") \
        .option("maxRecordsPerFile", 1_000_000) \
        .option("compression", "snappy") \
        .parquet(f"{path}{destino}")
    write_impala_table_partioned(df, "gold", destino, f"{path}{destino}")
    spark.catalog.clearCache()
    spark.sql(f"refresh table gold.{destino}")


# ============================================================
# RAW LIVRO INTERNO (PONTE PARA INFOPEN)
# ============================================================

sql_raw_livro_interno = f"""
    select
        cast({col_id_interno} as string) as id_interno_origem,
        cast({col_infopen} as string) as id_preso_infopen,
        trim(regexp_replace(coalesce(cast({col_nome_interno} as string), ''), '\\\\s+', ' ')) as nome_interno,
        upper(trim(regexp_replace(coalesce(cast({col_nome_interno} as string), ''), '\\\\s+', ' '))) as nome_interno_normalizado
    from bronze.livros_acesso_unidade_interno
"""

df_raw_livro_interno = spark.sql(sql_raw_livro_interno)

tabela = "tmp_raw_livro_interno"

df_raw_livro_interno.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_raw_livro_interno, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_raw_livro_interno")


# ============================================================
# BASE LIVRO OCORRENCIA
# ============================================================

df_base_livro_ocorrencia = spark.sql("""
    select
        cast(id as string) as id_ocorrencia_origem,
        trim(regexp_replace(coalesce(motivo, ''), '\\\\s+', ' ')) as motivo,
        trim(regexp_replace(coalesce(registro, ''), '\\\\s+', ' ')) as registro,
        trim(regexp_replace(coalesce(arquivo, ''), '\\\\s+', ' ')) as arquivo,
        to_timestamp(data_registro) as dt_registro,
        cast(equipe_id as string) as id_equipe_origem,
        cast(presidio_id as string) as id_presidio_origem
    from gold.tmp_raw_livro_ocorrencia
""")

tabela = "tmp_base_livro_ocorrencia"

df_base_livro_ocorrencia.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_livro_ocorrencia, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_livro_ocorrencia")


# ============================================================
# REGISTRO VINCULO OCORRENCIA AGREGADO
# ============================================================

df_base_livro_registrovinculoocorrencia_agg = spark.sql("""
    select
        cast(ocorrencia_id as string) as id_ocorrencia_origem,
        count(*) as qtd_registrovinculo_livro,
        min(to_timestamp(data_registro)) as dt_primeiro_registrovinculo_livro,
        max(to_timestamp(data_registro)) as dt_ultimo_registrovinculo_livro
    from gold.tmp_raw_livro_registrovinculoocorrencia
    group by cast(ocorrencia_id as string)
""")

tabela = "tmp_base_livro_registrovinculoocorrencia_agg"

df_base_livro_registrovinculoocorrencia_agg.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_livro_registrovinculoocorrencia_agg, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_livro_registrovinculoocorrencia_agg")


# ============================================================
# RL RAW OCORRENCIA X ENTIDADE LIVRO - CORRIGIDA
# ============================================================

df_base_livro_vinculacaoocorrencias_raw = spark.sql("""
    with base_union as (

        select
            cast(ocorrencia_id as string) as id_ocorrencia_origem,
            'LIVRO' as origem_sistema,
            'INTERNO_RAW' as tipo_entidade_raw,
            trim(regexp_replace(coalesce(interno, ''), '\\\\s+', ' ')) as valor_entidade_raw,
            to_timestamp(data_registro) as dt_registro,
            cast(equipe_id as string) as id_equipe_origem,
            cast(presidio_id as string) as id_presidio_origem
        from gold.tmp_raw_livro_vinculacaoocorrencias
        where ocorrencia_id is not null
          and trim(coalesce(interno, '')) <> ''

        union all

        select
            cast(ocorrencia_id as string),
            'LIVRO',
            'INTERNOS_RAW',
            trim(regexp_replace(coalesce(internos, ''), '\\\\s+', ' ')),
            to_timestamp(data_registro),
            cast(equipe_id as string),
            cast(presidio_id as string)
        from gold.tmp_raw_livro_vinculacaoocorrencias
        where ocorrencia_id is not null
          and trim(coalesce(internos, '')) <> ''

        union all

        select
            cast(ocorrencia_id as string),
            'LIVRO',
            'ADVOGADO_RAW',
            trim(regexp_replace(coalesce(advogado, ''), '\\\\s+', ' ')),
            to_timestamp(data_registro),
            cast(equipe_id as string),
            cast(presidio_id as string)
        from gold.tmp_raw_livro_vinculacaoocorrencias
        where ocorrencia_id is not null
          and trim(coalesce(advogado, '')) <> ''

        union all

        select
            cast(ocorrencia_id as string),
            'LIVRO',
            'ADVOGADOS_RAW',
            trim(regexp_replace(coalesce(advogados, ''), '\\\\s+', ' ')),
            to_timestamp(data_registro),
            cast(equipe_id as string),
            cast(presidio_id as string)
        from gold.tmp_raw_livro_vinculacaoocorrencias
        where ocorrencia_id is not null
          and trim(coalesce(advogados, '')) <> ''

        union all

        select
            cast(ocorrencia_id as string),
            'LIVRO',
            'ASS_RELIGIOSA_RAW',
            trim(regexp_replace(coalesce(ass_religiosa, ''), '\\\\s+', ' ')),
            to_timestamp(data_registro),
            cast(equipe_id as string),
            cast(presidio_id as string)
        from gold.tmp_raw_livro_vinculacaoocorrencias
        where ocorrencia_id is not null
          and trim(coalesce(ass_religiosa, '')) <> ''

        union all

        select
            cast(ocorrencia_id as string),
            'LIVRO',
            'ASS_RELIGIOSAS_RAW',
            trim(regexp_replace(coalesce(ass_religiosas, ''), '\\\\s+', ' ')),
            to_timestamp(data_registro),
            cast(equipe_id as string),
            cast(presidio_id as string)
        from gold.tmp_raw_livro_vinculacaoocorrencias
        where ocorrencia_id is not null
          and trim(coalesce(ass_religiosas, '')) <> ''

        union all

        select
            cast(ocorrencia_id as string),
            'LIVRO',
            'POLICIAL_RAW',
            trim(regexp_replace(coalesce(policial, ''), '\\\\s+', ' ')),
            to_timestamp(data_registro),
            cast(equipe_id as string),
            cast(presidio_id as string)
        from gold.tmp_raw_livro_vinculacaoocorrencias
        where ocorrencia_id is not null
          and trim(coalesce(policial, '')) <> ''

        union all

        select
            cast(ocorrencia_id as string),
            'LIVRO',
            'POLICIAIS_RAW',
            trim(regexp_replace(coalesce(policiais, ''), '\\\\s+', ' ')),
            to_timestamp(data_registro),
            cast(equipe_id as string),
            cast(presidio_id as string)
        from gold.tmp_raw_livro_vinculacaoocorrencias
        where ocorrencia_id is not null
          and trim(coalesce(policiais, '')) <> ''

        union all

        select
            cast(ocorrencia_id as string),
            'LIVRO',
            'SERVIDOR_RAW',
            trim(regexp_replace(coalesce(servidor, ''), '\\\\s+', ' ')),
            to_timestamp(data_registro),
            cast(equipe_id as string),
            cast(presidio_id as string)
        from gold.tmp_raw_livro_vinculacaoocorrencias
        where ocorrencia_id is not null
          and trim(coalesce(servidor, '')) <> ''

        union all

        select
            cast(ocorrencia_id as string),
            'LIVRO',
            'SERVIDORES_RAW',
            trim(regexp_replace(coalesce(servidores, ''), '\\\\s+', ' ')),
            to_timestamp(data_registro),
            cast(equipe_id as string),
            cast(presidio_id as string)
        from gold.tmp_raw_livro_vinculacaoocorrencias
        where ocorrencia_id is not null
          and trim(coalesce(servidores, '')) <> ''

        union all

        select
            cast(ocorrencia_id as string),
            'LIVRO',
            'VISITANTE_RAW',
            trim(regexp_replace(coalesce(visitante, ''), '\\\\s+', ' ')),
            to_timestamp(data_registro),
            cast(equipe_id as string),
            cast(presidio_id as string)
        from gold.tmp_raw_livro_vinculacaoocorrencias
        where ocorrencia_id is not null
          and trim(coalesce(visitante, '')) <> ''

        union all

        select
            cast(ocorrencia_id as string),
            'LIVRO',
            'VISITANTES_RAW',
            trim(regexp_replace(coalesce(visitantes, ''), '\\\\s+', ' ')),
            to_timestamp(data_registro),
            cast(equipe_id as string),
            cast(presidio_id as string)
        from gold.tmp_raw_livro_vinculacaoocorrencias
        where ocorrencia_id is not null
          and trim(coalesce(visitantes, '')) <> ''
    ),
    base_distinct as (
        select distinct
            id_ocorrencia_origem,
            origem_sistema,
            tipo_entidade_raw,
            valor_entidade_raw,
            dt_registro,
            id_equipe_origem,
            id_presidio_origem
        from base_union
        where id_ocorrencia_origem is not null
          and trim(coalesce(valor_entidade_raw, '')) <> ''
    )
    select
        concat(
            'RLRAW_',
            md5(
                concat_ws(
                    '|',
                    coalesce(id_ocorrencia_origem, ''),
                    coalesce(tipo_entidade_raw, ''),
                    coalesce(valor_entidade_raw, ''),
                    coalesce(cast(dt_registro as string), ''),
                    coalesce(id_equipe_origem, ''),
                    coalesce(id_presidio_origem, '')
                )
            )
        ) as id_rl_ocorrencia_entidade_raw,
        concat('OCR_LIVRO_', id_ocorrencia_origem) as id_fato_ocorrencia,
        id_ocorrencia_origem,
        origem_sistema,
        tipo_entidade_raw,
        valor_entidade_raw,
        dt_registro,
        id_equipe_origem,
        id_presidio_origem
    from base_distinct
""")

tabela = "tmp_base_livro_vinculacaoocorrencias_raw"

df_base_livro_vinculacaoocorrencias_raw.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_livro_vinculacaoocorrencias_raw, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_livro_vinculacaoocorrencias_raw")


# ============================================================
# AGREGADO DE VINCULOS RAW
# ============================================================

df_base_livro_vinculacaoocorrencias_agg = spark.sql("""
    select
        id_ocorrencia_origem,
        count(*) as qtd_vinculos_raw_livro,
        sum(case when tipo_entidade_raw in ('INTERNO_RAW', 'INTERNOS_RAW') then 1 else 0 end) as qtd_internos_raw,
        sum(case when tipo_entidade_raw in ('ADVOGADO_RAW', 'ADVOGADOS_RAW') then 1 else 0 end) as qtd_advogados_raw,
        sum(case when tipo_entidade_raw in ('ASS_RELIGIOSA_RAW', 'ASS_RELIGIOSAS_RAW') then 1 else 0 end) as qtd_ass_religiosas_raw,
        sum(case when tipo_entidade_raw in ('POLICIAL_RAW', 'POLICIAIS_RAW') then 1 else 0 end) as qtd_policiais_raw,
        sum(case when tipo_entidade_raw in ('SERVIDOR_RAW', 'SERVIDORES_RAW') then 1 else 0 end) as qtd_servidores_raw,
        sum(case when tipo_entidade_raw in ('VISITANTE_RAW', 'VISITANTES_RAW') then 1 else 0 end) as qtd_visitantes_raw,
        concat_ws(' | ', sort_array(collect_set(case when tipo_entidade_raw in ('INTERNO_RAW', 'INTERNOS_RAW') then valor_entidade_raw end))) as txt_internos_raw,
        concat_ws(' | ', sort_array(collect_set(case when tipo_entidade_raw in ('ADVOGADO_RAW', 'ADVOGADOS_RAW') then valor_entidade_raw end))) as txt_advogados_raw,
        concat_ws(' | ', sort_array(collect_set(case when tipo_entidade_raw in ('ASS_RELIGIOSA_RAW', 'ASS_RELIGIOSAS_RAW') then valor_entidade_raw end))) as txt_ass_religiosas_raw,
        concat_ws(' | ', sort_array(collect_set(case when tipo_entidade_raw in ('POLICIAL_RAW', 'POLICIAIS_RAW') then valor_entidade_raw end))) as txt_policiais_raw,
        concat_ws(' | ', sort_array(collect_set(case when tipo_entidade_raw in ('SERVIDOR_RAW', 'SERVIDORES_RAW') then valor_entidade_raw end))) as txt_servidores_raw,
        concat_ws(' | ', sort_array(collect_set(case when tipo_entidade_raw in ('VISITANTE_RAW', 'VISITANTES_RAW') then valor_entidade_raw end))) as txt_visitantes_raw
    from gold.tmp_base_livro_vinculacaoocorrencias_raw
    group by id_ocorrencia_origem
""")

tabela = "tmp_base_livro_vinculacaoocorrencias_agg"

df_base_livro_vinculacaoocorrencias_agg.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_livro_vinculacaoocorrencias_agg, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_livro_vinculacaoocorrencias_agg")


# ============================================================
# HISTORICOS AGREGADOS
# ============================================================

df_base_livro_historicalocorrencia_agg = spark.sql("""
    select
        cast(id as string) as id_ocorrencia_origem,
        count(*) as qtd_hist_ocorrencia_livro,
        min(to_timestamp(history_date)) as dt_primeiro_hist_ocorrencia_livro,
        max(to_timestamp(history_date)) as dt_ultimo_hist_ocorrencia_livro
    from gold.tmp_raw_livro_historicalocorrencia
    group by cast(id as string)
""")

tabela = "tmp_base_livro_historicalocorrencia_agg"

df_base_livro_historicalocorrencia_agg.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_livro_historicalocorrencia_agg, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_livro_historicalocorrencia_agg")


df_base_livro_historicalregistrovinculoocorrencia_agg = spark.sql("""
    select
        cast(ocorrencia_id as string) as id_ocorrencia_origem,
        count(*) as qtd_hist_registrovinculo_livro,
        min(to_timestamp(history_date)) as dt_primeiro_hist_registrovinculo_livro,
        max(to_timestamp(history_date)) as dt_ultimo_hist_registrovinculo_livro
    from gold.tmp_raw_livro_historicalregistrovinculoocorrencia
    group by cast(ocorrencia_id as string)
""")

tabela = "tmp_base_livro_historicalregistrovinculoocorrencia_agg"

df_base_livro_historicalregistrovinculoocorrencia_agg.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_livro_historicalregistrovinculoocorrencia_agg, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_livro_historicalregistrovinculoocorrencia_agg")


df_base_livro_historicalvinculacaoocorrencias_agg = spark.sql("""
    select
        cast(ocorrencia_id as string) as id_ocorrencia_origem,
        count(*) as qtd_hist_vinculacao_livro,
        min(to_timestamp(history_date)) as dt_primeiro_hist_vinculacao_livro,
        max(to_timestamp(history_date)) as dt_ultimo_hist_vinculacao_livro
    from gold.tmp_raw_livro_historicalvinculacaoocorrencias
    group by cast(ocorrencia_id as string)
""")

tabela = "tmp_base_livro_historicalvinculacaoocorrencias_agg"

df_base_livro_historicalvinculacaoocorrencias_agg.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_livro_historicalvinculacaoocorrencias_agg, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_livro_historicalvinculacaoocorrencias_agg")


# ============================================================
# CATALOGO DE INTERNOS DO LIVRO
# ============================================================

df_base_livro_interno_catalogo = spark.sql("""
    select
        id_interno_origem,
        id_preso_infopen,
        nome_interno,
        nome_interno_normalizado
    from gold.tmp_raw_livro_interno
    where id_preso_infopen is not null
      and nome_interno_normalizado is not null
      and nome_interno_normalizado <> ''
""")

tabela = "tmp_base_livro_interno_catalogo"

df_base_livro_interno_catalogo.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_livro_interno_catalogo, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_livro_interno_catalogo")


df_base_livro_interno_catalogo_nome_unico = spark.sql("""
    select
        nome_interno_normalizado,
        max(id_interno_origem) as id_interno_origem,
        max(id_preso_infopen) as id_preso_infopen
    from gold.tmp_base_livro_interno_catalogo
    group by nome_interno_normalizado
    having count(distinct id_preso_infopen) = 1
""")

tabela = "tmp_base_livro_interno_catalogo_nome_unico"

df_base_livro_interno_catalogo_nome_unico.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_livro_interno_catalogo_nome_unico, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_livro_interno_catalogo_nome_unico")


# ============================================================
# PONTE PESSOA X PRESO
# ============================================================

df_base_pessoa_preso_ponte_livro = spark.sql("""
    select distinct
        cast(id_preso as string) as id_preso_infopen,
        cast(id_preso as string) as id_preso_origem,
        id_pessoa as id_pessoa_presidiario
    from gold.sinp_pnt_pessoa_preso
    where id_preso is not null
      and id_pessoa is not null
""")

tabela = "tmp_base_pessoa_preso_ponte_livro"

df_base_pessoa_preso_ponte_livro.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_pessoa_preso_ponte_livro, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_pessoa_preso_ponte_livro")


df_base_presidiario_catalogo_livro = spark.sql("""
    select
        p.id_preso_infopen,
        p.id_pessoa_presidiario,
        e.nome_pessoa as nome_presidiario,
        e.documento as documento_presidiario
    from gold.tmp_base_pessoa_preso_ponte_livro p
    left join gold.sinp_ent_pessoa e
        on p.id_pessoa_presidiario = e.id_pessoa
""")

tabela = "tmp_base_presidiario_catalogo_livro"

df_base_presidiario_catalogo_livro.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_presidiario_catalogo_livro, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_presidiario_catalogo_livro")


# ============================================================
# TOKENIZACAO DOS INTERNOS DO LIVRO
# ============================================================

df_base_livro_interno_tokenizado = spark.sql("""
    with base as (
        select
            id_fato_ocorrencia,
            id_ocorrencia_origem,
            dt_registro,
            id_equipe_origem,
            id_presidio_origem,
            valor_entidade_raw
        from gold.tmp_base_livro_vinculacaoocorrencias_raw
        where tipo_entidade_raw in ('INTERNO_RAW', 'INTERNOS_RAW')
          and trim(coalesce(valor_entidade_raw, '')) <> ''
    )
    select
        id_fato_ocorrencia,
        id_ocorrencia_origem,
        dt_registro,
        id_equipe_origem,
        id_presidio_origem,
        trim(token) as nome_interno_raw,
        upper(trim(regexp_replace(trim(token), '\\\\s+', ' '))) as nome_interno_normalizado
    from (
        select
            id_fato_ocorrencia,
            id_ocorrencia_origem,
            dt_registro,
            id_equipe_origem,
            id_presidio_origem,
            explode(
                split(
                    regexp_replace(valor_entidade_raw, '\\\\s*[,;|]+\\\\s*', '|'),
                    '\\\\|'
                )
            ) as token
        from base
    ) z
    where trim(coalesce(token, '')) <> ''
""")

tabela = "tmp_base_livro_interno_tokenizado"

df_base_livro_interno_tokenizado.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_livro_interno_tokenizado, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_livro_interno_tokenizado")


# ============================================================
# RL OCORRENCIA X PRESO LIVRO
# ============================================================

df_rl_ocorrencia_preso_livro = spark.sql("""
    select
        concat(
            'RLPRESOLIVRO_',
            md5(
                concat_ws(
                    '|',
                    t.id_ocorrencia_origem,
                    coalesce(c.id_preso_infopen, ''),
                    t.nome_interno_normalizado
                )
            )
        ) as id_rl_ocorrencia_preso_livro,
        t.id_fato_ocorrencia,
        t.id_ocorrencia_origem,
        c.id_interno_origem,
        c.id_preso_infopen as id_preso_origem,
        p.id_pessoa_presidiario,
        p.nome_presidiario,
        p.documento_presidiario,
        t.nome_interno_raw,
        'NOME_UNICO_CATALOGO_LIVRO' as origem_resolucao
    from gold.tmp_base_livro_interno_tokenizado t
    inner join gold.tmp_base_livro_interno_catalogo_nome_unico c
        on t.nome_interno_normalizado = c.nome_interno_normalizado
    left join gold.tmp_base_presidiario_catalogo_livro p
        on c.id_preso_infopen = p.id_preso_infopen
""")

tabela = "tmp_rl_ocorrencia_preso_livro"

df_rl_ocorrencia_preso_livro.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_rl_ocorrencia_preso_livro, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_rl_ocorrencia_preso_livro")


# ============================================================
# AGREGADO PRESO LIVRO
# ============================================================

df_base_livro_preso_agg = spark.sql("""
    select
        id_ocorrencia_origem,
        count(*) as qtd_presos_resolvidos_livro,
        count(distinct id_preso_origem) as qtd_presos_distintos_livro,
        concat_ws(
            ' | ',
            sort_array(collect_set(id_preso_origem))
        ) as txt_ids_preso_infopen_livro,
        concat_ws(
            ' | ',
            sort_array(collect_set(cast(id_pessoa_presidiario as string)))
        ) as txt_ids_pessoa_presidiario_livro,
        concat_ws(
            ' | ',
            sort_array(collect_set(nome_presidiario))
        ) as txt_nomes_presidiario_livro
    from gold.tmp_rl_ocorrencia_preso_livro
    group by id_ocorrencia_origem
""")

tabela = "tmp_base_livro_preso_agg"

df_base_livro_preso_agg.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_livro_preso_agg, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_livro_preso_agg")


# ============================================================
# FATO OCORRENCIA LIVRO
# ============================================================

df_fat_ocorrencia_livro = spark.sql("""
    select
        concat('OCR_LIVRO_', o.id_ocorrencia_origem) as id_fato_ocorrencia,
        'LIVRO' as origem_sistema,

        o.id_ocorrencia_origem,
        o.id_presidio_origem,
        o.id_equipe_origem,

        o.dt_registro as dt_evento_referencia,
        o.dt_registro,
        o.motivo,
        o.registro,
        o.arquivo,

        case when coalesce(o.motivo, '') <> '' then 1 else 0 end as flag_tem_motivo,
        case when coalesce(o.registro, '') <> '' then 1 else 0 end as flag_tem_registro,
        case when coalesce(o.arquivo, '') <> '' then 1 else 0 end as flag_tem_arquivo,

        coalesce(rv.qtd_registrovinculo_livro, 0) as qtd_registrovinculo_livro,
        rv.dt_primeiro_registrovinculo_livro,
        rv.dt_ultimo_registrovinculo_livro,

        coalesce(v.qtd_vinculos_raw_livro, 0) as qtd_vinculos_raw_livro,
        coalesce(v.qtd_internos_raw, 0) as qtd_internos_raw,
        coalesce(v.qtd_advogados_raw, 0) as qtd_advogados_raw,
        coalesce(v.qtd_ass_religiosas_raw, 0) as qtd_ass_religiosas_raw,
        coalesce(v.qtd_policiais_raw, 0) as qtd_policiais_raw,
        coalesce(v.qtd_servidores_raw, 0) as qtd_servidores_raw,
        coalesce(v.qtd_visitantes_raw, 0) as qtd_visitantes_raw,

        v.txt_internos_raw,
        v.txt_advogados_raw,
        v.txt_ass_religiosas_raw,
        v.txt_policiais_raw,
        v.txt_servidores_raw,
        v.txt_visitantes_raw,

        coalesce(p.qtd_presos_resolvidos_livro, 0) as qtd_presos_resolvidos_livro,
        coalesce(p.qtd_presos_distintos_livro, 0) as qtd_presos_distintos_livro,
        p.txt_ids_preso_infopen_livro,
        p.txt_ids_pessoa_presidiario_livro,
        p.txt_nomes_presidiario_livro,

        case when coalesce(p.qtd_presos_distintos_livro, 0) > 0 then 1 else 0 end as flag_tem_preso_resolvido,
        case when coalesce(p.qtd_presos_distintos_livro, 0) > 1 then 1 else 0 end as flag_multiplos_presos_resolvidos,

        coalesce(ho.qtd_hist_ocorrencia_livro, 0) as qtd_hist_ocorrencia_livro,
        coalesce(hr.qtd_hist_registrovinculo_livro, 0) as qtd_hist_registrovinculo_livro,
        coalesce(hv.qtd_hist_vinculacao_livro, 0) as qtd_hist_vinculacao_livro,

        ho.dt_primeiro_hist_ocorrencia_livro,
        ho.dt_ultimo_hist_ocorrencia_livro,
        hr.dt_primeiro_hist_registrovinculo_livro,
        hr.dt_ultimo_hist_registrovinculo_livro,
        hv.dt_primeiro_hist_vinculacao_livro,
        hv.dt_ultimo_hist_vinculacao_livro,

        (
            coalesce(rv.qtd_registrovinculo_livro, 0) +
            coalesce(v.qtd_vinculos_raw_livro, 0) +
            coalesce(p.qtd_presos_distintos_livro, 0) +
            coalesce(ho.qtd_hist_ocorrencia_livro, 0) +
            coalesce(hr.qtd_hist_registrovinculo_livro, 0) +
            coalesce(hv.qtd_hist_vinculacao_livro, 0)
        ) as score_complexidade_basica
    from gold.tmp_base_livro_ocorrencia o
    left join gold.tmp_base_livro_registrovinculoocorrencia_agg rv
        on o.id_ocorrencia_origem = rv.id_ocorrencia_origem
    left join gold.tmp_base_livro_vinculacaoocorrencias_agg v
        on o.id_ocorrencia_origem = v.id_ocorrencia_origem
    left join gold.tmp_base_livro_preso_agg p
        on o.id_ocorrencia_origem = p.id_ocorrencia_origem
    left join gold.tmp_base_livro_historicalocorrencia_agg ho
        on o.id_ocorrencia_origem = ho.id_ocorrencia_origem
    left join gold.tmp_base_livro_historicalregistrovinculoocorrencia_agg hr
        on o.id_ocorrencia_origem = hr.id_ocorrencia_origem
    left join gold.tmp_base_livro_historicalvinculacaoocorrencias_agg hv
        on o.id_ocorrencia_origem = hv.id_ocorrencia_origem
""")

tabela = "tmp_fat_ocorrencia_livro"

df_fat_ocorrencia_livro.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_fat_ocorrencia_livro, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_fat_ocorrencia_livro")


# ============================================================
# FATO OCORRENCIA LIVRO FINAL
# ============================================================

df_fat_ocorrencia_livro_final = spark.sql("""
    select
        id_fato_ocorrencia,
        origem_sistema,
        id_ocorrencia_origem,
        id_presidio_origem,
        id_equipe_origem,
        dt_evento_referencia,
        dt_registro,
        motivo,
        registro,
        arquivo,
        flag_tem_motivo,
        flag_tem_registro,
        flag_tem_arquivo,
        qtd_registrovinculo_livro,
        dt_primeiro_registrovinculo_livro,
        dt_ultimo_registrovinculo_livro,
        qtd_vinculos_raw_livro,
        qtd_internos_raw,
        qtd_advogados_raw,
        qtd_ass_religiosas_raw,
        qtd_policiais_raw,
        qtd_servidores_raw,
        qtd_visitantes_raw,
        txt_internos_raw,
        txt_advogados_raw,
        txt_ass_religiosas_raw,
        txt_policiais_raw,
        txt_servidores_raw,
        txt_visitantes_raw,
        qtd_presos_resolvidos_livro,
        qtd_presos_distintos_livro,
        txt_ids_preso_infopen_livro,
        txt_ids_pessoa_presidiario_livro,
        txt_nomes_presidiario_livro,
        flag_tem_preso_resolvido,
        flag_multiplos_presos_resolvidos,
        qtd_hist_ocorrencia_livro,
        qtd_hist_registrovinculo_livro,
        qtd_hist_vinculacao_livro,
        dt_primeiro_hist_ocorrencia_livro,
        dt_ultimo_hist_ocorrencia_livro,
        dt_primeiro_hist_registrovinculo_livro,
        dt_ultimo_hist_registrovinculo_livro,
        dt_primeiro_hist_vinculacao_livro,
        dt_ultimo_hist_vinculacao_livro,
        score_complexidade_basica
    from gold.tmp_fat_ocorrencia_livro
""")

tabela = "sinp_fat_ocorrencia_livro"

df_fat_ocorrencia_livro_final.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_fat_ocorrencia_livro_final, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres(f"gold.{tabela}", "id_fato_ocorrencia")


# ============================================================
# RL OCORRENCIA X ENTIDADE RAW LIVRO FINAL
# ============================================================

df_rl_ocorrencia_entidade_livro_raw_final = spark.sql("""
    select distinct
        id_rl_ocorrencia_entidade_raw,
        id_fato_ocorrencia,
        id_ocorrencia_origem,
        origem_sistema,
        tipo_entidade_raw,
        valor_entidade_raw,
        dt_registro,
        id_equipe_origem,
        id_presidio_origem
    from gold.tmp_base_livro_vinculacaoocorrencias_raw
    where id_rl_ocorrencia_entidade_raw is not null
      and id_fato_ocorrencia is not null
      and id_ocorrencia_origem is not null
""")

tabela = "sinp_rl_ocorrencia_entidade_livro_raw"

df_rl_ocorrencia_entidade_livro_raw_final.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_rl_ocorrencia_entidade_livro_raw_final, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres(f"gold.{tabela}", "id_rl_ocorrencia_entidade_raw")


# ============================================================
# RL OCORRENCIA X PRESO LIVRO FINAL
# ============================================================

df_rl_ocorrencia_preso_livro_final = spark.sql("""
    select distinct
        id_rl_ocorrencia_preso_livro,
        id_fato_ocorrencia,
        id_ocorrencia_origem,
        id_interno_origem,
        id_preso_origem,
        id_pessoa_presidiario,
        nome_presidiario,
        documento_presidiario,
        nome_interno_raw,
        origem_resolucao
    from gold.tmp_rl_ocorrencia_preso_livro
""")

tabela = "sinp_rl_ocorrencia_preso_livro"

df_rl_ocorrencia_preso_livro_final.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_rl_ocorrencia_preso_livro_final, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres(f"gold.{tabela}", "id_rl_ocorrencia_preso_livro")

In [ ]:
import os
import re
import math
import json
import hashlib
import unicodedata
from datetime import datetime

import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql import types as T

# ============================================================
# TENTATIVA DE CAMADA SEMANTICA LOCAL
# ============================================================

try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    SKLEARN_OK = True
except Exception:
    SKLEARN_OK = False

from difflib import SequenceMatcher


# ============================================================
# VALIDACAO DE ENTRADA
# ============================================================

if spark.sql("show tables in gold like 'sinp_fat_ocorrencia_livro'").count() == 0:
    raise Exception("Tabela gold.sinp_fat_ocorrencia_livro não encontrada. Gere a fato do livro antes da classificação semântica.")


# ============================================================
# LIMPEZA DEFENSIVA
# ============================================================

tabelas_drop = [
    "tmp_ocorrencia_livro_texto_base",
    "tmp_ocorrencia_livro_texto_novo",
    "tmp_dim_classificacao_ocorrencia_livro_novo",
    "tmp_dim_classificacao_ocorrencia_livro_final",
    "sinp_dim_classificacao_ocorrencia_livro",
    "sinp_fat_ocorrencia_livro_classificada"
]

for t in tabelas_drop:
    spark.sql(f"drop table if exists gold.{t}")
    os.system(f"hdfs dfs -rm -r -skipTrash {path}{t} >/dev/null 2>&1")

spark.catalog.clearCache()
spark.sql("refresh table gold.sinp_fat_ocorrencia_livro")


# ============================================================
# TAXONOMIA FECHADA
# ============================================================

TAXONOMIA = [
    {
        "macroclasse": "SEGURANCA",
        "classe": "FUGA_EVASAO",
        "subclasse": "FUGA_CONSUMADA",
        "grau_risco": 10,
        "criticidade": "CRITICA",
        "patterns": [
            r"\bfuga\b", r"\bevas[aã]o\b", r"\bevadiu\b", r"\bforagid", r"\bempreendeu fuga\b"
        ],
        "examples": [
            "fuga consumada de interno",
            "evasao da unidade prisional",
            "interno evadiu"
        ]
    },
    {
        "macroclasse": "SEGURANCA",
        "classe": "FUGA_EVASAO",
        "subclasse": "TENTATIVA_DE_FUGA",
        "grau_risco": 9,
        "criticidade": "CRITICA",
        "patterns": [
            r"tentativa de fuga", r"tentou fugir", r"tentativa de evas[aã]o",
            r"serra", r"buraco na cela", r"rompimento de grade"
        ],
        "examples": [
            "tentativa de fuga",
            "interno tentou fugir",
            "tentativa de evasao da cela"
        ]
    },
    {
        "macroclasse": "SEGURANCA",
        "classe": "APREENSAO_ILICITO",
        "subclasse": "CELULAR_ELETRONICO",
        "grau_risco": 8,
        "criticidade": "ALTA",
        "patterns": [
            r"\bcelular\b", r"\btelefone\b", r"\bsmartphone\b", r"\bchip\b",
            r"\bcarregador\b", r"\bfone\b", r"\br[aá]dio\b", r"\baparelho eletr[oô]nico\b"
        ],
        "examples": [
            "apreensao de celular",
            "encontrado telefone na cela",
            "chip e carregador apreendidos"
        ]
    },
    {
        "macroclasse": "SEGURANCA",
        "classe": "APREENSAO_ILICITO",
        "subclasse": "DROGA_ENTORPECENTE",
        "grau_risco": 9,
        "criticidade": "CRITICA",
        "patterns": [
            r"\bdroga\b", r"\bentorpec", r"\bmaconha\b", r"\bcoca[ií]na\b",
            r"\bcrack\b", r"\bsubst[aâ]ncia\b", r"\btr[aá]fico\b"
        ],
        "examples": [
            "apreensao de droga",
            "entorpecente encontrado",
            "maconha apreendida"
        ]
    },
    {
        "macroclasse": "SEGURANCA",
        "classe": "APREENSAO_ILICITO",
        "subclasse": "ARMA_OBJETO_PERFUROCORTANTE",
        "grau_risco": 9,
        "criticidade": "CRITICA",
        "patterns": [
            r"\barma\b", r"\bfaca\b", r"\bl[aâ]mina\b", r"\bestoque\b",
            r"\bchucho\b", r"\bperfurocortante\b", r"\bobjeto cortante\b"
        ],
        "examples": [
            "arma artesanal apreendida",
            "faca encontrada na cela",
            "objeto perfurocortante"
        ]
    },
    {
        "macroclasse": "SEGURANCA",
        "classe": "CONTRABANDO",
        "subclasse": "ARREMESSO_OU_INGRESSO_ILICITO",
        "grau_risco": 8,
        "criticidade": "ALTA",
        "patterns": [
            r"\barremesso\b", r"\bcontrabando\b", r"\bingresso il[ií]cito\b",
            r"\bobjeto proibido\b", r"\bmaterial proibido\b"
        ],
        "examples": [
            "arremesso para interior da unidade",
            "entrada de material proibido",
            "contrabando apreendido"
        ]
    },
    {
        "macroclasse": "DISCIPLINA",
        "classe": "VIOLENCIA",
        "subclasse": "AGRESSAO_BRIGA",
        "grau_risco": 9,
        "criticidade": "CRITICA",
        "patterns": [
            r"\bagress[aã]o\b", r"\bagred", r"\bbriga\b", r"\bluta corporal\b",
            r"\bvias de fato\b", r"\bespanc"
        ],
        "examples": [
            "agressao entre internos",
            "briga na cela",
            "vias de fato"
        ]
    },
    {
        "macroclasse": "DISCIPLINA",
        "classe": "VIOLENCIA",
        "subclasse": "AMEACA_COACAO",
        "grau_risco": 8,
        "criticidade": "ALTA",
        "patterns": [
            r"\bamea[cç]a\b", r"\bamea[cç]ou\b", r"\bintimid", r"\bcoa[cç][aã]o\b"
        ],
        "examples": [
            "ameaca contra servidor",
            "coacao entre internos",
            "interno intimidou outro"
        ]
    },
    {
        "macroclasse": "DISCIPLINA",
        "classe": "INDISCIPLINA",
        "subclasse": "DESOBEDIENCIA_TUMULTO",
        "grau_risco": 7,
        "criticidade": "ALTA",
        "patterns": [
            r"\bindisciplina\b", r"\bdesobedi", r"\binsubordina", r"\bdesacato\b",
            r"\btumulto\b", r"\bdesordem\b", r"\bmotim\b", r"\brebeli[aã]o\b"
        ],
        "examples": [
            "ato de indisciplina",
            "desobediencia a ordem legal",
            "tumulto no pavilhao"
        ]
    },
    {
        "macroclasse": "PATRIMONIO",
        "classe": "DANO",
        "subclasse": "DANO_AO_PATRIMONIO",
        "grau_risco": 7,
        "criticidade": "ALTA",
        "patterns": [
            r"\bdano\b", r"\bdepreda", r"\bquebra\b", r"\bdestrui", r"\binc[eê]ndio\b"
        ],
        "examples": [
            "dano ao patrimonio publico",
            "depredacao da cela",
            "quebra de estrutura"
        ]
    },
    {
        "macroclasse": "SAUDE",
        "classe": "AUTOLESAO",
        "subclasse": "AUTOLESAO_OU_SUICIDIO",
        "grau_risco": 8,
        "criticidade": "ALTA",
        "patterns": [
            r"\bautoles", r"\bautomutil", r"\bsuic[ií]dio\b", r"\btentativa de suic[ií]dio\b",
            r"\benforc", r"\bauto exterm"
        ],
        "examples": [
            "autolesao em cela",
            "tentativa de suicidio",
            "interno se automutilou"
        ]
    },
    {
        "macroclasse": "VISITA",
        "classe": "VISITA_IRREGULAR",
        "subclasse": "IRREGULARIDADE_DE_VISITA",
        "grau_risco": 6,
        "criticidade": "MEDIA",
        "patterns": [
            r"\bvisit", r"\bvisita irregular\b", r"\bvisitante\b", r"\bentrada irregular\b"
        ],
        "examples": [
            "irregularidade em visita",
            "visitante com material nao autorizado",
            "entrada irregular de visitante"
        ]
    },
    {
        "macroclasse": "SERVIDOR",
        "classe": "CONDUTA_FUNCIONAL",
        "subclasse": "SERVIDOR_ENVOLVIDO",
        "grau_risco": 8,
        "criticidade": "ALTA",
        "patterns": [
            r"\bservidor\b", r"\bpolicial penal\b", r"\bagente penitenci[aá]rio\b",
            r"\bfuncion[aá]rio\b", r"\bconduta funcional\b"
        ],
        "examples": [
            "servidor envolvido na ocorrencia",
            "conduta irregular de servidor",
            "policial penal citado"
        ]
    },
    {
        "macroclasse": "OPERACIONAL",
        "classe": "REVISTA_FISCALIZACAO",
        "subclasse": "REVISTA_OU_INSPECAO",
        "grau_risco": 4,
        "criticidade": "MEDIA",
        "patterns": [
            r"\brevista\b", r"\binspe[cç][aã]o\b", r"\bfiscaliza", r"\bvarredura\b", r"\bbusca\b"
        ],
        "examples": [
            "revista de cela",
            "inspecao de rotina",
            "busca em pavilhao"
        ]
    },
    {
        "macroclasse": "OPERACIONAL",
        "classe": "PROCEDIMENTO_ADMINISTRATIVO",
        "subclasse": "REGISTRO_OPERACIONAL",
        "grau_risco": 2,
        "criticidade": "BAIXA",
        "patterns": [
            r"\bprocedimento\b", r"\bregistro\b", r"\bcomunica[cç][aã]o\b",
            r"\bapoio\b", r"\bacompanhamento\b", r"\borienta[cç][aã]o\b"
        ],
        "examples": [
            "registro operacional",
            "procedimento administrativo",
            "apoio a atividade"
        ]
    },
    {
        "macroclasse": "OUTROS",
        "classe": "OUTROS",
        "subclasse": "NAO_CLASSIFICADO",
        "grau_risco": 1,
        "criticidade": "BAIXA",
        "patterns": [],
        "examples": [
            "outros",
            "nao classificado",
            "diversos"
        ]
    }
]


# ============================================================
# NORMALIZACAO
# ============================================================

MAPA_SUBSTITUICAO = {
    "cel.": "celular",
    "cel ": "celular ",
    "tel ": "telefone ",
    "apreensao": "apreensao",
    "entorpec.": "entorpecente",
    "entorpec ": "entorpecente ",
    "subst ": "substancia ",
    "obj ": "objeto ",
    "perfuro cortante": "perfurocortante",
    "pol penal": "policial penal",
    "ag penitenciario": "agente penitenciario",
    "ag penitenciária": "agente penitenciario",
    "ag penit": "agente penitenciario",
    "evasao": "evasao",
    "rebelião": "rebelião",
    "desob.": "desobediencia",
    "auto lesao": "autolesao",
    "auto-exterm": "auto exterm",
}

def remover_acentos(txt):
    if txt is None:
        return ""
    return "".join(
        c for c in unicodedata.normalize("NFKD", str(txt))
        if not unicodedata.combining(c)
    )

def normalizar_texto(txt):
    txt = remover_acentos(txt).lower().strip()
    txt = re.sub(r"[\r\n\t]+", " ", txt)
    txt = re.sub(r"[/_]+", " ", txt)
    txt = re.sub(r"[^a-z0-9\s\-]", " ", txt)
    txt = re.sub(r"\s+", " ", txt).strip()

    for k, v in MAPA_SUBSTITUICAO.items():
        txt = txt.replace(k, v)

    txt = re.sub(r"\s+", " ", txt).strip()
    return txt

def gerar_id_texto(motivo, registro):
    base = f"{motivo or ''}|{registro or ''}"
    return hashlib.md5(base.encode("utf-8")).hexdigest()

def criticidade_num(txt):
    mapa = {"BAIXA": 1, "MEDIA": 2, "ALTA": 3, "CRITICA": 4}
    return mapa.get(txt, 0)


# ============================================================
# CONSTRUIR BASE DE REFERENCIA SEMANTICA
# ============================================================

refs = []
for item in TAXONOMIA:
    for ex in item["examples"]:
        refs.append({
            "macroclasse": item["macroclasse"],
            "classe": item["classe"],
            "subclasse": item["subclasse"],
            "grau_risco": item["grau_risco"],
            "criticidade": item["criticidade"],
            "texto_referencia": normalizar_texto(ex),
            "patterns": item["patterns"]
        })

if SKLEARN_OK:
    corpus_ref = [r["texto_referencia"] for r in refs]
    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1)
    ref_matrix = vectorizer.fit_transform(corpus_ref)
else:
    vectorizer = None
    ref_matrix = None


# ============================================================
# CLASSIFICADOR HIBRIDO
# ============================================================

def classificar_por_regra(texto_norm):
    melhor = None
    melhor_score = -1
    sinais = []

    for item in TAXONOMIA:
        if item["classe"] == "OUTROS":
            continue

        hits = []
        for p in item["patterns"]:
            if re.search(p, texto_norm):
                hits.append(p)

        score = len(hits)

        if score > 0:
            score_ajustado = score * 10 + item["grau_risco"] + criticidade_num(item["criticidade"])
            if score_ajustado > melhor_score:
                melhor_score = score_ajustado
                melhor = item
                sinais = hits

    if melhor is None:
        return None

    confianca = min(99, 70 + len(sinais) * 8 + melhor["grau_risco"])
    return {
        "macroclasse": melhor["macroclasse"],
        "classe": melhor["classe"],
        "subclasse": melhor["subclasse"],
        "grau_risco": melhor["grau_risco"],
        "criticidade": melhor["criticidade"],
        "confianca_classificacao": float(confianca),
        "origem_classificacao": "REGRA",
        "justificativa_classificacao": f"match_regra:{', '.join(sinais[:8])}",
        "sinais_detectados": json.dumps(sinais[:20], ensure_ascii=False)
    }

def classificar_por_similaridade(texto_norm):
    if not texto_norm:
        return None

    if SKLEARN_OK:
        vec = vectorizer.transform([texto_norm])
        sims = cosine_similarity(vec, ref_matrix)[0]
        idx = int(sims.argmax())
        score = float(sims[idx])
        ref = refs[idx]

        if score < 0.33:
            return None

        confianca = round(score * 100, 2)
        return {
            "macroclasse": ref["macroclasse"],
            "classe": ref["classe"],
            "subclasse": ref["subclasse"],
            "grau_risco": ref["grau_risco"],
            "criticidade": ref["criticidade"],
            "confianca_classificacao": confianca,
            "origem_classificacao": "SIMILARIDADE",
            "justificativa_classificacao": f"match_semantico:{ref['texto_referencia']}",
            "sinais_detectados": json.dumps([ref["texto_referencia"]], ensure_ascii=False)
        }

    melhor = None
    melhor_score = -1.0

    for ref in refs:
        score = SequenceMatcher(None, texto_norm, ref["texto_referencia"]).ratio()
        if score > melhor_score:
            melhor_score = score
            melhor = ref

    if melhor is None or melhor_score < 0.45:
        return None

    return {
        "macroclasse": melhor["macroclasse"],
        "classe": melhor["classe"],
        "subclasse": melhor["subclasse"],
        "grau_risco": melhor["grau_risco"],
        "criticidade": melhor["criticidade"],
        "confianca_classificacao": round(melhor_score * 100, 2),
        "origem_classificacao": "SIMILARIDADE",
        "justificativa_classificacao": f"match_aproximado:{melhor['texto_referencia']}",
        "sinais_detectados": json.dumps([melhor["texto_referencia"]], ensure_ascii=False)
    }

def classificar_texto(motivo_original, registro_original):
    motivo_original = motivo_original or ""
    registro_original = registro_original or ""

    motivo_norm = normalizar_texto(motivo_original)
    registro_norm = normalizar_texto(registro_original)
    texto_norm = normalizar_texto(f"{motivo_original} | {registro_original}")

    # prioridade alta para regra
    r = classificar_por_regra(texto_norm)
    if r is not None:
        return {
            "motivo_normalizado": motivo_norm,
            "registro_normalizado": registro_norm,
            "texto_classificacao_normalizado": texto_norm,
            **r
        }

    # fallback semantico
    s = classificar_por_similaridade(texto_norm)
    if s is not None:
        return {
            "motivo_normalizado": motivo_norm,
            "registro_normalizado": registro_norm,
            "texto_classificacao_normalizado": texto_norm,
            **s
        }

    return {
        "motivo_normalizado": motivo_norm,
        "registro_normalizado": registro_norm,
        "texto_classificacao_normalizado": texto_norm,
        "macroclasse": "OUTROS",
        "classe": "OUTROS",
        "subclasse": "NAO_CLASSIFICADO",
        "grau_risco": 1,
        "criticidade": "BAIXA",
        "confianca_classificacao": 15.0,
        "origem_classificacao": "FALLBACK",
        "justificativa_classificacao": "sem_match_regra_ou_semantico",
        "sinais_detectados": json.dumps([], ensure_ascii=False)
    }


# ============================================================
# BASE DE TEXTOS DISTINTOS PARA CLASSIFICAR
# ============================================================

df_texto_base = spark.sql("""
    select distinct
        md5(concat_ws('|', coalesce(motivo, ''), coalesce(registro, ''))) as id_texto_classificacao,
        coalesce(motivo, '') as motivo_original,
        coalesce(registro, '') as registro_original
    from gold.sinp_fat_ocorrencia_livro
""")

tabela = "tmp_ocorrencia_livro_texto_base"

df_texto_base.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_texto_base, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_ocorrencia_livro_texto_base")


# ============================================================
# IDENTIFICAR SOMENTE TEXTOS NOVOS
# ============================================================

tem_dim_anterior = spark.sql("show tables in gold like 'sinp_dim_classificacao_ocorrencia_livro'").count() > 0

if tem_dim_anterior:
    spark.sql("refresh table gold.sinp_dim_classificacao_ocorrencia_livro")

    df_texto_novo = spark.sql("""
        select
            b.id_texto_classificacao,
            b.motivo_original,
            b.registro_original
        from gold.tmp_ocorrencia_livro_texto_base b
        left join gold.sinp_dim_classificacao_ocorrencia_livro d
            on b.id_texto_classificacao = d.id_texto_classificacao
        where d.id_texto_classificacao is null
    """)
else:
    df_texto_novo = spark.sql("""
        select
            id_texto_classificacao,
            motivo_original,
            registro_original
        from gold.tmp_ocorrencia_livro_texto_base
    """)

tabela = "tmp_ocorrencia_livro_texto_novo"

df_texto_novo.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_texto_novo, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_ocorrencia_livro_texto_novo")


# ============================================================
# CLASSIFICAR EM PYTHON APENAS OS NOVOS
# ============================================================

schema_classificacao = T.StructType([
    T.StructField("id_texto_classificacao", T.StringType(), False),
    T.StructField("motivo_original", T.StringType(), True),
    T.StructField("registro_original", T.StringType(), True),
    T.StructField("motivo_normalizado", T.StringType(), True),
    T.StructField("registro_normalizado", T.StringType(), True),
    T.StructField("texto_classificacao_normalizado", T.StringType(), True),
    T.StructField("macroclasse", T.StringType(), True),
    T.StructField("classe", T.StringType(), True),
    T.StructField("subclasse", T.StringType(), True),
    T.StructField("grau_risco", T.IntegerType(), True),
    T.StructField("criticidade", T.StringType(), True),
    T.StructField("confianca_classificacao", T.DoubleType(), True),
    T.StructField("origem_classificacao", T.StringType(), True),
    T.StructField("versao_modelo", T.StringType(), True),
    T.StructField("justificativa_classificacao", T.StringType(), True),
    T.StructField("sinais_detectados", T.StringType(), True),
    T.StructField("dt_classificacao", T.TimestampType(), True),
])

novos = df_texto_novo.collect()

registros_classificados = []
versao_modelo = "CLASSIF_LIVRO_V1"

for row in novos:
    motivo_original = row["motivo_original"]
    registro_original = row["registro_original"]

    c = classificar_texto(motivo_original, registro_original)

    registros_classificados.append({
        "id_texto_classificacao": row["id_texto_classificacao"],
        "motivo_original": motivo_original,
        "registro_original": registro_original,
        "motivo_normalizado": c["motivo_normalizado"],
        "registro_normalizado": c["registro_normalizado"],
        "texto_classificacao_normalizado": c["texto_classificacao_normalizado"],
        "macroclasse": c["macroclasse"],
        "classe": c["classe"],
        "subclasse": c["subclasse"],
        "grau_risco": int(c["grau_risco"]),
        "criticidade": c["criticidade"],
        "confianca_classificacao": float(c["confianca_classificacao"]),
        "origem_classificacao": c["origem_classificacao"],
        "versao_modelo": versao_modelo,
        "justificativa_classificacao": c["justificativa_classificacao"],
        "sinais_detectados": c["sinais_detectados"],
        "dt_classificacao": datetime.now()
    })

if len(registros_classificados) > 0:
    df_classificacao_nova = spark.createDataFrame(pd.DataFrame(registros_classificados), schema=schema_classificacao)
else:
    df_classificacao_nova = spark.createDataFrame([], schema=schema_classificacao)

tabela = "tmp_dim_classificacao_ocorrencia_livro_novo"

df_classificacao_nova.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_classificacao_nova, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_dim_classificacao_ocorrencia_livro_novo")


# ============================================================
# DIM FINAL DE CLASSIFICACAO
# ============================================================

if tem_dim_anterior:
    df_dim_final = spark.sql("""
        select * from gold.sinp_dim_classificacao_ocorrencia_livro
        union all
        select * from gold.tmp_dim_classificacao_ocorrencia_livro_novo
    """)
else:
    df_dim_final = spark.sql("""
        select * from gold.tmp_dim_classificacao_ocorrencia_livro_novo
    """)

tabela = "tmp_dim_classificacao_ocorrencia_livro_final"

df_dim_final.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_dim_final, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_dim_classificacao_ocorrencia_livro_final")


df_dim_persistida = spark.sql("""
    select distinct
        id_texto_classificacao,
        motivo_original,
        registro_original,
        motivo_normalizado,
        registro_normalizado,
        texto_classificacao_normalizado,
        macroclasse,
        classe,
        subclasse,
        grau_risco,
        criticidade,
        confianca_classificacao,
        origem_classificacao,
        versao_modelo,
        justificativa_classificacao,
        sinais_detectados,
        dt_classificacao
    from gold.tmp_dim_classificacao_ocorrencia_livro_final
""")

tabela = "sinp_dim_classificacao_ocorrencia_livro"

df_dim_persistida.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_dim_persistida, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres(f"gold.{tabela}", "id_texto_classificacao")


# ============================================================
# ENRIQUECER FATO DO LIVRO
# ============================================================

df_fat_ocorrencia_livro_classificada = spark.sql("""
    select
        f.*,

        md5(concat_ws('|', coalesce(f.motivo, ''), coalesce(f.registro, ''))) as id_texto_classificacao,

        d.motivo_normalizado,
        d.registro_normalizado,
        d.texto_classificacao_normalizado,

        d.macroclasse as macroclasse_motivo,
        d.classe as classe_motivo,
        d.subclasse as subclasse_motivo,
        d.grau_risco as grau_risco_motivo,
        d.criticidade as criticidade_motivo,
        d.confianca_classificacao,
        d.origem_classificacao,
        d.versao_modelo,
        d.justificativa_classificacao,
        d.sinais_detectados,
        d.dt_classificacao,

        case
            when d.grau_risco >= 8 then 1
            else 0
        end as flag_motivo_critico,

        case
            when d.classe in ('APREENSAO_ILICITO', 'CONTRABANDO') then 1
            else 0
        end as flag_motivo_ilicito,

        case
            when d.classe in ('VIOLENCIA') then 1
            else 0
        end as flag_motivo_violencia,

        case
            when d.classe in ('FUGA_EVASAO') then 1
            else 0
        end as flag_motivo_fuga,

        case
            when d.classe in ('VISITA_IRREGULAR') then 1
            else 0
        end as flag_motivo_visita,

        case
            when d.classe in ('CONDUTA_FUNCIONAL') then 1
            else 0
        end as flag_motivo_servidor,

        coalesce(f.score_complexidade_basica, 0) + coalesce(d.grau_risco, 0) as score_risco_ocorrencia_livro
    from gold.sinp_fat_ocorrencia_livro f
    left join gold.sinp_dim_classificacao_ocorrencia_livro d
        on md5(concat_ws('|', coalesce(f.motivo, ''), coalesce(f.registro, ''))) = d.id_texto_classificacao
""")

tabela = "sinp_fat_ocorrencia_livro_classificada"

df_fat_ocorrencia_livro_classificada.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_fat_ocorrencia_livro_classificada, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres(f"gold.{tabela}", "id_fato_ocorrencia")

In [ ]:
import os

# ============================================================
# VALIDACAO DE ENTRADA
# ============================================================

if spark.sql("show tables in gold like 'sinp_fat_ocorrencia_livro_classificada'").count() == 0:
    raise Exception("Tabela gold.sinp_fat_ocorrencia_livro_classificada não encontrada. Gere a classificação semântica antes desta etapa.")

# ============================================================
# LIMPEZA DEFENSIVA
# ============================================================

tabelas_drop = [
    "tmp_sinal_ocorrencia_livro_metricas",
    "tmp_sinal_ocorrencia_livro_cenarios_base",
    "tmp_sinal_ocorrencia_livro_flags",
    "tmp_sinal_ocorrencia_livro_cenarios_final",
    "tmp_rl_ocorrencia_livro_cenario",
    "sinp_fat_ocorrencia_livro_risco",
    "sinp_rl_ocorrencia_livro_cenario"
]

for t in tabelas_drop:
    spark.sql(f"drop table if exists gold.{t}")
    os.system(f"hdfs dfs -rm -r -skipTrash {path}{t} >/dev/null 2>&1")

spark.catalog.clearCache()
spark.sql("refresh table gold.sinp_fat_ocorrencia_livro_classificada")


# ============================================================
# METRICAS BASE DE RISCO
# ============================================================

df_sinal_ocorrencia_livro_metricas = spark.sql("""
    select
        f.*,

        coalesce(qtd_advogados_raw, 0) +
        coalesce(qtd_ass_religiosas_raw, 0) +
        coalesce(qtd_policiais_raw, 0) +
        coalesce(qtd_servidores_raw, 0) +
        coalesce(qtd_visitantes_raw, 0) as qtd_envolvidos_externos_raw,

        coalesce(qtd_internos_raw, 0) +
        coalesce(qtd_presos_distintos_livro, 0) as qtd_envolvidos_internos_total,

        coalesce(qtd_internos_raw, 0) +
        coalesce(qtd_advogados_raw, 0) +
        coalesce(qtd_ass_religiosas_raw, 0) +
        coalesce(qtd_policiais_raw, 0) +
        coalesce(qtd_servidores_raw, 0) +
        coalesce(qtd_visitantes_raw, 0) +
        coalesce(qtd_presos_distintos_livro, 0) as qtd_envolvidos_total,

        coalesce(qtd_hist_ocorrencia_livro, 0) +
        coalesce(qtd_hist_registrovinculo_livro, 0) +
        coalesce(qtd_hist_vinculacao_livro, 0) as qtd_historico_total,

        length(coalesce(motivo, '')) as tam_motivo,
        length(coalesce(registro, '')) as tam_registro,
        length(concat_ws(' ', coalesce(motivo, ''), coalesce(registro, ''))) as tam_texto_total,

        case
            when length(concat_ws(' ', coalesce(motivo, ''), coalesce(registro, ''))) >= 250 then 1
            else 0
        end as flag_texto_denso,

        case
            when coalesce(qtd_internos_raw, 0) +
                 coalesce(qtd_advogados_raw, 0) +
                 coalesce(qtd_ass_religiosas_raw, 0) +
                 coalesce(qtd_policiais_raw, 0) +
                 coalesce(qtd_servidores_raw, 0) +
                 coalesce(qtd_visitantes_raw, 0) +
                 coalesce(qtd_presos_distintos_livro, 0) >= 6 then 1
            else 0
        end as flag_muitos_envolvidos,

        case
            when coalesce(qtd_hist_ocorrencia_livro, 0) +
                 coalesce(qtd_hist_registrovinculo_livro, 0) +
                 coalesce(qtd_hist_vinculacao_livro, 0) >= 5 then 1
            else 0
        end as flag_historico_intenso

    from gold.sinp_fat_ocorrencia_livro_classificada f
""")

tabela = "tmp_sinal_ocorrencia_livro_metricas"

df_sinal_ocorrencia_livro_metricas.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_sinal_ocorrencia_livro_metricas, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_sinal_ocorrencia_livro_metricas")


# ============================================================
# SCORE DOS CENARIOS
# ============================================================

df_sinal_ocorrencia_livro_cenarios_base = spark.sql("""
    select
        m.*,

        case
            when coalesce(flag_motivo_fuga, 0) = 1 then
                coalesce(grau_risco_motivo, 0)
                + case when coalesce(flag_multiplos_presos_resolvidos, 0) = 1 then 2 else 0 end
                + case when coalesce(flag_historico_intenso, 0) = 1 then 2 else 0 end
            else 0
        end as score_livro001_fuga,

        case
            when coalesce(flag_motivo_ilicito, 0) = 1
             and coalesce(qtd_envolvidos_externos_raw, 0) > 0 then
                coalesce(grau_risco_motivo, 0)
                + least(coalesce(qtd_envolvidos_externos_raw, 0), 5)
                + case when coalesce(qtd_visitantes_raw, 0) > 0 then 2 else 0 end
                + case when coalesce(qtd_advogados_raw, 0) > 0 then 2 else 0 end
            else 0
        end as score_livro002_ilicito_rede,

        case
            when coalesce(flag_motivo_violencia, 0) = 1 then
                coalesce(grau_risco_motivo, 0)
                + case when coalesce(qtd_historico_total, 0) >= 3 then 2 else 0 end
                + case when coalesce(qtd_presos_distintos_livro, 0) > 1 then 2 else 0 end
                + case when coalesce(qtd_envolvidos_total, 0) >= 4 then 1 else 0 end
            else 0
        end as score_livro003_violencia,

        case
            when coalesce(flag_motivo_servidor, 0) = 1
              or coalesce(qtd_servidores_raw, 0) > 0 then
                greatest(coalesce(grau_risco_motivo, 0), 6)
                + case when coalesce(qtd_servidores_raw, 0) > 0 then 2 else 0 end
                + case when coalesce(qtd_historico_total, 0) >= 2 then 1 else 0 end
            else 0
        end as score_livro004_servidor,

        case
            when coalesce(flag_motivo_visita, 0) = 1
              or (coalesce(qtd_visitantes_raw, 0) > 0 and coalesce(flag_motivo_ilicito, 0) = 1) then
                greatest(coalesce(grau_risco_motivo, 0), 5)
                + case when coalesce(qtd_visitantes_raw, 0) > 0 then 2 else 0 end
                + case when coalesce(flag_motivo_ilicito, 0) = 1 then 2 else 0 end
            else 0
        end as score_livro005_visita,

        case
            when coalesce(flag_muitos_envolvidos, 0) = 1
              or (coalesce(qtd_presos_distintos_livro, 0) > 1 and coalesce(qtd_envolvidos_externos_raw, 0) >= 2) then
                5
                + least(coalesce(qtd_envolvidos_total, 0), 6)
                + case when coalesce(qtd_presos_distintos_livro, 0) > 1 then 2 else 0 end
                + case when coalesce(qtd_envolvidos_externos_raw, 0) >= 2 then 2 else 0 end
            else 0
        end as score_livro006_complexidade,

        case
            when coalesce(flag_historico_intenso, 0) = 1 then
                4
                + least(coalesce(qtd_historico_total, 0), 6)
                + case when coalesce(score_risco_ocorrencia_livro, 0) >= 8 then 2 else 0 end
            else 0
        end as score_livro007_persistencia

    from gold.tmp_sinal_ocorrencia_livro_metricas m
""")

tabela = "tmp_sinal_ocorrencia_livro_cenarios_base"

df_sinal_ocorrencia_livro_cenarios_base.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_sinal_ocorrencia_livro_cenarios_base, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_sinal_ocorrencia_livro_cenarios_base")


# ============================================================
# FLAGS DOS CENARIOS
# ============================================================

df_sinal_ocorrencia_livro_flags = spark.sql("""
    select
        b.*,

        case when coalesce(score_livro001_fuga, 0) > 0 then 1 else 0 end as flag_livro001_fuga,
        case when coalesce(score_livro002_ilicito_rede, 0) > 0 then 1 else 0 end as flag_livro002_ilicito_rede,
        case when coalesce(score_livro003_violencia, 0) > 0 then 1 else 0 end as flag_livro003_violencia,
        case when coalesce(score_livro004_servidor, 0) > 0 then 1 else 0 end as flag_livro004_servidor,
        case when coalesce(score_livro005_visita, 0) > 0 then 1 else 0 end as flag_livro005_visita,
        case when coalesce(score_livro006_complexidade, 0) > 0 then 1 else 0 end as flag_livro006_complexidade,
        case when coalesce(score_livro007_persistencia, 0) > 0 then 1 else 0 end as flag_livro007_persistencia

    from gold.tmp_sinal_ocorrencia_livro_cenarios_base b
""")

tabela = "tmp_sinal_ocorrencia_livro_flags"

df_sinal_ocorrencia_livro_flags.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_sinal_ocorrencia_livro_flags, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_sinal_ocorrencia_livro_flags")


# ============================================================
# TABELA WIDE DE RISCO
# ============================================================

df_sinal_ocorrencia_livro_cenarios_final = spark.sql("""
    select
        f.*,

        greatest(
            coalesce(score_livro001_fuga, 0),
            coalesce(score_livro002_ilicito_rede, 0),
            coalesce(score_livro003_violencia, 0),
            coalesce(score_livro004_servidor, 0),
            coalesce(score_livro005_visita, 0),
            coalesce(score_livro006_complexidade, 0),
            coalesce(score_livro007_persistencia, 0)
        ) as score_cenario_maximo,

        (
            coalesce(score_livro001_fuga, 0) +
            coalesce(score_livro002_ilicito_rede, 0) +
            coalesce(score_livro003_violencia, 0) +
            coalesce(score_livro004_servidor, 0) +
            coalesce(score_livro005_visita, 0) +
            coalesce(score_livro006_complexidade, 0) +
            coalesce(score_livro007_persistencia, 0)
        ) as score_cenario_total,

        (
            coalesce(flag_livro001_fuga, 0) +
            coalesce(flag_livro002_ilicito_rede, 0) +
            coalesce(flag_livro003_violencia, 0) +
            coalesce(flag_livro004_servidor, 0) +
            coalesce(flag_livro005_visita, 0) +
            coalesce(flag_livro006_complexidade, 0) +
            coalesce(flag_livro007_persistencia, 0)
        ) as qtd_cenarios_disparados

    from gold.tmp_sinal_ocorrencia_livro_flags f
""")

tabela = "tmp_sinal_ocorrencia_livro_cenarios_final"

df_sinal_ocorrencia_livro_cenarios_final.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_sinal_ocorrencia_livro_cenarios_final, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_sinal_ocorrencia_livro_cenarios_final")


# ============================================================
# FATO DE RISCO DA OCORRENCIA LIVRO
# ============================================================

df_fat_ocorrencia_livro_risco = spark.sql("""
    select
        id_fato_ocorrencia,
        origem_sistema,
        id_ocorrencia_origem,
        id_presidio_origem,
        id_equipe_origem,
        dt_evento_referencia,
        dt_registro,

        motivo,
        registro,
        arquivo,

        macroclasse_motivo,
        classe_motivo,
        subclasse_motivo,
        grau_risco_motivo,
        criticidade_motivo,
        confianca_classificacao,
        origem_classificacao,
        versao_modelo,

        qtd_envolvidos_externos_raw,
        qtd_envolvidos_internos_total,
        qtd_envolvidos_total,
        qtd_historico_total,
        tam_motivo,
        tam_registro,
        tam_texto_total,
        flag_texto_denso,
        flag_muitos_envolvidos,
        flag_historico_intenso,

        flag_livro001_fuga,
        score_livro001_fuga,

        flag_livro002_ilicito_rede,
        score_livro002_ilicito_rede,

        flag_livro003_violencia,
        score_livro003_violencia,

        flag_livro004_servidor,
        score_livro004_servidor,

        flag_livro005_visita,
        score_livro005_visita,

        flag_livro006_complexidade,
        score_livro006_complexidade,

        flag_livro007_persistencia,
        score_livro007_persistencia,

        qtd_cenarios_disparados,
        score_cenario_maximo,
        score_cenario_total,

        case
            when score_cenario_maximo >= 15 then 'CRITICA'
            when score_cenario_maximo >= 10 then 'ALTA'
            when score_cenario_maximo >= 6 then 'MEDIA'
            when score_cenario_maximo > 0 then 'BAIXA'
            else 'SEM_SINAL'
        end as criticidade_cenario_maxima
    from gold.tmp_sinal_ocorrencia_livro_cenarios_final
""")

tabela = "sinp_fat_ocorrencia_livro_risco"

df_fat_ocorrencia_livro_risco.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_fat_ocorrencia_livro_risco, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres(f"gold.{tabela}", "id_fato_ocorrencia")


# ============================================================
# CENARIOS DISPARADOS - FORMATO LONG
# ============================================================

df_rl_ocorrencia_livro_cenario = spark.sql("""
    select
        concat('SCNLIVRO_', md5(concat_ws('|', id_fato_ocorrencia, 'LIVRO001'))) as id_cenario_ocorrencia_livro,
        id_fato_ocorrencia,
        id_ocorrencia_origem,
        id_presidio_origem,
        id_equipe_origem,
        dt_evento_referencia,
        'LIVRO001' as scenario_code,
        'FUGA_OU_TENTATIVA' as scenario_name,
        'Ocorrência classificada como fuga ou tentativa de fuga.' as scenario_description,
        score_livro001_fuga as score_cenario,
        case
            when score_livro001_fuga >= 15 then 'CRITICA'
            when score_livro001_fuga >= 10 then 'ALTA'
            when score_livro001_fuga >= 6 then 'MEDIA'
            else 'BAIXA'
        end as criticidade_cenario,
        concat(
            'classe=', coalesce(classe_motivo, ''),
            '; subclasse=', coalesce(subclasse_motivo, ''),
            '; presos=', cast(coalesce(qtd_presos_distintos_livro, 0) as string),
            '; historico=', cast(coalesce(qtd_historico_total, 0) as string)
        ) as reason
    from gold.tmp_sinal_ocorrencia_livro_cenarios_final
    where score_livro001_fuga > 0

    union all

    select
        concat('SCNLIVRO_', md5(concat_ws('|', id_fato_ocorrencia, 'LIVRO002'))),
        id_fato_ocorrencia,
        id_ocorrencia_origem,
        id_presidio_origem,
        id_equipe_origem,
        dt_evento_referencia,
        'LIVRO002',
        'ILICITO_COM_REDE_EXTERNA',
        'Ocorrência com indício de ilícito associado a envolvidos externos.',
        score_livro002_ilicito_rede,
        case
            when score_livro002_ilicito_rede >= 15 then 'CRITICA'
            when score_livro002_ilicito_rede >= 10 then 'ALTA'
            when score_livro002_ilicito_rede >= 6 then 'MEDIA'
            else 'BAIXA'
        end,
        concat(
            'classe=', coalesce(classe_motivo, ''),
            '; env_externos=', cast(coalesce(qtd_envolvidos_externos_raw, 0) as string),
            '; visitantes=', cast(coalesce(qtd_visitantes_raw, 0) as string),
            '; advogados=', cast(coalesce(qtd_advogados_raw, 0) as string)
        )
    from gold.tmp_sinal_ocorrencia_livro_cenarios_final
    where score_livro002_ilicito_rede > 0

    union all

    select
        concat('SCNLIVRO_', md5(concat_ws('|', id_fato_ocorrencia, 'LIVRO003'))),
        id_fato_ocorrencia,
        id_ocorrencia_origem,
        id_presidio_origem,
        id_equipe_origem,
        dt_evento_referencia,
        'LIVRO003',
        'VIOLENCIA_COM_ALTA_DINAMICA',
        'Ocorrência violenta com alta dinâmica operacional.',
        score_livro003_violencia,
        case
            when score_livro003_violencia >= 15 then 'CRITICA'
            when score_livro003_violencia >= 10 then 'ALTA'
            when score_livro003_violencia >= 6 then 'MEDIA'
            else 'BAIXA'
        end,
        concat(
            'classe=', coalesce(classe_motivo, ''),
            '; presos=', cast(coalesce(qtd_presos_distintos_livro, 0) as string),
            '; historico=', cast(coalesce(qtd_historico_total, 0) as string),
            '; envolvidos=', cast(coalesce(qtd_envolvidos_total, 0) as string)
        )
    from gold.tmp_sinal_ocorrencia_livro_cenarios_final
    where score_livro003_violencia > 0

    union all

    select
        concat('SCNLIVRO_', md5(concat_ws('|', id_fato_ocorrencia, 'LIVRO004'))),
        id_fato_ocorrencia,
        id_ocorrencia_origem,
        id_presidio_origem,
        id_equipe_origem,
        dt_evento_referencia,
        'LIVRO004',
        'CONDUTA_FUNCIONAL_SENSIVEL',
        'Ocorrência com participação ou sensibilidade funcional envolvendo servidor.',
        score_livro004_servidor,
        case
            when score_livro004_servidor >= 15 then 'CRITICA'
            when score_livro004_servidor >= 10 then 'ALTA'
            when score_livro004_servidor >= 6 then 'MEDIA'
            else 'BAIXA'
        end,
        concat(
            'classe=', coalesce(classe_motivo, ''),
            '; servidores=', cast(coalesce(qtd_servidores_raw, 0) as string),
            '; historico=', cast(coalesce(qtd_historico_total, 0) as string)
        )
    from gold.tmp_sinal_ocorrencia_livro_cenarios_final
    where score_livro004_servidor > 0

    union all

    select
        concat('SCNLIVRO_', md5(concat_ws('|', id_fato_ocorrencia, 'LIVRO005'))),
        id_fato_ocorrencia,
        id_ocorrencia_origem,
        id_presidio_origem,
        id_equipe_origem,
        dt_evento_referencia,
        'LIVRO005',
        'VISITA_IRREGULAR_OU_SENSIVEL',
        'Ocorrência ligada a visita irregular ou sensível.',
        score_livro005_visita,
        case
            when score_livro005_visita >= 15 then 'CRITICA'
            when score_livro005_visita >= 10 then 'ALTA'
            when score_livro005_visita >= 6 then 'MEDIA'
            else 'BAIXA'
        end,
        concat(
            'classe=', coalesce(classe_motivo, ''),
            '; visitantes=', cast(coalesce(qtd_visitantes_raw, 0) as string),
            '; ilicito=', cast(coalesce(flag_motivo_ilicito, 0) as string)
        )
    from gold.tmp_sinal_ocorrencia_livro_cenarios_final
    where score_livro005_visita > 0

    union all

    select
        concat('SCNLIVRO_', md5(concat_ws('|', id_fato_ocorrencia, 'LIVRO006'))),
        id_fato_ocorrencia,
        id_ocorrencia_origem,
        id_presidio_origem,
        id_equipe_origem,
        dt_evento_referencia,
        'LIVRO006',
        'OCORRENCIA_COMPLEXA_MULTIENVOLVIDOS',
        'Ocorrência com múltiplos envolvidos e elevada complexidade relacional.',
        score_livro006_complexidade,
        case
            when score_livro006_complexidade >= 15 then 'CRITICA'
            when score_livro006_complexidade >= 10 then 'ALTA'
            when score_livro006_complexidade >= 6 then 'MEDIA'
            else 'BAIXA'
        end,
        concat(
            'envolvidos_total=', cast(coalesce(qtd_envolvidos_total, 0) as string),
            '; externos=', cast(coalesce(qtd_envolvidos_externos_raw, 0) as string),
            '; presos=', cast(coalesce(qtd_presos_distintos_livro, 0) as string)
        )
    from gold.tmp_sinal_ocorrencia_livro_cenarios_final
    where score_livro006_complexidade > 0

    union all

    select
        concat('SCNLIVRO_', md5(concat_ws('|', id_fato_ocorrencia, 'LIVRO007'))),
        id_fato_ocorrencia,
        id_ocorrencia_origem,
        id_presidio_origem,
        id_equipe_origem,
        dt_evento_referencia,
        'LIVRO007',
        'PERSISTENCIA_HISTORICA_ELEVADA',
        'Ocorrência com persistência histórica elevada e dinâmica operacional contínua.',
        score_livro007_persistencia,
        case
            when score_livro007_persistencia >= 15 then 'CRITICA'
            when score_livro007_persistencia >= 10 then 'ALTA'
            when score_livro007_persistencia >= 6 then 'MEDIA'
            else 'BAIXA'
        end,
        concat(
            'historico=', cast(coalesce(qtd_historico_total, 0) as string),
            '; score_base=', cast(coalesce(score_risco_ocorrencia_livro, 0) as string)
        )
    from gold.tmp_sinal_ocorrencia_livro_cenarios_final
    where score_livro007_persistencia > 0
""")

tabela = "tmp_rl_ocorrencia_livro_cenario"

df_rl_ocorrencia_livro_cenario.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_rl_ocorrencia_livro_cenario, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_rl_ocorrencia_livro_cenario")


df_rl_ocorrencia_livro_cenario_final = spark.sql("""
    select
        id_cenario_ocorrencia_livro,
        id_fato_ocorrencia,
        id_ocorrencia_origem,
        id_presidio_origem,
        id_equipe_origem,
        dt_evento_referencia,
        scenario_code,
        scenario_name,
        scenario_description,
        score_cenario,
        criticidade_cenario,
        reason
    from gold.tmp_rl_ocorrencia_livro_cenario
""")

tabela = "sinp_rl_ocorrencia_livro_cenario"

df_rl_ocorrencia_livro_cenario_final.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_rl_ocorrencia_livro_cenario_final, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres(f"gold.{tabela}", "id_cenario_ocorrencia_livro")


# ============================================================
# VALIDACOES
# ============================================================

spark.sql("""
select
    count(*) as total_ocorrencias,
    sum(case when qtd_cenarios_disparados > 0 then 1 else 0 end) as ocorrencias_com_sinal,
    sum(case when criticidade_cenario_maxima = 'CRITICA' then 1 else 0 end) as ocorrencias_criticas,
    sum(case when criticidade_cenario_maxima = 'ALTA' then 1 else 0 end) as ocorrencias_altas
from gold.sinp_fat_ocorrencia_livro_risco
""").show(truncate=False)

spark.sql("""
select
    scenario_code,
    scenario_name,
    count(*) as qtd
from gold.sinp_rl_ocorrencia_livro_cenario
group by scenario_code, scenario_name
order by qtd desc, scenario_code
""").show(200, truncate=False)

spark.sql("""
select
    criticidade_cenario,
    count(*) as qtd
from gold.sinp_rl_ocorrencia_livro_cenario
group by criticidade_cenario
order by qtd desc
""").show(truncate=False)

spark.sql("""
select
    id_fato_ocorrencia,
    scenario_code,
    scenario_name,
    score_cenario,
    criticidade_cenario,
    reason
from gold.sinp_rl_ocorrencia_livro_cenario
order by score_cenario desc, scenario_code
""").show(200, truncate=False)

In [ ]:
import os

# ============================================================
# LIMPEZA DEFENSIVA
# ============================================================

tabelas_drop = [
    "tmp_base_pessoa_preso_ponte_movimentacao",
    "tmp_base_header_movimentacao_entrada",
    "tmp_base_header_movimentacao_saida",
    "tmp_base_movimentacao_header",
    "tmp_base_pessoa_movimentacao_entrada",
    "tmp_base_pessoa_movimentacao_saida",
    "tmp_base_pessoa_movimentacao_raw",
    "tmp_base_policial_movimentacao_entrada",
    "tmp_base_policial_movimentacao_saida",
    "tmp_base_policial_movimentacao_raw",
    "tmp_base_movimentacao_infopen_ctx",
    "tmp_base_rl_pessoa_movimentacao",
    "tmp_base_rl_policial_movimentacao",
    "tmp_base_agg_pessoa_movimentacao",
    "tmp_base_agg_policial_movimentacao",
    "tmp_base_agg_infopen_movimentacao",
    "tmp_base_veiculo_raw",
    "tmp_base_rl_movimentacao_veiculo",
    "sinp_ent_movimentacao",
    "sinp_ent_veiculo",
    "sinp_rl_pessoa_movimentacao",
    "sinp_rl_movimentacao_veiculo",
    "sinp_rl_policial_movimentacao"
]

for t in tabelas_drop:
    spark.sql(f"drop table if exists gold.{t}")
    os.system(f"hdfs dfs -rm -r -skipTrash {path}{t} >/dev/null 2>&1")

spark.catalog.clearCache()

spark.sql("refresh table bronze.livros_acesso_unidade_caracteristicaescolta")
spark.sql("refresh table bronze.livros_acesso_unidade_caracteristicaescoltasaida")
spark.sql("refresh table bronze.livros_acesso_unidade_internoescolta")
spark.sql("refresh table bronze.livros_acesso_unidade_internoescoltasaida")
spark.sql("refresh table bronze.livros_acesso_unidade_policialescolta")
spark.sql("refresh table bronze.livros_acesso_unidade_policialescoltasaida")
spark.sql("refresh table bronze.infopen_movimentacoes")
spark.sql("refresh table bronze.infopen_tipos_movimentacao")
spark.sql("refresh table gold.sinp_pnt_pessoa_preso")
spark.sql("refresh table gold.sinp_ent_pessoa")


# ============================================================
# PONTE PESSOA X PRESO
# ============================================================

df_base_pessoa_preso_ponte_movimentacao = spark.sql("""
    select distinct
        cast(id_preso as string) as id_preso_origem,
        id_pessoa as id_pessoa_presidiario
    from gold.sinp_pnt_pessoa_preso
    where id_preso is not null
      and id_pessoa is not null
""")

tabela = "tmp_base_pessoa_preso_ponte_movimentacao"

df_base_pessoa_preso_ponte_movimentacao.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_pessoa_preso_ponte_movimentacao, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_pessoa_preso_ponte_movimentacao")


# ============================================================
# HEADER MOVIMENTACAO - ESCOLTA ENTRADA
# ============================================================

df_base_header_movimentacao_entrada = spark.sql("""
    select
        cast(id as string) as id_evento_origem,
        'LIVROS' as origem_sistema,
        'ESCOLTA_ENTRADA' as tipo_movimentacao,
        trim(regexp_replace(coalesce(instituicao, ''), '\\\\s+', ' ')) as instituicao,
        trim(regexp_replace(coalesce(origem, ''), '\\\\s+', ' ')) as origem_local,
        cast(null as string) as destino_local,
        trim(regexp_replace(coalesce(motivo, ''), '\\\\s+', ' ')) as motivo,
        cast(null as string) as solicitacao,
        trim(regexp_replace(coalesce(autorizacao, ''), '\\\\s+', ' ')) as autorizacao,
        trim(regexp_replace(coalesce(documento, ''), '\\\\s+', ' ')) as documento,
        trim(regexp_replace(coalesce(viatura, ''), '\\\\s+', ' ')) as veiculo_raw,
        cast(hr_chegada as string) as hr_inicio,
        cast(null as string) as hr_fim,
        to_timestamp(data_registro) as dt_registro,
        cast(equipe_id as string) as id_equipe_origem,
        cast(presidio_id as string) as id_presidio_origem,
        1 as flag_escolta
    from bronze.livros_acesso_unidade_caracteristicaescolta
""")

tabela = "tmp_base_header_movimentacao_entrada"

df_base_header_movimentacao_entrada.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_header_movimentacao_entrada, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_header_movimentacao_entrada")


# ============================================================
# HEADER MOVIMENTACAO - ESCOLTA SAIDA
# ============================================================

df_base_header_movimentacao_saida = spark.sql("""
    select
        cast(id as string) as id_evento_origem,
        'LIVROS' as origem_sistema,
        'ESCOLTA_SAIDA' as tipo_movimentacao,
        trim(regexp_replace(coalesce(instituicao, ''), '\\\\s+', ' ')) as instituicao,
        cast(null as string) as origem_local,
        trim(regexp_replace(coalesce(destino, ''), '\\\\s+', ' ')) as destino_local,
        trim(regexp_replace(coalesce(motivo, ''), '\\\\s+', ' ')) as motivo,
        trim(regexp_replace(coalesce(solicitacao, ''), '\\\\s+', ' ')) as solicitacao,
        trim(regexp_replace(coalesce(autorizacao, ''), '\\\\s+', ' ')) as autorizacao,
        trim(regexp_replace(coalesce(documento, ''), '\\\\s+', ' ')) as documento,
        trim(regexp_replace(coalesce(viatura, ''), '\\\\s+', ' ')) as veiculo_raw,
        cast(hr_saida as string) as hr_inicio,
        cast(hr_retorno as string) as hr_fim,
        to_timestamp(data_registro) as dt_registro,
        cast(equipe_id as string) as id_equipe_origem,
        cast(presidio_id as string) as id_presidio_origem,
        1 as flag_escolta
    from bronze.livros_acesso_unidade_caracteristicaescoltasaida
""")

tabela = "tmp_base_header_movimentacao_saida"

df_base_header_movimentacao_saida.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_header_movimentacao_saida, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_header_movimentacao_saida")


# ============================================================
# HEADER PADRONIZADO DA MOVIMENTACAO
# ============================================================

df_base_movimentacao_header = spark.sql("""
    select
        concat(
            'MOV_',
            md5(
                concat_ws(
                    '|',
                    coalesce(origem_sistema, ''),
                    coalesce(tipo_movimentacao, ''),
                    coalesce(id_evento_origem, ''),
                    coalesce(id_presidio_origem, ''),
                    coalesce(cast(dt_registro as string), '')
                )
            )
        ) as id_movimentacao,
        id_evento_origem,
        origem_sistema,
        tipo_movimentacao,
        instituicao,
        origem_local,
        destino_local,
        motivo,
        solicitacao,
        autorizacao,
        documento,
        veiculo_raw,
        hr_inicio,
        hr_fim,
        dt_registro,
        id_equipe_origem,
        id_presidio_origem,
        flag_escolta
    from (
        select * from gold.tmp_base_header_movimentacao_entrada
        union all
        select * from gold.tmp_base_header_movimentacao_saida
    ) x
""")

tabela = "tmp_base_movimentacao_header"

df_base_movimentacao_header.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_movimentacao_header, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_movimentacao_header")


# ============================================================
# PESSOA/MOVIMENTACAO - ESCOLTA ENTRADA
# ============================================================

df_base_pessoa_movimentacao_entrada = spark.sql("""
    select
        'ESCOLTA_ENTRADA' as tipo_movimentacao,
        cast(escolta_id as string) as id_evento_origem,
        cast(id as string) as id_item_origem,
        cast(infopen as string) as id_preso_origem,
        trim(regexp_replace(coalesce(nome, ''), '\\\\s+', ' ')) as nome_presidiario_raw,
        to_timestamp(data_registro) as dt_registro,
        cast(equipe_id as string) as id_equipe_origem,
        cast(presidio_id as string) as id_presidio_origem
    from bronze.livros_acesso_unidade_internoescolta
""")

tabela = "tmp_base_pessoa_movimentacao_entrada"

df_base_pessoa_movimentacao_entrada.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_pessoa_movimentacao_entrada, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_pessoa_movimentacao_entrada")


# ============================================================
# PESSOA/MOVIMENTACAO - ESCOLTA SAIDA
# ============================================================

df_base_pessoa_movimentacao_saida = spark.sql("""
    select
        'ESCOLTA_SAIDA' as tipo_movimentacao,
        cast(escolta_id as string) as id_evento_origem,
        cast(id as string) as id_item_origem,
        cast(infopen as string) as id_preso_origem,
        trim(regexp_replace(coalesce(nome, ''), '\\\\s+', ' ')) as nome_presidiario_raw,
        to_timestamp(data_registro) as dt_registro,
        cast(equipe_id as string) as id_equipe_origem,
        cast(presidio_id as string) as id_presidio_origem
    from bronze.livros_acesso_unidade_internoescoltasaida
""")

tabela = "tmp_base_pessoa_movimentacao_saida"

df_base_pessoa_movimentacao_saida.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_pessoa_movimentacao_saida, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_pessoa_movimentacao_saida")


# ============================================================
# PESSOA/MOVIMENTACAO RAW PADRONIZADA
# ============================================================

df_base_pessoa_movimentacao_raw = spark.sql("""
    select
        h.id_movimentacao,
        p.tipo_movimentacao,
        p.id_evento_origem,
        p.id_item_origem,
        p.id_preso_origem,
        pp.id_pessoa_presidiario,
        p.nome_presidiario_raw,
        p.dt_registro,
        p.id_equipe_origem,
        p.id_presidio_origem
    from (
        select * from gold.tmp_base_pessoa_movimentacao_entrada
        union all
        select * from gold.tmp_base_pessoa_movimentacao_saida
    ) p
    inner join gold.tmp_base_movimentacao_header h
        on p.tipo_movimentacao = h.tipo_movimentacao
       and p.id_evento_origem = h.id_evento_origem
    left join gold.tmp_base_pessoa_preso_ponte_movimentacao pp
        on p.id_preso_origem = pp.id_preso_origem
""")

tabela = "tmp_base_pessoa_movimentacao_raw"

df_base_pessoa_movimentacao_raw.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_pessoa_movimentacao_raw, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_pessoa_movimentacao_raw")


# ============================================================
# POLICIAL/MOVIMENTACAO - ESCOLTA ENTRADA
# ============================================================

df_base_policial_movimentacao_entrada = spark.sql("""
    select
        'ESCOLTA_ENTRADA' as tipo_movimentacao,
        cast(escolta_id as string) as id_evento_origem,
        cast(id as string) as id_item_origem,
        trim(regexp_replace(coalesce(nome, ''), '\\\\s+', ' ')) as nome_policial_raw,
        trim(regexp_replace(coalesce(documento, ''), '\\\\s+', ' ')) as documento_policial_raw,
        0 as flag_condutor,
        to_timestamp(data_registro) as dt_registro,
        cast(equipe_id as string) as id_equipe_origem,
        cast(presidio_id as string) as id_presidio_origem
    from bronze.livros_acesso_unidade_policialescolta
""")

tabela = "tmp_base_policial_movimentacao_entrada"

df_base_policial_movimentacao_entrada.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_policial_movimentacao_entrada, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_policial_movimentacao_entrada")


# ============================================================
# POLICIAL/MOVIMENTACAO - ESCOLTA SAIDA
# ============================================================

df_base_policial_movimentacao_saida = spark.sql("""
    select
        'ESCOLTA_SAIDA' as tipo_movimentacao,
        cast(escolta_id as string) as id_evento_origem,
        cast(id as string) as id_item_origem,
        trim(regexp_replace(coalesce(nome, ''), '\\\\s+', ' ')) as nome_policial_raw,
        trim(regexp_replace(coalesce(documento, ''), '\\\\s+', ' ')) as documento_policial_raw,
        case
            when coalesce(condutor, false) = true then 1
            else 0
        end as flag_condutor,
        to_timestamp(data_registro) as dt_registro,
        cast(equipe_id as string) as id_equipe_origem,
        cast(presidio_id as string) as id_presidio_origem
    from bronze.livros_acesso_unidade_policialescoltasaida
""")

tabela = "tmp_base_policial_movimentacao_saida"

df_base_policial_movimentacao_saida.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_policial_movimentacao_saida, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_policial_movimentacao_saida")


# ============================================================
# POLICIAL/MOVIMENTACAO RAW PADRONIZADA
# ============================================================

df_base_policial_movimentacao_raw = spark.sql("""
    select
        h.id_movimentacao,
        p.tipo_movimentacao,
        p.id_evento_origem,
        p.id_item_origem,
        p.nome_policial_raw,
        p.documento_policial_raw,
        p.flag_condutor,
        p.dt_registro,
        p.id_equipe_origem,
        p.id_presidio_origem
    from (
        select * from gold.tmp_base_policial_movimentacao_entrada
        union all
        select * from gold.tmp_base_policial_movimentacao_saida
    ) p
    inner join gold.tmp_base_movimentacao_header h
        on p.tipo_movimentacao = h.tipo_movimentacao
       and p.id_evento_origem = h.id_evento_origem
""")

tabela = "tmp_base_policial_movimentacao_raw"

df_base_policial_movimentacao_raw.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_policial_movimentacao_raw, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_policial_movimentacao_raw")


# ============================================================
# CONTEXTO INFOPEN PARA A MOVIMENTACAO
# ============================================================

df_base_movimentacao_infopen_ctx = spark.sql("""
    select
        cast(m.id_movimentacao as string) as id_movimentacao_infopen_origem,
        cast(m.id_preso as string) as id_preso_origem,
        cast(m.id_tipomovimentacao as string) as id_tipomovimentacao_infopen,
        trim(regexp_replace(coalesce(t.tipomovimentacao_descricao, ''), '\\\\s+', ' ')) as ds_tipomovimentacao_infopen,
        to_timestamp(m.movimentacao_data) as dt_movimentacao_infopen,
        to_date(m.movimentacao_data) as dt_movimentacao_infopen_ref
    from bronze.infopen_movimentacoes m
    left join bronze.infopen_tipos_movimentacao t
        on m.id_tipomovimentacao = t.id_tipomovimentacao
""")

tabela = "tmp_base_movimentacao_infopen_ctx"

df_base_movimentacao_infopen_ctx.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_movimentacao_infopen_ctx, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_movimentacao_infopen_ctx")


# ============================================================
# RL PESSOA X MOVIMENTACAO
# ============================================================

df_base_rl_pessoa_movimentacao = spark.sql("""
    select
        concat(
            'RLPM_',
            md5(
                concat_ws(
                    '|',
                    coalesce(r.id_movimentacao, ''),
                    coalesce(r.id_preso_origem, ''),
                    coalesce(r.id_item_origem, '')
                )
            )
        ) as id_rl_pessoa_movimentacao,
        r.id_movimentacao,
        r.tipo_movimentacao,
        r.id_evento_origem,
        r.id_item_origem,
        r.id_preso_origem,
        r.id_pessoa_presidiario,
        r.nome_presidiario_raw,
        r.dt_registro,
        r.id_equipe_origem,
        r.id_presidio_origem,
        count(i.id_movimentacao_infopen_origem) as qtd_movimentacoes_infopen_relacionadas,
        concat_ws(
            ' | ',
            sort_array(collect_set(i.ds_tipomovimentacao_infopen))
        ) as txt_tipos_movimentacao_infopen_relacionadas,
        case
            when count(i.id_movimentacao_infopen_origem) > 0 then 1
            else 0
        end as flag_tem_movimentacao_infopen_mesmo_dia
    from gold.tmp_base_pessoa_movimentacao_raw r
    left join gold.tmp_base_movimentacao_infopen_ctx i
        on r.id_preso_origem = i.id_preso_origem
       and to_date(r.dt_registro) = i.dt_movimentacao_infopen_ref
    group by
        r.id_movimentacao,
        r.tipo_movimentacao,
        r.id_evento_origem,
        r.id_item_origem,
        r.id_preso_origem,
        r.id_pessoa_presidiario,
        r.nome_presidiario_raw,
        r.dt_registro,
        r.id_equipe_origem,
        r.id_presidio_origem
""")

tabela = "tmp_base_rl_pessoa_movimentacao"

df_base_rl_pessoa_movimentacao.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_rl_pessoa_movimentacao, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_rl_pessoa_movimentacao")


# ============================================================
# RL POLICIAL X MOVIMENTACAO
# ============================================================

df_base_rl_policial_movimentacao = spark.sql("""
    select
        concat(
            'RLPOLM_',
            md5(
                concat_ws(
                    '|',
                    coalesce(id_movimentacao, ''),
                    coalesce(id_item_origem, ''),
                    coalesce(nome_policial_raw, ''),
                    coalesce(documento_policial_raw, '')
                )
            )
        ) as id_rl_policial_movimentacao,
        id_movimentacao,
        tipo_movimentacao,
        id_evento_origem,
        id_item_origem,
        nome_policial_raw,
        documento_policial_raw,
        flag_condutor,
        dt_registro,
        id_equipe_origem,
        id_presidio_origem
    from gold.tmp_base_policial_movimentacao_raw
""")

tabela = "tmp_base_rl_policial_movimentacao"

df_base_rl_policial_movimentacao.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_rl_policial_movimentacao, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_rl_policial_movimentacao")


# ============================================================
# AGREGADOS PESSOA X MOVIMENTACAO
# ============================================================

df_base_agg_pessoa_movimentacao = spark.sql("""
    select
        id_movimentacao,
        count(*) as qtd_presidiarios_relacionados,
        count(distinct id_preso_origem) as qtd_presidiarios_distintos,
        sum(case when id_pessoa_presidiario is not null then 1 else 0 end) as qtd_presidiarios_resolvidos_pessoa,
        sum(case when flag_tem_movimentacao_infopen_mesmo_dia = 1 then 1 else 0 end) as qtd_presidiarios_com_movimentacao_infopen_mesmo_dia,
        sum(qtd_movimentacoes_infopen_relacionadas) as qtd_movimentacoes_infopen_relacionadas,
        concat_ws(
            ' | ',
            sort_array(collect_set(id_preso_origem))
        ) as txt_ids_preso_relacionados,
        concat_ws(
            ' | ',
            sort_array(collect_set(cast(id_pessoa_presidiario as string)))
        ) as txt_ids_pessoa_relacionados,
        concat_ws(
            ' | ',
            sort_array(collect_set(nome_presidiario_raw))
        ) as txt_nomes_presidiarios_relacionados,
        concat_ws(
            ' | ',
            sort_array(collect_set(txt_tipos_movimentacao_infopen_relacionadas))
        ) as txt_tipos_movimentacao_infopen_relacionadas
    from gold.tmp_base_rl_pessoa_movimentacao
    group by id_movimentacao
""")

tabela = "tmp_base_agg_pessoa_movimentacao"

df_base_agg_pessoa_movimentacao.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_agg_pessoa_movimentacao, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_agg_pessoa_movimentacao")


# ============================================================
# AGREGADOS POLICIAL X MOVIMENTACAO
# ============================================================

df_base_agg_policial_movimentacao = spark.sql("""
    select
        id_movimentacao,
        count(*) as qtd_policiais_relacionados,
        count(distinct documento_policial_raw) as qtd_policiais_distintos,
        sum(coalesce(flag_condutor, 0)) as qtd_condutores,
        concat_ws(
            ' | ',
            sort_array(collect_set(nome_policial_raw))
        ) as txt_nomes_policiais_relacionados
    from gold.tmp_base_rl_policial_movimentacao
    group by id_movimentacao
""")

tabela = "tmp_base_agg_policial_movimentacao"

df_base_agg_policial_movimentacao.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_agg_policial_movimentacao, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_agg_policial_movimentacao")


# ============================================================
# AGREGADO INFOPEN POR MOVIMENTACAO
# ============================================================

df_base_agg_infopen_movimentacao = spark.sql("""
    select
        id_movimentacao,
        case
            when coalesce(qtd_movimentacoes_infopen_relacionadas, 0) > 0 then 1
            else 0
        end as flag_tem_movimentacao_infopen_mesmo_dia
    from gold.tmp_base_agg_pessoa_movimentacao
""")

tabela = "tmp_base_agg_infopen_movimentacao"

df_base_agg_infopen_movimentacao.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_agg_infopen_movimentacao, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_agg_infopen_movimentacao")


# ============================================================
# VEICULO RAW - DEDUPLICADO CORRETAMENTE
# ============================================================

df_base_veiculo_raw = spark.sql("""
    select
        id_veiculo,
        ds_veiculo,
        ds_veiculo_normalizado
    from (
        select
            concat(
                'VEI_',
                md5(
                    concat_ws(
                        '|',
                        upper(trim(regexp_replace(coalesce(veiculo_raw, ''), '\\\\s+', ' ')))
                    )
                )
            ) as id_veiculo,

            trim(regexp_replace(coalesce(veiculo_raw, ''), '\\\\s+', ' ')) as ds_veiculo,

            upper(trim(regexp_replace(coalesce(veiculo_raw, ''), '\\\\s+', ' '))) as ds_veiculo_normalizado,

            row_number() over (
                partition by
                    concat(
                        'VEI_',
                        md5(
                            concat_ws(
                                '|',
                                upper(trim(regexp_replace(coalesce(veiculo_raw, ''), '\\\\s+', ' ')))
                            )
                        )
                    )
                order by
                    case
                        when trim(regexp_replace(coalesce(veiculo_raw, ''), '\\\\s+', ' ')) =
                             upper(trim(regexp_replace(coalesce(veiculo_raw, ''), '\\\\s+', ' ')))
                        then 1 else 2
                    end,
                    length(trim(regexp_replace(coalesce(veiculo_raw, ''), '\\\\s+', ' '))) desc,
                    trim(regexp_replace(coalesce(veiculo_raw, ''), '\\\\s+', ' ')) asc
            ) as rn
        from gold.tmp_base_movimentacao_header
        where trim(coalesce(veiculo_raw, '')) <> ''
    ) x
    where rn = 1
""")

tabela = "tmp_base_veiculo_raw"

df_base_veiculo_raw.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_veiculo_raw, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_veiculo_raw")


# ============================================================
# RL MOVIMENTACAO X VEICULO
# ============================================================

df_base_rl_movimentacao_veiculo = spark.sql("""
    select distinct
        concat(
            'RLMV_',
            md5(
                concat_ws(
                    '|',
                    coalesce(m.id_movimentacao, ''),
                    coalesce(v.id_veiculo, '')
                )
            )
        ) as id_rl_movimentacao_veiculo,
        m.id_movimentacao,
        v.id_veiculo,
        m.dt_registro,
        m.id_equipe_origem,
        m.id_presidio_origem
    from gold.tmp_base_movimentacao_header m
    inner join gold.tmp_base_veiculo_raw v
        on upper(trim(regexp_replace(coalesce(m.veiculo_raw, ''), '\\\\s+', ' '))) = v.ds_veiculo_normalizado
    where trim(coalesce(m.veiculo_raw, '')) <> ''
""")

tabela = "tmp_base_rl_movimentacao_veiculo"

df_base_rl_movimentacao_veiculo.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_base_rl_movimentacao_veiculo, "gold", tabela, f"{path}{tabela}")
spark.catalog.clearCache()
spark.sql("refresh table gold.tmp_base_rl_movimentacao_veiculo")


# ============================================================
# ENTIDADE VEICULO
# ============================================================

df_ent_veiculo = spark.sql("""
    select
        id_veiculo,
        ds_veiculo
    from gold.tmp_base_veiculo_raw
""")

tabela = "sinp_ent_veiculo"

df_ent_veiculo.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_ent_veiculo, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres(f"gold.{tabela}", "id_veiculo")


# ============================================================
# ENTIDADE MOVIMENTACAO
# ============================================================

df_ent_movimentacao = spark.sql("""
    select
        h.id_movimentacao,
        h.id_evento_origem,
        h.origem_sistema,
        h.tipo_movimentacao,
        h.instituicao,
        h.origem_local,
        h.destino_local,
        h.motivo,
        h.solicitacao,
        h.autorizacao,
        h.documento,
        h.veiculo_raw,
        h.hr_inicio,
        h.hr_fim,
        h.dt_registro,
        h.id_equipe_origem,
        h.id_presidio_origem,
        h.flag_escolta,

        case when trim(coalesce(h.veiculo_raw, '')) <> '' then 1 else 0 end as flag_tem_veiculo,
        case when trim(coalesce(h.documento, '')) <> '' then 1 else 0 end as flag_tem_documento,
        case when trim(coalesce(h.autorizacao, '')) <> '' then 1 else 0 end as flag_tem_autorizacao,

        coalesce(p.qtd_presidiarios_relacionados, 0) as qtd_presidiarios_relacionados,
        coalesce(p.qtd_presidiarios_distintos, 0) as qtd_presidiarios_distintos,
        coalesce(p.qtd_presidiarios_resolvidos_pessoa, 0) as qtd_presidiarios_resolvidos_pessoa,
        coalesce(p.qtd_presidiarios_com_movimentacao_infopen_mesmo_dia, 0) as qtd_presidiarios_com_movimentacao_infopen_mesmo_dia,
        coalesce(p.qtd_movimentacoes_infopen_relacionadas, 0) as qtd_movimentacoes_infopen_relacionadas,
        p.txt_ids_preso_relacionados,
        p.txt_ids_pessoa_relacionados,
        p.txt_nomes_presidiarios_relacionados,
        p.txt_tipos_movimentacao_infopen_relacionadas,

        coalesce(pol.qtd_policiais_relacionados, 0) as qtd_policiais_relacionados,
        coalesce(pol.qtd_policiais_distintos, 0) as qtd_policiais_distintos,
        coalesce(pol.qtd_condutores, 0) as qtd_condutores,
        pol.txt_nomes_policiais_relacionados,

        coalesce(i.flag_tem_movimentacao_infopen_mesmo_dia, 0) as flag_tem_movimentacao_infopen_mesmo_dia,

        case when coalesce(p.qtd_presidiarios_distintos, 0) > 1 then 1 else 0 end as flag_movimentacao_coletiva,
        case when coalesce(pol.qtd_policiais_distintos, 0) > 1 then 1 else 0 end as flag_multiplos_policiais
    from gold.tmp_base_movimentacao_header h
    left join gold.tmp_base_agg_pessoa_movimentacao p
        on h.id_movimentacao = p.id_movimentacao
    left join gold.tmp_base_agg_policial_movimentacao pol
        on h.id_movimentacao = pol.id_movimentacao
    left join gold.tmp_base_agg_infopen_movimentacao i
        on h.id_movimentacao = i.id_movimentacao
""")

tabela = "sinp_ent_movimentacao"

df_ent_movimentacao.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_ent_movimentacao, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres(f"gold.{tabela}", "id_movimentacao")


# ============================================================
# RL PESSOA X MOVIMENTACAO
# ============================================================

df_rl_pessoa_movimentacao = spark.sql("""
    select
        id_rl_pessoa_movimentacao,
        id_movimentacao,
        tipo_movimentacao,
        id_evento_origem,
        id_item_origem,
        id_preso_origem,
        id_pessoa_presidiario,
        nome_presidiario_raw,
        dt_registro,
        id_equipe_origem,
        id_presidio_origem,
        qtd_movimentacoes_infopen_relacionadas,
        txt_tipos_movimentacao_infopen_relacionadas,
        flag_tem_movimentacao_infopen_mesmo_dia
    from gold.tmp_base_rl_pessoa_movimentacao
""")

tabela = "sinp_rl_pessoa_movimentacao"

df_rl_pessoa_movimentacao.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_rl_pessoa_movimentacao, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres(f"gold.{tabela}", "id_rl_pessoa_movimentacao")


# ============================================================
# RL MOVIMENTACAO X VEICULO
# ============================================================

df_rl_movimentacao_veiculo = spark.sql("""
    select
        id_rl_movimentacao_veiculo,
        id_movimentacao,
        id_veiculo,
        dt_registro,
        id_equipe_origem,
        id_presidio_origem
    from gold.tmp_base_rl_movimentacao_veiculo
""")

tabela = "sinp_rl_movimentacao_veiculo"

df_rl_movimentacao_veiculo.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_rl_movimentacao_veiculo, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres(f"gold.{tabela}", "id_rl_movimentacao_veiculo")


# ============================================================
# RL POLICIAL X MOVIMENTACAO
# ============================================================

df_rl_policial_movimentacao = spark.sql("""
    select
        id_rl_policial_movimentacao,
        id_movimentacao,
        tipo_movimentacao,
        id_evento_origem,
        id_item_origem,
        nome_policial_raw,
        documento_policial_raw,
        flag_condutor,
        dt_registro,
        id_equipe_origem,
        id_presidio_origem
    from gold.tmp_base_rl_policial_movimentacao
""")

tabela = "sinp_rl_policial_movimentacao"

df_rl_policial_movimentacao.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_rl_policial_movimentacao, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres(f"gold.{tabela}", "id_rl_policial_movimentacao")


# ============================================================
# VALIDACOES
# ============================================================

spark.sql("""
select
    count(*) as qtd_veiculos,
    count(distinct id_veiculo) as qtd_veiculos_distintos
from gold.sinp_ent_veiculo
""").show(truncate=False)

spark.sql("""
select
    id_veiculo,
    count(*) as qtd
from gold.sinp_ent_veiculo
group by id_veiculo
having count(*) > 1
order by qtd desc
""").show(50, truncate=False)

spark.sql("""
select
    tipo_movimentacao,
    count(*) as qtd_movimentacoes,
    sum(qtd_presidiarios_distintos) as soma_presidiarios,
    sum(qtd_policiais_distintos) as soma_policiais,
    sum(flag_movimentacao_coletiva) as movimentacoes_coletivas,
    sum(flag_tem_veiculo) as movimentacoes_com_veiculo,
    sum(flag_tem_movimentacao_infopen_mesmo_dia) as movimentacoes_com_ctx_infopen
from gold.sinp_ent_movimentacao
group by tipo_movimentacao
order by tipo_movimentacao
""").show(truncate=False)

spark.sql("""
select
    count(*) as qtd_rl_pessoa_movimentacao,
    sum(case when id_pessoa_presidiario is not null then 1 else 0 end) as qtd_rl_pessoa_resolvida
from gold.sinp_rl_pessoa_movimentacao
""").show(truncate=False)

spark.sql("""
select
    count(*) as qtd_rl_movimentacao_veiculo
from gold.sinp_rl_movimentacao_veiculo
""").show(truncate=False)

spark.sql("""
select
    count(*) as qtd_rl_policial_movimentacao,
    sum(coalesce(flag_condutor, 0)) as qtd_condutores
from gold.sinp_rl_policial_movimentacao
""").show(truncate=False)

In [44]:
spark.sql("refresh table bronze.livros_acesso_unidade_historicalpolicial")
spark.sql("refresh table gold.sinp_ent_pessoa")

spark.sql("refresh table gold.sinp_fil_pront_crim")
spark.sql("refresh table gold.sinp_fil_pront_prof")
spark.sql("refresh table gold.sinp_fil_pront_psico")
spark.sql("refresh table gold.sinp_fil_pront_soc")

tabela = "sinp_pnt_sh_pront"

pnt_sh_pront = spark.sql("""
    with base as (
        select distinct
            trim(regexp_extract(coalesce(documento, ''), '^([^ ]+)', 1)) as documento,
            cast(history_id as bigint) as id_usuario_preenchimento,
            cast(trim(regexp_extract(coalesce(documento, ''), '^([^ ]+)', 1)) as bigint) as id_preso_num
        from bronze.livros_acesso_unidade_historicalpolicial
        where documento is not null
          and trim(documento) <> ''
          and history_id is not null
          and trim(regexp_extract(coalesce(documento, ''), '^([^ ]+)', 1)) rlike '^[0-9]+$'
    ),

    pessoa as (
        select distinct
            cast(id_preso as bigint) as id_preso_num,
            id_pessoa
        from gold.sinp_ent_pessoa
        where id_preso is not null
          and id_pessoa is not null
    ),

    prontuario as (
        select distinct
            cast(id_usuario_preenchimento as bigint) as id_usuario_preenchimento,
            cast(id_pessoa_prontcrim as string) as id_pessoa_prontcrim
        from gold.sinp_fil_pront_crim
        where id_usuario_preenchimento is not null
          and id_pessoa_prontcrim is not null

        union all

        select distinct
            cast(id_usuario_preenchimento as bigint) as id_usuario_preenchimento,
            cast(id_pessoa_prontprof as string) as id_pessoa_prontcrim
        from gold.sinp_fil_pront_prof
        where id_usuario_preenchimento is not null
          and id_pessoa_prontprof is not null

        union all

        select distinct
            cast(id_usuario_preenchimento as bigint) as id_usuario_preenchimento,
            cast(id_pessoa_prontpsico as string) as id_pessoa_prontcrim
        from gold.sinp_fil_pront_psico
        where id_usuario_preenchimento is not null
          and id_pessoa_prontpsico is not null

        union all

        select distinct
            cast(id_usuario_preenchimento as bigint) as id_usuario_preenchimento,
            cast(id_pessoa_prontsoc as string) as id_pessoa_prontcrim
        from gold.sinp_fil_pront_soc
        where id_usuario_preenchimento is not null
          and id_pessoa_prontsoc is not null
    ),

    prontuario_dedup as (
        select distinct
            id_usuario_preenchimento,
            id_pessoa_prontcrim
        from prontuario
    )

    select distinct
        substr(
            md5(
                concat_ws(
                    '|',
                    cast(b.id_preso_num as string),
                    cast(b.id_usuario_preenchimento as string),
                    coalesce(p.id_pessoa, ''),
                    coalesce(pr.id_pessoa_prontcrim, '')
                )
            ),
            1,
            30
        ) as id_pnt_sh_pront,

        b.documento,
        b.id_usuario_preenchimento,
        b.id_preso_num as id_preso,
        p.id_pessoa,
        pr.id_pessoa_prontcrim

    from base b

    left join pessoa p
        on b.id_preso_num = p.id_preso_num

    left join prontuario_dedup pr
        on b.id_usuario_preenchimento = pr.id_usuario_preenchimento
""")

pnt_sh_pront.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(
    pnt_sh_pront,
    "gold",
    tabela,
    f"{path}{tabela}"
)

enviar_gold_para_postgres(
    f"gold.{tabela}",
    "id_pnt_sh_pront"
)

spark.sql(f"""
    select
        count(*) as total_registros,
        count(distinct id_pnt_sh_pront) as total_ids_distintos,
        count(distinct documento) as total_documentos_distintos,
        count(distinct id_usuario_preenchimento) as total_usuarios_preenchimento_distintos,
        count(distinct id_preso) as total_id_preso_distintos,
        count(distinct id_pessoa) as total_id_pessoa_distintos,
        count(distinct id_pessoa_prontcrim) as total_id_pessoa_prontcrim_distintos,
        sum(case when id_pessoa is not null then 1 else 0 end) as registros_com_id_pessoa,
        sum(case when id_pessoa_prontcrim is not null then 1 else 0 end) as registros_com_id_pessoa_prontcrim,
        max(length(id_pnt_sh_pront)) as tamanho_max_id
    from gold.{tabela}
""").show(truncate=False)

spark.sql(f"""
    select *
    from gold.{tabela}
""").show(20, False)

CREATE EXTERNAL TABLE gold.sinp_pnt_sh_pront 
                        (id_pnt_sh_pront varchar(30),
    documento varchar(20),
    id_usuario_preenchimento bigint,
    id_preso bigint,
    id_pessoa char(15),
    id_pessoa_prontcrim varchar(32))
                        STORED AS PARQUET LOCATION '/data_lake/gold/intlpris/sinp_pnt_sh_pront'
                        
DROP TABLE gold.sinp_pnt_sh_pront 
Estatísticas atualizadas para gold.sinp_pnt_sh_pront
Tabela enviada com sucesso para o PostgreSQL: sinp.sinp_pnt_sh_pront
PK definida: id_pnt_sh_pront
+---------------+-------------------+--------------------------+--------------------------------------+------------------------+-------------------------+-----------------------------------+-----------------------+---------------------------------+--------------+
|total_registros|total_ids_distintos|total_documentos_distintos|total_usuarios_preenchimento_distintos|total_id_preso_distintos|total_id_pessoa_distintos|total_id_pessoa_prontcrim_dis

ModuleNotFoundError: No module named 'contexto'